# LLM Case Study — IMF Country Report No. 25/265 (Mongolia, 2025 Article IV)
## Tasks 1-4: PDF processing, structured extraction, RAG and dashboard

**How to run (Colab):** `Runtime → Run all`. Nothing else is needed.
- Cell 0 clones this notebook's public GitHub repository (set `REPO_URL` once before submitting), which holds the report PDF in `Data/`, the LLM response cache and pinned `requirements.txt`. Google Drive is **not** used unless `USE_DRIVE=1` (author's development runs only).
- API keys are optional for reproducing results: all responses are replayed from `outputs/cache/llm/`. To regenerate, add a free-tier `GEMINI_API_KEY` in Colab Secrets.
- Models (Gemini API free tier, 15 requests/min, 250K tokens/min, 500 requests/day each): generator `gemini-3.1-flash-lite`, judge `gemini-3.5-flash-lite`. A per-model rate limiter keeps every call under 14 requests/min. If a run hits the daily quota, re-run after the reset (midnight Pacific): only pending items are sent.

Every LLM call is cached under `outputs/cache/llm/` keyed by a hash of its prompt, model and schema, so re-runs make no API calls, and shipping that folder lets the notebook replay without keys.




## Bootstrap
Clones the project repository so the notebook runs for any Colab user without Google Drive.

In [ ]:
# ==============================================================================
# CELL 0: PROJECT BOOTSTRAP — makes the notebook runnable by ANY Colab user
# ------------------------------------------------------------------------------
# "Open in Colab" from GitHub loads only the .ipynb, not the repository files,
# and drive.mount() would mount Drive (where the PDF is absent).
# So the notebook clones its own public repository, which contains:
#   Data/<report>.pdf          the IMF report
#   outputs/cache/llm/*.json   cached LLM responses (replay without API keys)
#   requirements.txt           pinned package versions
# Resolution order: PROJECT_DIR env var > current folder if it already holds
# Data/ (local clone) > fresh clone of REPO_URL.
# ==============================================================================
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = os.environ.get("REPO_URL", "https://github.com/sri-lekha-19/IMF-Mongolia-Case-study-")


def in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def bootstrap_project_dir(repo_url: str) -> Path:
    """Return the project root, cloning the public repo when running in a fresh runtime."""
    if os.environ.get("PROJECT_DIR"):
        return Path(os.environ["PROJECT_DIR"]).expanduser().resolve()
    if (Path.cwd() / "Data").is_dir():
        return Path.cwd().resolve()
    if os.environ.get("USE_DRIVE") == "1":
        return Path.cwd().resolve()           # Cell 2 resolves the Drive folder instead
    if "<your-github-user>" in repo_url:
        raise ValueError("Set REPO_URL to your public GitHub repository before submitting.")
    base = Path("/content") if in_colab() else Path.cwd()
    target = base / Path(repo_url.rstrip("/")).stem
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", repo_url, str(target)], check=True)
    return target.resolve()


os.environ["PROJECT_DIR"] = str(bootstrap_project_dir(REPO_URL))
print(f"✅ Project root: {os.environ['PROJECT_DIR']}")

## 0. Setup
Dependencies and imports.

In [ ]:
# ==============================================================================
# CELL 1: DEPENDENCIES & IMPORTS
# ------------------------------------------------------------------------------
# Every third-party module used anywhere in the notebook is imported here, so a
# fresh Colab runtime ("Runtime > Run all") never hits a NameError.
# FIX vs previous version: pymupdf, camelot and AutoTokenizer were installed but
# never imported (the type hint `pymupdf.Document` crashed the ingestion cell).
# ==============================================================================
import os
import sys
import subprocess
from pathlib import Path

PACKAGES = ["pymupdf", "pandas", "tabulate", "camelot-py", "transformers", "sentence-transformers",
            "faiss-cpu", "seaborn", "matplotlib", "google-genai", "pydantic", "tenacity"]
REQUIREMENTS = Path(os.environ["PROJECT_DIR"]) / "requirements.txt"


def install_dependencies() -> None:
    """Pinned versions from requirements.txt when shipped (reproducible parsing -> cache hits)."""
    args = ["-r", str(REQUIREMENTS)] if REQUIREMENTS.exists() else PACKAGES
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)
    print(f"Installed from {'requirements.txt' if REQUIREMENTS.exists() else 'unpinned package list'}")


install_dependencies()

import os
import re
import json
import math
import time
import hashlib
import logging
from pathlib import Path
from collections import Counter
from dataclasses import dataclass, field
from typing import List, Dict, Any, Tuple, Optional, Literal, Type

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pymupdf
import camelot
from IPython.display import display
from pydantic import BaseModel, Field, ValidationError, field_validator
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer
import faiss
from google import genai
from google.genai import types
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
sns.set_theme(style="whitegrid")
plt.rcParams.update({"font.size": 12, "axes.labelsize": 14, "axes.titlesize": 16})
print("✅ Imports complete.")

### Configuration

In [ ]:
# ==============================================================================
# CELL 2: PIPELINE CONFIGURATION (single source of truth — no hardcoded paths)
# ------------------------------------------------------------------------------
# Every path, model name and tunable parameter is resolved here from environment
# variables with documented defaults. PROJECT_DIR is set by Cell 0 (cloned repo);
# Google Drive is used only for the author's own runs with USE_DRIVE=1.
# ==============================================================================

def _env(name: str, default: str) -> str:
    """Read an environment variable, falling back to a documented default."""
    value = os.environ.get(name, "").strip()
    return value if value else default


def _detect_colab() -> bool:
    """
    True if in Colab. Google Drive is mounted ONLY when USE_DRIVE=1 (your own
    development runs): drive.mount() asks every user to authorise and mounts
    THEIR Drive
    """
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    if os.environ.get("USE_DRIVE") == "1":
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    return True


def resolve_project_dir(in_colab: bool) -> Path:
    """
    Resolution order:
      1. USE_DRIVE=1 in Colab: <Drive root>/<PROJECT_FOLDER> (author's development runs)
      2. PROJECT_DIR env var (set by Cell 0 to the cloned repository)
      3. Current working directory
    """
    if in_colab and os.environ.get("USE_DRIVE") == "1":
        drive_root = Path(_env("DRIVE_ROOT", "/content/drive/MyDrive"))
        return (drive_root / _env("PROJECT_FOLDER", "Code")).resolve()
    if os.environ.get("PROJECT_DIR"):
        return Path(os.environ["PROJECT_DIR"]).expanduser().resolve()
    return Path.cwd().resolve()


def discover_pdf(data_dir: Path) -> Path:
    """Use PDF_FILENAME if set; otherwise require exactly one PDF in data_dir."""
    if os.environ.get("PDF_FILENAME"):
        return data_dir / os.environ["PDF_FILENAME"]
    pdfs = sorted(data_dir.glob("*.pdf"))
    if len(pdfs) != 1:
        raise FileNotFoundError(
            f"Expected exactly one PDF in {data_dir}, found {len(pdfs)}. "
            "Place the IMF report there or set PDF_FILENAME."
        )
    return pdfs[0]


def load_secret(name: str, in_colab: bool) -> str:
    """Environment variable first, then Colab Secrets. Never printed or logged."""
    value = os.environ.get(name, "")
    if not value and in_colab:
        try:
            from google.colab import userdata
            value = userdata.get(name) or ""
        except Exception:
            value = ""
    return value


@dataclass(frozen=True)
class PipelineConfig:
    project_dir: Path
    pdf_path: Path
    # --- LLM generator (summarisation / extraction) ---
    gemini_models: Tuple[str, ...]            # generator chain, tried in order
    # --- LLM judge (evaluation) ---
    gemini_judge_models: Tuple[str, ...]      # judge chain (distinct from generator)
    judge_max_source_tokens: int
    # --- Chunking ---
    tokenizer_name: str
    chunk_max_tokens: int
    chunk_overlap_tokens: int
    # --- Summarisation ---
    section_context_max_tokens: int
    section_summary_max_words: int
    exec_summary_min_bullets: int
    exec_summary_max_bullets: int
    # --- API pacing ---
    # --- Free-tier rate limits (per model, per Google Cloud project) ---
    gemini_rpm_limit: int
    gemini_tpm_limit: int
    gemini_rpd_limit: int
    # --- Development controls (quota saving; all OFF for the final run) ---
    llm_mode: str                             # "live" | "mock" (no API calls, outputs_mock/)
    run_judge: bool                           # False = judge only from cache, no new judge calls
    dev_max_sections: int                     # 0 = all sections; N = summarise first N only

    @property
    def data_dir(self) -> Path: return self.project_dir / "Data"
    @property
    def output_dir(self) -> Path:
        # Mock runs write to a separate folder so placeholder results can never mix with real ones.
        return self.project_dir / ("outputs_mock" if self.llm_mode == "mock" else "outputs")
    @property
    def text_dir(self) -> Path: return self.output_dir / "extracted" / "text"
    @property
    def table_dir(self) -> Path: return self.output_dir / "extracted" / "tables"
    @property
    def figure_dir(self) -> Path: return self.output_dir / "figures"
    @property
    def task1_dir(self) -> Path: return self.output_dir / "task1"
    @property
    def llm_cache_dir(self) -> Path:
        # Always the REAL cache, also in mock mode (real answers are reused; mocks are never cached).
        return self.project_dir / "outputs" / "cache" / "llm"


IN_COLAB = _detect_colab()
_PROJECT_DIR = resolve_project_dir(IN_COLAB)
(_PROJECT_DIR / "Data").mkdir(parents=True, exist_ok=True)

GEMINI_API_KEY = load_secret("GEMINI_API_KEY", IN_COLAB)

CFG = PipelineConfig(
    project_dir=_PROJECT_DIR,
    pdf_path=discover_pdf(_PROJECT_DIR / "Data"),
    # MODEL CHOICE — from the AI Studio free-tier limits of this project (Nov 2026):
    #   model                   RPM  TPM    RPD
    #   gemini-3.1-flash-lite   15   250K   500   <- generator
    #   gemini-3.5-flash-lite   15   250K   500   <- judge
    #   gemini-3.6-flash         5   250K    20   (too few requests/day)
    #   gemini-3.8-flash         5   250K    20   (too few requests/day)
    # One full run needs ~42 generator calls and ~40 judge calls, so only the two
    # 500-RPD Flash-Lite models fit, with ~10 full runs/day of headroom each.
    # Separate models give each role its own quota and keep the judge from grading
    # its own output. No cross-fallback by default: a judge falling back to the
    # generator model would turn audits into self-evaluation.
    gemini_models=tuple(m for m in [_env("GEMINI_MODEL", "gemini-3.1-flash-lite"),
                                    os.environ.get("GEMINI_FALLBACK_MODEL", "")] if m),
    gemini_judge_models=tuple(m.strip() for m in
                              _env("GEMINI_JUDGE_MODELS", "gemini-3.5-flash-lite").split(",") if m.strip()),
    # Ceiling on the source text a judge reads (DistilBERT tokens). The largest
    # section source is Annex IV (debt sustainability tables, ~18K tokens as chunks);
    # 24K judges every section on its full text, and even that request stays under
    # 15% of the 250K tokens-per-minute limit.
    judge_max_source_tokens=int(_env("JUDGE_MAX_SOURCE_TOKENS", "24000")),
    tokenizer_name=_env("TOKENIZER_NAME", "distilbert-base-uncased"),
    chunk_max_tokens=int(_env("CHUNK_MAX_TOKENS", "400")),
    chunk_overlap_tokens=int(_env("CHUNK_OVERLAP_TOKENS", "50")),
    section_context_max_tokens=int(_env("SECTION_CONTEXT_MAX_TOKENS", "12000")),
    section_summary_max_words=int(_env("SECTION_SUMMARY_MAX_WORDS", "99")),
    exec_summary_min_bullets=int(_env("EXEC_MIN_BULLETS", "8")),
    exec_summary_max_bullets=int(_env("EXEC_MAX_BULLETS", "12")),
    # One below the 15 RPM limit and 90% of 250K TPM, to leave margin for clock skew
    # between this notebook and Google's counter.
    gemini_rpm_limit=int(_env("GEMINI_RPM_LIMIT", "14")),
    gemini_tpm_limit=int(_env("GEMINI_TPM_LIMIT", "225000")),
    gemini_rpd_limit=int(_env("GEMINI_RPD_LIMIT", "500")),
    # DEVELOPMENT CONTROLS — leave unset for the final run:
    #   LLM_MODE=mock       every uncached call returns a schema-valid placeholder (0 API calls);
    #                       cached real answers are still used; outputs go to outputs_mock/.
    #   RUN_JUDGE=0         judge audits come only from cache; no new judge calls.
    #   DEV_MAX_SECTIONS=N  summarise (and judge) only the first N sections.
    llm_mode=_env("LLM_MODE", "live").lower(),
    run_judge=_env("RUN_JUDGE", "1") != "0",
    dev_max_sections=int(_env("DEV_MAX_SECTIONS", "0")),
)

for folder in [CFG.data_dir, CFG.text_dir, CFG.table_dir, CFG.figure_dir,
               CFG.task1_dir, CFG.llm_cache_dir]:
    folder.mkdir(parents=True, exist_ok=True)

print("✅ Configuration resolved")
print(f"   Project dir    : {CFG.project_dir}")
print(f"   PDF            : {CFG.pdf_path.name}")
print(f"   Generator chain: {' -> '.join(CFG.gemini_models)}")
print(f"   Judge chain    : {' -> '.join(CFG.gemini_judge_models)}")
print(f"   Gemini key set : {bool(GEMINI_API_KEY)}")
_dev = [f for f, on in [(f"LLM_MODE={CFG.llm_mode}", CFG.llm_mode != "live"), ("RUN_JUDGE=0", not CFG.run_judge),
                        (f"DEV_MAX_SECTIONS={CFG.dev_max_sections}", CFG.dev_max_sections > 0)] if on]
if _dev:
    print(f"⚠️ DEVELOPMENT SETTINGS ACTIVE: {', '.join(_dev)} — outputs are incomplete; unset them for the final run.")

### Disk I/O and LLM cache

In [ ]:
# ==============================================================================
# CELL 3: DISK I/O & CONTENT-ADDRESSED CACHE
# ------------------------------------------------------------------------------
# Here the cache key is a hash of everything that
# determines the output, and only validated successes are written.
# ==============================================================================

def save_json(obj: Any, path: Path) -> Path:
    """Write JSON atomically (temp file + rename) so a crash never leaves half a file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)
    tmp.replace(path)
    return path


def load_json(path: Path) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def stable_hash(payload: Dict[str, Any]) -> str:
    """Deterministic SHA-256 of a JSON-serialisable payload."""
    blob = json.dumps(payload, sort_keys=True, ensure_ascii=False, default=str)
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()[:24]


class LLMCache:
    """One JSON file per LLM call, named by the hash of its inputs."""

    def __init__(self, cache_dir: Path):
        self.cache_dir = cache_dir
        self.cache_dir.mkdir(parents=True, exist_ok=True)

    def _path(self, key: str) -> Path:
        return self.cache_dir / f"{key}.json"

    def get(self, key: str) -> Optional[Dict[str, Any]]:
        path = self._path(key)
        if not path.exists():
            return None
        try:
            return load_json(path)
        except Exception:
            logging.warning(f"Unreadable cache entry {path.name}; ignoring it.")
            return None

    def put(self, key: str, record: Dict[str, Any]) -> None:
        save_json(record, self._path(key))


def cache_is_persistent(cache_dir: Path, in_colab: bool) -> bool:
    """In Colab only Google Drive survives a runtime restart; /content (the cloned repo) is wiped."""
    return not in_colab or str(cache_dir.resolve()).startswith("/content/drive")


LLM_CACHE = LLMCache(CFG.llm_cache_dir)
print(f"✅ Cache ready: {CFG.llm_cache_dir} ({len(list(CFG.llm_cache_dir.glob('*.json')))} entries)")
if not cache_is_persistent(CFG.llm_cache_dir, IN_COLAB) and GEMINI_API_KEY:
    # A marker (no key) only replays the shipped cache, so this warning targets the author's live runs.
    print("⚠️ This cache is on the Colab runtime disk and is deleted when the runtime restarts, so the next "
          "session would re-send every LLM call and burn free-tier quota.\n"
          "   For development runs set USE_DRIVE=1 (cache kept on Google Drive), or publish outputs with "
          "Cell 99 before the session ends.")

### LLM access layer
Schema-enforced Gemini calls, per-model rate limiter, quota handling and disk cache.

In [ ]:
# ==============================================================================
# CELL 4: LLM ACCESS LAYER (Gemini generator + Gemini judge, schema-enforced, cached)
# ------------------------------------------------------------------------------
# * STRUCTURED OUTPUT: every call sends the Pydantic schema as `response_schema`
#   with response_mime_type="application/json";
# * RATE LIMITS (free tier, per model per project): a sliding 60-second window
#   per model allows at most GEMINI_RPM_LIMIT requests and GEMINI_TPM_LIMIT
#   estimated input tokens. It runs before EVERY attempt, retries included —
# * 429 HANDLING (tenacity): a per-minute 429 waits for the server's retryDelay;
#   a DAILY-quota 429 is not retried — the model is marked exhausted for the
#   session and QuotaExhaustedError lets loops stop cleanly (finished items stay
#   cached and are not re-sent on the next run).
# * 503 "high demand" (server overload, not quota): longer jittered backoff,
# ==============================================================================
import random
import typing
import types as pytypes            # stdlib; `types` is google.genai.types in this notebook
from collections import deque, defaultdict
from datetime import datetime
from zoneinfo import ZoneInfo

GEMINI_CLIENT = genai.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None

_TRANSIENT_MARKERS = ("429", "resource_exhausted", "rate_limit", "rate limit",
                      "500", "internal", "503", "unavailable", "deadline", "timeout")
_MODEL_UNAVAILABLE_MARKERS = ("model_not_found", "does not exist", "not_found", "404",
                              "decommissioned", "deprecated", "no longer available")
UNAVAILABLE_MODELS: Dict[str, str] = {}      # model -> reason (whole session)


class QuotaExhaustedError(RuntimeError):
    """Every model in the chain is out of daily quota or unavailable for this session."""


class LLMSkipped(QuotaExhaustedError):
    """A call was deliberately not made (e.g. RUN_JUDGE=0 and no cached answer); handled like a pause."""


# ------------------------------------------------------------------ rate limit
class RateLimiter:
    """Per-model sliding 60-s window: <= rpm requests and <= tpm estimated input tokens."""

    def __init__(self, rpm: int, tpm: int, window_s: float = 60.0):
        self.rpm, self.tpm, self.window_s = rpm, tpm, window_s
        self.events: Dict[str, deque] = defaultdict(deque)          # model -> deque[(time, tokens)]
        self.total_wait_s = 0.0

    def acquire(self, model: str, tokens: int) -> None:
        if tokens > self.tpm:
            raise ValueError(f"Request of ~{tokens} tokens exceeds the per-minute token limit {self.tpm}; "
                             f"lower SECTION_CONTEXT_MAX_TOKENS / JUDGE_MAX_SOURCE_TOKENS.")
        q = self.events[model]
        while True:
            now = time.monotonic()
            while q and now - q[0][0] >= self.window_s:
                q.popleft()
            if len(q) < self.rpm and sum(t for _, t in q) + tokens <= self.tpm:
                q.append((now, tokens))
                return
            wait = self.window_s - (now - q[0][0]) + 0.05            # until the oldest call leaves the window
            self.total_wait_s += wait
            time.sleep(wait)


def estimate_tokens(*texts: str) -> int:
    """Conservative Gemini input-token estimate (~3 characters per token; tables/numbers tokenise densely)."""
    return int(sum(len(t) for t in texts) / 3) + 1


RATE_LIMITER = RateLimiter(CFG.gemini_rpm_limit, CFG.gemini_tpm_limit)


class UsageTracker:
    """Counts API attempts per model per Pacific-time day (the free-tier daily reset), persisted on disk."""

    def __init__(self, folder: Path, rpd_limit: int):
        self.folder, self.rpd_limit = folder, rpd_limit
        self.folder.mkdir(parents=True, exist_ok=True)
        self.session = Counter()
        self.cache_hits = Counter()
        self.warned = set()

    def _path(self) -> Path:
        return self.folder / f"usage_{datetime.now(ZoneInfo('America/Los_Angeles')):%Y-%m-%d}.json"

    def today(self) -> Dict[str, int]:
        path = self._path()
        return load_json(path) if path.exists() else {}

    def record(self, model: str) -> None:
        counts = self.today()
        counts[model] = counts.get(model, 0) + 1
        save_json(counts, self._path())
        self.session[model] += 1
        if counts[model] >= 0.9 * self.rpd_limit and model not in self.warned:
            self.warned.add(model)
            logging.warning(f"'{model}' has used {counts[model]}/{self.rpd_limit} requests today (Pacific time).")

    def report(self) -> str:
        today = self.today()
        rows = [f"{m}: {self.session[m]} this session, {today.get(m, 0)}/{self.rpd_limit} today"
                for m in sorted(set(today) | set(self.session))]
        hits = sum(self.cache_hits.values())
        misses = getattr(self, "cache_misses", 0)
        return ("API usage — " + ("; ".join(rows) if rows else "no API calls") + f" | cache hits this session: {hits}"
                + f" | cache misses without key: {misses}" + (" ⚠️ items pending" if misses else ""))


USAGE = UsageTracker(CFG.project_dir / "outputs" / "cache" / "usage", CFG.gemini_rpd_limit)


# ------------------------------------------------------------- error handling
def is_daily_quota_error(exc: Exception) -> bool:
    """Free-tier 429 whose violated quota is per day: retrying today cannot succeed."""
    text = str(exc)
    return ("429" in text or "RESOURCE_EXHAUSTED" in text) and "PerDay" in text


def retry_delay_seconds(exc: Exception) -> Optional[float]:
    """Server-suggested wait from a Gemini 429 (RetryInfo.retryDelay, e.g. '37s')."""
    m = re.search(r"retryDelay['\"]?\s*[:=]\s*['\"]?(\d+(?:\.\d+)?)s", str(exc))
    return float(m.group(1)) if m else None


def is_transient_api_error(exc: Exception) -> bool:
    """Retry per-minute limits and server errors; never daily quota or bad requests."""
    if is_daily_quota_error(exc):
        return False
    return any(marker in str(exc).lower() for marker in _TRANSIENT_MARKERS)


def is_model_unavailable_error(exc: Exception) -> bool:
    """Permanent errors: the model id is retired, misspelt or not enabled for this key."""
    return any(marker in str(exc).lower() for marker in _MODEL_UNAVAILABLE_MARKERS)


def is_overload_error(exc: Exception) -> bool:
    """503 'high demand': Google-side congestion, usually clears within minutes."""
    text = str(exc).lower()
    return "503" in text or "high demand" in text or "overloaded" in text


def _wait_for_retry(retry_state) -> float:
    """retryDelay for per-minute 429s; long jittered backoff for 503 overload; else 4-60 s exponential."""
    exc = retry_state.outcome.exception()
    delay = retry_delay_seconds(exc)
    if delay is not None:
        return delay + 1.0
    if is_overload_error(exc):
        wait = min(120.0, 10.0 * 2 ** (retry_state.attempt_number - 1))
        logging.warning(f"Gemini overloaded (503); retrying in ~{wait:.0f} s "
                        f"(attempt {retry_state.attempt_number}/6). Interrupting is safe: finished calls are cached.")
        return wait + random.uniform(0, 5)
    return wait_exponential(multiplier=2, min=4, max=60)(retry_state)


_retry_transient = retry(stop=stop_after_attempt(6), wait=_wait_for_retry,
                         retry=retry_if_exception(is_transient_api_error), reraise=True)


def _strip_code_fences(text: str) -> str:
    text = re.sub(r"^```(?:json)?\s*", "", text.strip())
    return re.sub(r"\s*```$", "", text)


@_retry_transient
def _call_gemini(model: str, system: str, prompt: str, schema: Type[BaseModel]) -> str:
    """One API attempt. The limiter runs inside the retry wrapper, so retries are paced too."""
    RATE_LIMITER.acquire(model, estimate_tokens(system, prompt))
    USAGE.record(model)
    response = GEMINI_CLIENT.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=system,
            response_mime_type="application/json",
            response_schema=schema,
        ),
    )
    if not response.text:
        raise ValueError("Empty response from Gemini.")
    return response.text


# ---------------------------------------------------------- model availability
def list_gemini_models() -> Optional[set]:
    """Model ids this key can call (None if listing fails -> configured ids are tried as-is)."""
    if GEMINI_CLIENT is None:
        return None
    try:
        return {m.name.split("/")[-1] for m in GEMINI_CLIENT.models.list()}
    except Exception as exc:
        logging.warning(f"Could not list Gemini models ({str(exc)[:120]}); using configured ids as-is.")
        return None


def mark_unavailable(models: Tuple[str, ...], available: Optional[set], role: str) -> None:
    """Mark configured ids the key cannot use (the chain itself is unchanged, so cache keys stay stable)."""
    if available is None:
        return
    for m in models:
        if m not in available:
            UNAVAILABLE_MODELS[m] = "not in this key's model list"
            logging.warning(f"{role} model '{m}' is not available to this API key; skipped.")


# ------------------------------------------------------------------- caching
# Earlier notebook versions keyed generator answers on the full chain; looked up
# only on a miss, so previously paid-for summaries are reused and migrated.
LEGACY_CACHE_CHAINS = [("gemini-3.1-flash-lite", "gemini-3.8-flash"),
                       ("gemini-3.1-flash-lite", "gemini-2.5-flash-lite")]


def _cache_key(task, prompt_version, models, system, prompt, schema) -> str:
    return stable_hash({"task": task, "v": prompt_version, "provider": "gemini", "models": list(models),
                        "system": system, "prompt": prompt, "schema": schema.model_json_schema()})


def cache_lookup(task, prompt_version, models, system, prompt, schema) -> Tuple[str, Optional[Dict]]:
    key = _cache_key(task, prompt_version, models[:1], system, prompt, schema)
    hit = LLM_CACHE.get(key)
    if hit is None:
        for chain in LEGACY_CACHE_CHAINS:
            if chain[0] == models[0]:
                hit = LLM_CACHE.get(_cache_key(task, prompt_version, chain, system, prompt, schema))
                if hit is not None:
                    LLM_CACHE.put(key, hit)
                    break
    return key, hit


# --------------------------------------------------------------- mock answers
def mock_value(annotation: Any, name: str = "") -> Any:
    """Minimal schema-valid placeholder for one field (Literal -> first option, list -> 5 items)."""
    origin, args = typing.get_origin(annotation), typing.get_args(annotation)
    if origin is Literal:
        return args[0]
    if origin in (typing.Union, getattr(pytypes, "UnionType", typing.Union)):
        return None if type(None) in args else mock_value(args[0], name)
    if origin in (list, List):
        return [mock_value(args[0], name) for _ in range(5)]
    if isinstance(annotation, type) and issubclass(annotation, BaseModel):
        return {f: mock_value(info.annotation, f) for f, info in annotation.model_fields.items()}
    return {int: 3, float: 0.0, bool: False}.get(annotation, f"[MOCK {name}]")


def mock_record(task: str, schema: Type[BaseModel], models: Tuple[str, ...]) -> Dict[str, Any]:
    parsed = schema.model_validate(mock_value(schema)).model_dump()
    return {"task": task, "provider": "mock", "model_used": f"mock:{models[0]}", "latency_s": 0.0,
            "schema_retry": False, "est_input_tokens": 0, "parsed": parsed, "cached": False, "mock": True}


# ------------------------------------------------------------------ main call
def llm_structured(task: str, system: str, prompt: str, schema: Type[BaseModel],
                   models: Optional[Tuple[str, ...]] = None, prompt_version: str = "v1",
                   cache_only: bool = False) -> Dict[str, Any]:
    """
    One schema-constrained call with caching, model fallback and validation.
    Order: real cache -> mock (LLM_MODE=mock) -> skip (cache_only) -> live API call.
    Returns {parsed, model_used, latency_s, cached, task, ...}. Nothing is cached on failure.
    """
    models = models or CFG.gemini_models
    key, cached = cache_lookup(task, prompt_version, models, system, prompt, schema)
    if cached is not None:
        USAGE.cache_hits[models[0]] += 1
        return {**cached, "cached": True}
    if CFG.llm_mode == "mock":
        return mock_record(task, schema, models)
    if cache_only:
        raise LLMSkipped(f"[{task}] not in cache and new calls are disabled for this step (RUN_JUDGE=0).")
    if GEMINI_CLIENT is None:
        USAGE.cache_misses = getattr(USAGE, "cache_misses", 0) + 1
        # Key-free marking run: a missing cached answer pauses this item (reported as
        # pending) instead of crashing "Run all". The check run should show 0 misses.
        raise LLMSkipped(f"[{task}] cache miss and no GEMINI_API_KEY; item left pending. "
                         "Set the key or ship the complete outputs/cache/llm.")

    live_models = [m for m in models if m not in UNAVAILABLE_MODELS]
    if not live_models:
        raise QuotaExhaustedError(f"[{task}] every model in {list(models)} is unavailable this session: "
                                  f"{ {m: UNAVAILABLE_MODELS.get(m) for m in models} }")
    errors: List[str] = []
    for model in live_models:
        current_prompt = prompt
        for attempt in (1, 2):                                   # 2nd attempt = schema-repair retry
            t0 = time.perf_counter()
            try:
                raw = _call_gemini(model, system, current_prompt, schema)
                parsed = schema.model_validate_json(_strip_code_fences(raw))
                record = {"task": task, "provider": "gemini", "model_used": model,
                          "latency_s": round(time.perf_counter() - t0, 2), "schema_retry": attempt > 1,
                          "est_input_tokens": estimate_tokens(system, current_prompt), "parsed": parsed.model_dump()}
                LLM_CACHE.put(key, record)
                return {**record, "cached": False}
            except ValidationError as ve:
                errors.append(f"{model} schema: {str(ve)[:200]}")
                current_prompt = (prompt + "\n\nYOUR PREVIOUS OUTPUT FAILED SCHEMA "
                                  f"VALIDATION:\n{str(ve)[:600]}\nReturn corrected JSON only.")
            except Exception as exc:
                errors.append(f"{model}: {str(exc)[:200]}")
                if is_daily_quota_error(exc):
                    UNAVAILABLE_MODELS[model] = "daily free-tier quota exhausted"
                    logging.warning(f"Daily free-tier quota exhausted for '{model}'; skipping it until the "
                                    f"quota resets (midnight Pacific time).")
                elif is_model_unavailable_error(exc):
                    UNAVAILABLE_MODELS[model] = str(exc)[:200]
                    logging.warning(f"Model '{model}' unavailable ({str(exc)[:90]}); skipped for this session.")
                break                                            # non-transient: next model
        if model not in UNAVAILABLE_MODELS and len(live_models) > 1:
            logging.warning(f"[{task}] model {model} failed; trying next in chain.")
    if all(m in UNAVAILABLE_MODELS for m in models):
        raise QuotaExhaustedError(f"[{task}] all models exhausted/unavailable: {errors}")
    raise RuntimeError(f"[{task}] all models failed: {errors}")


# EXECUTION
JUDGE_MODELS = CFG.gemini_judge_models
AVAILABLE_GEMINI_MODELS = list_gemini_models()
mark_unavailable(CFG.gemini_models, AVAILABLE_GEMINI_MODELS, "Generator")
mark_unavailable(JUDGE_MODELS, AVAILABLE_GEMINI_MODELS, "Judge")
if set(JUDGE_MODELS) & set(CFG.gemini_models):
    logging.warning("Judge and generator share a model; those audits will be labelled self_evaluation.")
print(f"✅ LLM layer ready | Gemini client: {GEMINI_CLIENT is not None}")
print(f"   Generator: {' -> '.join(CFG.gemini_models)} | Judge: {' -> '.join(JUDGE_MODELS)}")
print(f"   Rate limit per model: {CFG.gemini_rpm_limit} req/min, {CFG.gemini_tpm_limit:,} est. tokens/min")
print(f"   {USAGE.report()}")

## 1. Document ingestion

In [ ]:
# ==============================================================================
# CELL 5: DOCUMENT INGESTION, METADATA & RAW EXTRACTION (PyMuPDF)
# ==============================================================================

def validate_pdf(pdf_path: Path) -> bool:
    """Check that the input PDF exists and has a .pdf extension."""
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF file not found at: {pdf_path}")
    if pdf_path.suffix.lower() != ".pdf":
        raise ValueError("The input file must have a .pdf extension.")
    return True


def load_pdf(pdf_path: Path) -> "pymupdf.Document":
    """Open a PDF with PyMuPDF; fail loudly on empty or unreadable files."""
    validate_pdf(pdf_path)
    try:
        document = pymupdf.open(pdf_path)
    except Exception as error:
        raise RuntimeError(f"Unable to open PDF: {error}") from error
    if document.page_count == 0:
        document.close()
        raise ValueError("The PDF contains no pages.")
    return document


def extract_document_metadata(document: "pymupdf.Document") -> Dict[str, Any]:
    """Basic metadata + page count (title is reused in prompts)."""
    metadata = document.metadata or {}
    return {"title": metadata.get("title"), "creator": metadata.get("creator"),
            "producer": metadata.get("producer"), "page_count": document.page_count}


def extract_page_text(document: "pymupdf.Document") -> List[Dict[str, Any]]:
    """Reading-order text per page (used for header/footer frequency analysis)."""
    return [{"page_number": n, "text": page.get_text("text", sort=True)}
            for n, page in enumerate(document, start=1)]


def extract_page_blocks(document: "pymupdf.Document") -> List[Dict[str, Any]]:
    """Text blocks with bounding boxes, dehyphenated at line ends."""
    blocks_out = []
    flags = pymupdf.TEXTFLAGS_BLOCKS | pymupdf.TEXT_DEHYPHENATE
    for page_number, page in enumerate(document, start=1):
        for block in page.get_text("blocks", flags=flags, sort=True):
            x0, y0, x1, y1, text, block_number, block_type = block[:7]
            if block_type != 0 or not text.strip():
                continue                                  # skip images / blank blocks
            blocks_out.append({"page_number": page_number, "block_number": block_number,
                               "text": text.strip(),
                               "bbox": [round(x0, 2), round(y0, 2), round(x1, 2), round(y1, 2)]})
    return blocks_out


# EXECUTION
document = load_pdf(CFG.pdf_path)
document_metadata = extract_document_metadata(document)
page_texts = extract_page_text(document)
text_blocks = extract_page_blocks(document)

save_json(document_metadata, CFG.output_dir / "document_metadata.json")
save_json(page_texts, CFG.text_dir / "raw_page_text.json")
save_json(text_blocks, CFG.text_dir / "raw_text_blocks.json")

for key, value in document_metadata.items():
    print(f"{key:12s}: {value}")
print(f"\n✅ Pages: {len(page_texts)} | Text blocks: {len(text_blocks)}")

## 2. Text cleaning

In [ ]:
# ==============================================================================
# CELL 6: TEXT CLEANING — RUNNING HEADERS, FOOTERS, PAGE NUMBERS
# -----------------------------------------------------------------------------
# ==============================================================================

def normalize_line(line: str) -> str:
    """Collapse whitespace and lowercase."""
    return re.sub(r"\s+", " ", line).strip().lower()


def header_signature(line: str) -> str:
    """Page-number-invariant signature used only for header/footer detection."""
    norm = normalize_line(line)
    return re.sub(r"\d+", "#", norm) if re.search(r"[a-z]", norm) else norm


def identify_repeated_lines(pages: List[Dict[str, Any]], margin_lines: int = 5,
                            min_page_share: float = 0.30) -> set:
    """Signatures recurring in the top/bottom margin of >= min_page_share of pages."""
    counts: Counter = Counter()
    for page in pages:
        lines = [header_signature(l) for l in page["text"].splitlines() if l.strip()]
        counts.update(set(lines[:margin_lines] + lines[-margin_lines:]))
    threshold = max(2, math.ceil(len(pages) * min_page_share))
    return {sig for sig, c in counts.items() if c >= threshold}


def clean_lines(text: str, repeated_signatures: set) -> List[str]:
    """Drop recurring header/footer lines and standalone page numbers."""
    kept = []
    for line in text.splitlines():
        stripped = line.strip()
        if not stripped or header_signature(stripped) in repeated_signatures:
            continue
        if re.fullmatch(r"[-–—]?\s*\d+\s*[-–—]?", stripped):
            continue
        kept.append(stripped)
    return kept


def clean_narrative_blocks(blocks: List[Dict[str, Any]], repeated_signatures: set) -> List[Dict[str, Any]]:
    """Clean every block while keeping page/bbox metadata."""
    cleaned = []
    for block in blocks:
        lines = clean_lines(block["text"], repeated_signatures)
        if lines:
            cleaned.append({**block, "type": "narrative", "text": " ".join(lines)})
    return cleaned


# EXECUTION
repeated_lines = identify_repeated_lines(page_texts)
cleaned_narrative_blocks = clean_narrative_blocks(text_blocks, repeated_lines)
save_json(cleaned_narrative_blocks, CFG.text_dir / "cleaned_narrative_blocks.json")

print("Recurring margin signatures removed:")
for sig in sorted(repeated_lines):
    print(f"   • {sig}")
print(f"\n✅ Blocks: {len(text_blocks)} -> {len(cleaned_narrative_blocks)} after cleaning")

## 3. Table extraction

In [ ]:
# ==============================================================================
# CELL 7: TABLE EXTRACTION (Camelot) — CONFIG-DRIVEN, HUMAN-CURATED
# ------------------------------------------------------------------------------
# Page lists and rejections below come from manual inspection of the PDF
# (human oversight step). Pages are PDF page numbers.
# ==============================================================================

TABLE_EXTRACTION_CONFIG = {
    "stream":  {"pages": [5, 10, 17, 34, 35, 36, 37, 38, 39, 40, 45, 59, 67, 87, 94],
                "parameters": {}},
    "lattice": {"pages": [41, 42, 43, 49, 50, 51, 56, 59, 62, 76],
                "parameters": {"line_scale": 15}},
}
# Stream candidates rejected after inspection: {pdf_page: [table_index, ...]}
REJECTED_STREAM_TABLES = {10: [2], 17: [1, 2], 45: [1], 87: [1, 2]}


def extract_configured_tables(pdf_path: Path, extraction_config: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Run Camelot per configured flavour/page; one failure never stops the loop."""
    candidates = []
    for flavor, conf in extraction_config.items():
        for page_number in conf["pages"]:
            try:
                tables = camelot.read_pdf(str(pdf_path), pages=str(page_number),
                                          flavor=flavor, **conf["parameters"])
            except Exception as error:
                logging.warning(f"Table extraction failed | page {page_number} | {flavor}: {error}")
                continue
            for table_index, table in enumerate(tables, start=1):
                candidates.append({"page_number": page_number, "flavor": flavor,
                                   "table_index": table_index, "dataframe": table.df,
                                   "parsing_report": table.parsing_report,
                                   "bbox": list(table._bbox)})
    return candidates


def select_accepted_tables(candidates: List[Dict[str, Any]], rejected: Dict[int, List[int]]) -> List[Dict[str, Any]]:
    """Drop stream candidates listed in the rejection map."""
    return [c for c in candidates
            if not (c["flavor"] == "stream" and c["table_index"] in rejected.get(c["page_number"], []))]


def summarise_candidates(candidates: List[Dict[str, Any]]) -> pd.DataFrame:
    """Camelot quality diagnostics for the report (accuracy / whitespace)."""
    return pd.DataFrame([{"page": c["page_number"], "method": c["flavor"], "table": c["table_index"],
                          "rows": c["dataframe"].shape[0], "columns": c["dataframe"].shape[1],
                          "accuracy": c["parsing_report"].get("accuracy"),
                          "whitespace": c["parsing_report"].get("whitespace")} for c in candidates])


def inspect_tables(tables: List[Dict[str, Any]], pages: List[int]) -> None:
    """Optional visual check of tables on given pages."""
    for t in tables:
        if t["page_number"] in pages:
            print(f"\nPage {t['page_number']} | {t['flavor']} | table {t['table_index']}")
            display(t["dataframe"])


# EXECUTION
table_candidates = extract_configured_tables(CFG.pdf_path, TABLE_EXTRACTION_CONFIG)
accepted_tables = select_accepted_tables(table_candidates, REJECTED_STREAM_TABLES)
candidate_summary = summarise_candidates(table_candidates)
candidate_summary.to_csv(CFG.table_dir / "table_candidate_diagnostics.csv", index=False)
display(candidate_summary)
print(f"✅ Candidates: {len(table_candidates)} | Rejected: "
      f"{len(table_candidates) - len(accepted_tables)} | Accepted: {len(accepted_tables)}")

## 4. Structural block classification

In [ ]:
# ==============================================================================
# CELL 8: STRUCTURAL CLASSIFICATION OF NARRATIVE BLOCKS
# ------------------------------------------------------------------------------
# Labels each block (heading / caption / source note / footnote / paragraph ...)
# so that headings drive the section map and captions survive de-duplication.
# ==============================================================================

TABLE_CAPTION_RE = re.compile(r"^(?:(?:annex|appendix)\s+[ivxlcdm\d]+\s*[.:,-]?\s*)?(?:text\s+)?table\s+\d+\b", re.I)
FIGURE_CAPTION_RE = re.compile(r"^(?:(?:annex|appendix)\s+[ivxlcdm\d]+\s*[.:,-]?\s*)?(?:figure|fig\.)\s+\d+\b", re.I)


def classify_block(text: str) -> str:
    """Rule-based structural label for one cleaned block."""
    text = re.sub(r"\s+", " ", text).strip()
    words = text.split()
    letters = re.findall(r"[A-Za-z]", text)
    upper_ratio = sum(ch.isupper() for ch in letters) / len(letters) if letters else 0
    if TABLE_CAPTION_RE.match(text):
        return "table_caption"
    if FIGURE_CAPTION_RE.match(text):
        return "figure_caption"
    if re.match(r"^(source|sources|note|notes)\s*:", text, re.I):
        return "source_note_candidate"
    if re.match(r"^\(?\s*(in\s+)?(percent|percentage|million|billion|thousand|mn|usd|mnt)\b", text, re.I):
        return "table_units_candidate"
    if re.match(r"^\d+\s*/", text):
        return "footnote_candidate"
    if len(text) <= 160 and len(words) <= 12 and (
            upper_ratio >= 0.65 or re.match(r"^(annex|appendix|chapter|section)\b", text, re.I)):
        return "heading_candidate"
    return "paragraph_candidate" if len(words) >= 12 else "other_candidate"


def classify_narrative_blocks(blocks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    return [{**b, "candidate_type": classify_block(b["text"])} for b in blocks]


def identify_caption_continuations(blocks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Mark the short block directly under a caption as its continuation (layout-based)."""
    by_key = {(b["page_number"], b["block_number"]): dict(b) for b in blocks}
    previous_caption, continuation_found = None, False
    for b in sorted(blocks, key=lambda x: (x["page_number"], x["bbox"][1], x["bbox"][0])):
        if b["candidate_type"] in ("table_caption", "figure_caption"):
            previous_caption, continuation_found = b, False
            continue
        if previous_caption is None or continuation_found:
            continue
        if b["page_number"] != previous_caption["page_number"]:
            previous_caption = None
            continue
        pb, bb = previous_caption["bbox"], b["bbox"]
        gap = bb[1] - pb[3]
        overlap = max(0, min(bb[2], pb[2]) - max(bb[0], pb[0]))
        min_width = min(bb[2] - bb[0], pb[2] - pb[0])
        ratio = overlap / min_width if min_width > 0 else 0
        if (-2 <= gap <= 5 and ratio >= 0.4 and len(b["text"].split()) <= 8
                and b["candidate_type"] in ("other_candidate", "paragraph_candidate")):
            by_key[(b["page_number"], b["block_number"])]["candidate_type"] = "caption_continuation_candidate"
            continuation_found = True
    return list(by_key.values())


# EXECUTION
classified_blocks = identify_caption_continuations(classify_narrative_blocks(cleaned_narrative_blocks))
print("✅ Block types:", dict(Counter(b["candidate_type"] for b in classified_blocks)))

## 5. Parsed document

In [ ]:
# ==============================================================================
# CELL 9: TABLE/NARRATIVE DE-DUPLICATION & FINAL PARSED DOCUMENT
# ------------------------------------------------------------------------------
# ==============================================================================

PROTECTED_BLOCK_TYPES = ("table_caption", "figure_caption", "caption_continuation_candidate",
                         "source_note_candidate", "footnote_candidate", "heading_candidate")


def normalize_content(text: str) -> str:
    text = re.sub(r"[^a-z0-9.%+-]+", " ", str(text).lower())
    return re.sub(r"\s+", " ", text).strip()


def filter_duplicate_table_blocks(blocks: List[Dict[str, Any]], tables: List[Dict[str, Any]],
                                  coverage_threshold: float = 0.90) -> Tuple[List[Dict], List[Dict]]:
    """Drop narrative blocks whose tokens are >= threshold covered by a table on the same page."""
    table_tokens_by_page: Dict[int, List[set]] = {}
    for t in tables:
        flat = " ".join(t["dataframe"].fillna("").astype(str).values.flatten())
        table_tokens_by_page.setdefault(t["page_number"], []).append(set(normalize_content(flat).split()))
    kept, removed = [], []
    for b in blocks:
        tokens = set(normalize_content(b["text"]).split())
        if len(tokens) < 8 or b["candidate_type"] in PROTECTED_BLOCK_TYPES:
            kept.append(b)
            continue
        dup = any(len(tokens & tt) / len(tokens) >= coverage_threshold
                  for tt in table_tokens_by_page.get(b["page_number"], []))
        (removed if dup else kept).append(b)
    return kept, removed


def build_parsed_document(blocks: List[Dict[str, Any]], tables: List[Dict[str, Any]],
                          metadata: Dict[str, Any]) -> Dict[str, Any]:
    """Page-ordered structure: narrative blocks + accepted tables per page."""
    pages = []
    for p in sorted({b["page_number"] for b in blocks} | {t["page_number"] for t in tables}):
        page_blocks = sorted([dict(b) for b in blocks if b["page_number"] == p], key=lambda b: b["block_number"])
        page_tables = sorted([{"table_index": t["table_index"], "flavor": t["flavor"],
                               "data": t["dataframe"].fillna("").astype(str).values.tolist(),
                               "bbox": t["bbox"], "parsing_report": t["parsing_report"]}
                              for t in tables if t["page_number"] == p],
                             key=lambda t: (t["flavor"], t["table_index"]))
        pages.append({"page_number": p, "narrative_blocks": page_blocks, "tables": page_tables})
    return {"document_metadata": metadata, "pages": pages}


def clean_table_cell(value: Any) -> str:
    return "" if value is None else re.sub(r"\s+", " ", str(value)).replace("|", r"\|").strip()


def dataframe_to_markdown(df: pd.DataFrame, headers: Optional[List[str]] = None) -> str:
    headers = [clean_table_cell(h) for h in (headers or [f"Column {i + 1}" for i in range(df.shape[1])])]
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    lines += ["| " + " | ".join(clean_table_cell(c) for c in row) + " |" for row in df.values.tolist()]
    return "\n".join(lines)


def parsed_document_to_markdown(parsed: Dict[str, Any]) -> str:
    parts = []
    for page in parsed["pages"]:
        parts.append(f"\n## Page {page['page_number']}\n")
        parts += [b["text"] for b in page["narrative_blocks"]]
        for t in page["tables"]:
            parts.append(f"\n### Table {t['table_index']} ({t['flavor'].title()} extraction)\n")
            parts.append(dataframe_to_markdown(pd.DataFrame(t["data"])))
    return "\n\n".join(parts)


# EXECUTION
filtered_narrative_blocks, removed_table_duplicates = filter_duplicate_table_blocks(classified_blocks, accepted_tables)
parsed_document = build_parsed_document(filtered_narrative_blocks, accepted_tables, document_metadata)
save_json(parsed_document, CFG.text_dir / "parsed_document.json")
(CFG.text_dir / "parsed_document.md").write_text(parsed_document_to_markdown(parsed_document), encoding="utf-8")

print(f"✅ Narrative blocks kept: {len(filtered_narrative_blocks)} | "
      f"removed as table duplicates: {len(removed_table_duplicates)} | pages: {len(parsed_document['pages'])}")

## 6. Section map (from the report's table of contents)
Review the printed table: `heading_verified` should be True for every chapter, subsection and annex.

In [ ]:
# ==============================================================================
# CELL 10: SECTION MAP DERIVED FROM THE REPORT'S OWN TABLE OF CONTENTS
# ------------------------------------------------------------------------------
# Here sections are parsed from the CONTENTS pages, the printed->PDF offset is
# inferred from where the first heading actually appears, and IMF package
# components outside the TOC (press release, key issues, informational annex,
# authorities' statement) are located by generic Article IV anchors.
# ==============================================================================

# Package components that IMF Article IV documents carry outside the staff-report TOC.
# Patterns are case-sensitive on purpose (the cover page mentions these in lower case).
PACKAGE_ANCHORS = [
    {"section_id": "press_release", "title": "Press Release & Executive Board Assessment",
     "kind": "press_release", "pattern": r"\bPR\s?\d{2}/\d{2,3}\b|FOR IMMEDIATE RELEASE", "after_toc": False},
    {"section_id": "key_issues", "title": "Key Issues (Staff Report Overview)",
     "kind": "key_issues", "pattern": r"^KEY ISSUES$", "after_toc": False},
    {"section_id": "informational_annex", "title": "Informational Annex (Fund Relations)",
     "kind": "informational_annex", "pattern": r"INFORMATIONAL ANNEX", "after_toc": True},
    {"section_id": "authorities_statement", "title": "Statement by the Alternate Executive Director",
     "kind": "authorities_statement", "pattern": r"^Statement by .*Executive Director", "after_toc": True},
]
SUMMARISE_KINDS = {"press_release", "key_issues", "chapter", "subsection", "annex",
                   "informational_annex", "authorities_statement"}
EXEC_SUMMARY_EXCLUDE_KINDS = {"informational_annex"}      # administrative content only
TOC_ENTRY_RE = re.compile(r"(?P<title>[A-Z0-9][^_]{2,160}?)\s*_{3,}\s*(?P<page>\d{1,3})(?!\d)")
TOC_GROUPS = {"FIGURES": "figure", "TABLES": "table", "ANNEXES": "annex"}


def slugify(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")[:60]


def page_blocks(parsed: Dict[str, Any]) -> Dict[int, List[str]]:
    return {p["page_number"]: [b["text"].strip() for b in p["narrative_blocks"]] for p in parsed["pages"]}


def find_toc_pages(blocks_by_page: Dict[int, List[str]]) -> List[int]:
    """The page holding a 'CONTENTS' block plus consecutive pages with TOC leader lines."""
    start = next((p for p in sorted(blocks_by_page) if "CONTENTS" in blocks_by_page[p]), None)
    if start is None:
        raise ValueError("No CONTENTS page found — supply a manual section map.")
    pages, p = [start], start + 1
    while p in blocks_by_page and len(TOC_ENTRY_RE.findall(" ".join(blocks_by_page[p]))) >= 2:
        pages.append(p)
        p += 1
    return pages


def parse_toc_entries(toc_text: str) -> List[Dict[str, Any]]:
    """Parse 'TITLE ____ page' entries; FIGURES/TABLES/ANNEXES prefixes switch the group."""
    entries, group = [], "main"
    for m in TOC_ENTRY_RE.finditer(toc_text):
        title = re.sub(r"\s+", " ", m.group("title")).strip()
        first = title.split(" ", 1)[0]
        if first in TOC_GROUPS:
            group, title = TOC_GROUPS[first], title.split(" ", 1)[1]
        if group == "main":
            kind = "subsection" if re.match(r"^[A-Z]\.\s", title) else "chapter"
            sid = slugify(title)
        elif group == "annex":
            roman = title.split(".", 1)[0]
            kind, sid, title = "annex", f"annex_{roman}", f"Annex {title}"
        else:
            kind, sid = group, f"{group}_{slugify(title)}"
        entries.append({"section_id": sid, "title": title, "kind": kind, "printed_page": int(m.group("page"))})
    return entries


def infer_page_offset(entries: List[Dict[str, Any]], blocks_by_page: Dict[int, List[str]], after_page: int) -> int:
    """PDF page of the first chapter heading minus its printed page number."""
    first = next(e for e in entries if e["kind"] == "chapter")
    for p in sorted(blocks_by_page):
        if p > after_page and any(b.upper() == first["title"].upper() for b in blocks_by_page[p]):
            return p - first["printed_page"]
    raise ValueError(f"Heading '{first['title']}' not found after the TOC — cannot infer page offset.")


def _norm(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip().upper()


def locate_heading_block(title: str, blocks: List[str]) -> Optional[int]:
    """Index of the block on a page that starts with the section heading (whitespace-insensitive)."""
    target = _norm(title)[:25]
    return next((i for i, b in enumerate(blocks) if _norm(b).startswith(target)), None)


def find_anchor(anchor: Dict[str, Any], blocks_by_page: Dict[int, List[str]], min_page: int) -> Optional[Tuple[int, int]]:
    """(page, block_index) of the first block matching an anchor pattern."""
    rx = re.compile(anchor["pattern"])
    for p in sorted(blocks_by_page):
        if p < min_page:
            continue
        for i, b in enumerate(blocks_by_page[p]):
            if rx.search(b):
                return p, i
    return None


def build_section_starts(parsed: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Every section start as (start_page, start_block), sorted in reading order."""
    blocks_by_page = page_blocks(parsed)
    toc_pages = find_toc_pages(blocks_by_page)
    # Only blocks that contain leader lines are TOC entries (skips headers / mission text).
    toc_text = " ".join(b for p in toc_pages for b in blocks_by_page[p] if TOC_ENTRY_RE.search(b))
    entries = parse_toc_entries(toc_text)
    offset = infer_page_offset(entries, blocks_by_page, toc_pages[-1])
    logging.info(f"Printed->PDF page offset inferred: +{offset}")

    starts = []
    for e in entries:
        page = e.pop("printed_page") + offset
        idx = locate_heading_block(e["title"], blocks_by_page.get(page, []))
        verifiable = e["kind"] in ("chapter", "subsection", "annex")
        starts.append({**e, "start_page": page, "start_block": idx or 0,
                       "heading_verified": (idx is not None) if verifiable else None})
    starts.append({"section_id": "table_of_contents", "title": "Table of Contents", "kind": "toc",
                   "start_page": toc_pages[0], "start_block": 0, "heading_verified": None})
    for a in PACKAGE_ANCHORS:
        hit = find_anchor(a, blocks_by_page, toc_pages[-1] + 1 if a["after_toc"] else 1)
        if hit is None:
            logging.warning(f"Anchor '{a['section_id']}' not found; section skipped.")
            continue
        starts.append({**{k: a[k] for k in ("section_id", "title", "kind")},
                       "start_page": hit[0], "start_block": hit[1], "heading_verified": None})
    starts.sort(key=lambda s: (s["start_page"], s["start_block"]))
    if (starts[0]["start_page"], starts[0]["start_block"]) > (min(blocks_by_page), 0):
        starts.insert(0, {"section_id": "front_matter", "title": "Cover / Front Matter", "kind": "front_matter",
                          "start_page": min(blocks_by_page), "start_block": 0, "heading_verified": None})
    return starts


def assign_sections(parsed: Dict[str, Any], starts: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Tag every narrative block and table with the section active at its position.
    Block-level boundaries matter: page 16 of this report holds the end of
    'Outlook and Risks', the 'Policy Discussions' intro and the start of 'A.'.
    Tables have no reliable in-page order, so they take the section active at
    the end of their page (approximation, documented).
    """
    keys = [(s["start_page"], s["start_block"]) for s in starts]

    def section_at(page: int, block_idx: int) -> str:
        pos = max(i for i, k in enumerate(keys) if k <= (page, block_idx)) if keys[0] <= (page, block_idx) else 0
        return starts[pos]["section_id"]

    for page in parsed["pages"]:
        n_blocks = len(page["narrative_blocks"])
        for i, block in enumerate(page["narrative_blocks"]):
            block["section_id"] = section_at(page["page_number"], i)
        for table in page["tables"]:
            table["section_id"] = section_at(page["page_number"], max(n_blocks - 1, 0))
    return parsed


def merge_consecutive(sections: List[Dict[str, Any]], kind: str, merged_id: str, title: str) -> Dict[str, str]:
    """Collapse runs of per-figure / per-table entries; returns old_id -> merged_id."""
    remap = {}
    for s in sections:
        if s["kind"] == kind:
            remap[s["section_id"]] = merged_id
    return remap


def build_section_map(parsed: Dict[str, Any]) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    starts = build_section_starts(parsed)
    remap = {**merge_consecutive(starts, "figure", "figures", "Figures"),
             **merge_consecutive(starts, "table", "tables", "Statistical Tables")}
    merged_titles = {"figures": ("Figures", "figure"), "tables": ("Statistical Tables", "table")}
    parsed = assign_sections(parsed, starts)
    for page in parsed["pages"]:
        for item in page["narrative_blocks"] + page["tables"]:
            item["section_id"] = remap.get(item["section_id"], item["section_id"])

    # Aggregate the realised extent of each section from its tagged blocks.
    meta = {s["section_id"]: s for s in starts}
    extent: Dict[str, Dict[str, Any]] = {}
    for page in parsed["pages"]:
        for item in page["narrative_blocks"] + page["tables"]:
            sid = item["section_id"]
            e = extent.setdefault(sid, {"pages": set(), "words": 0})
            e["pages"].add(page["page_number"])
            e["words"] += len(item.get("text", " ".join(" ".join(r) for r in item.get("data", []))).split())

    sections, seen = [], set()
    for s in starts:
        sid = remap.get(s["section_id"], s["section_id"])
        if sid in seen or sid not in extent:
            continue                                   # merged duplicate or empty heading
        seen.add(sid)
        title, kind = merged_titles.get(sid, (s["title"], s["kind"]))
        sections.append({"section_id": sid, "title": title, "kind": kind,
                         "start_page": min(extent[sid]["pages"]), "end_page": max(extent[sid]["pages"]),
                         "word_count": extent[sid]["words"],
                         "heading_verified": meta.get(sid, {}).get("heading_verified"),
                         "summarise": kind in SUMMARISE_KINDS,
                         "include_in_exec_summary": kind in SUMMARISE_KINDS and kind not in EXEC_SUMMARY_EXCLUDE_KINDS})
    return sections, parsed


# EXECUTION — tags parsed_document blocks/tables with `section_id` in place.
SECTION_MAP, parsed_document = build_section_map(parsed_document)
SECTIONS_BY_ID = {s["section_id"]: s for s in SECTION_MAP}
save_json(SECTION_MAP, CFG.text_dir / "section_map.json")
save_json(parsed_document, CFG.text_dir / "parsed_document.json")    # now section-tagged

section_map_df = pd.DataFrame(SECTION_MAP)
display(section_map_df)
unverified = section_map_df[section_map_df["heading_verified"] == False]
if not unverified.empty:
    logging.warning(f"Headings not found on computed start page: {unverified['section_id'].tolist()}")
print(f"✅ {len(SECTION_MAP)} sections | {int(section_map_df['summarise'].sum())} to summarise")

## 7. Token-aware chunking

In [ ]:
# ==============================================================================
# CELL 11: TOKEN-AWARE, SECTION-AWARE CHUNKING (offset-safe)
# ------------------------------------------------------------------------------
#    DistilBERT WordPiece tokenizer this lowercases all text and splits decimals:
#    "64.5 percent of GDP" -> "64. 5 percent of gdp". Every downstream numeric
#    extraction (Task 2 JSON) and RAG answer (Task 3) would read corrupted
#    numbers. Chunks are now sliced from the ORIGINAL string using the
#    tokenizer's character offset_mapping, so the text is byte-identical.
#  * Sliding windows ran per page, so paragraphs crossing a page break never
#    shared an overlap. Windows now run over each SECTION's continuous text;
#    every chunk records the pages it spans and its section_id.
# Tables keep the previous multi-row-header fusion and are never split mid-row.
# ==============================================================================

tokenizer = AutoTokenizer.from_pretrained(CFG.tokenizer_name)
if not tokenizer.is_fast:
    raise RuntimeError("A fast tokenizer is required for offset mapping.")


def count_tokens(text: str) -> int:
    return len(tokenizer(text, add_special_tokens=False, verbose=False)["input_ids"])


def section_text_stream(parsed: Dict[str, Any], section_id: str) -> Tuple[str, List[Tuple[int, int, int]]]:
    """Concatenate a section's narrative blocks; return text + [(char_start, char_end, page)]."""
    parts, spans, cursor = [], [], 0
    for page in parsed["pages"]:
        for block in page["narrative_blocks"]:
            if block.get("section_id") != section_id:
                continue
            spans.append((cursor, cursor + len(block["text"]), page["page_number"]))
            parts.append(block["text"])
            cursor += len(block["text"]) + 2            # +2 for the "\n\n" joiner
    return "\n\n".join(parts), spans


def pages_for_span(spans: List[Tuple[int, int, int]], start: int, end: int) -> List[int]:
    """Pages whose blocks overlap the character span [start, end)."""
    return sorted({p for (s, e, p) in spans if s < end and e > start})


def token_windows(text: str, max_tokens: int, overlap: int) -> List[Tuple[int, int, int]]:
    """Overlapping windows as (char_start, char_end, n_tokens) over the ORIGINAL string."""
    offsets = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True,
                        verbose=False)["offset_mapping"]
    if not offsets:
        return []
    step, windows = max(1, max_tokens - overlap), []
    for i in range(0, len(offsets), step):
        window = offsets[i:i + max_tokens]
        windows.append((window[0][0], window[-1][1], len(window)))
        if i + max_tokens >= len(offsets):
            break
    return windows


def chunk_narrative(text: str, page_marks: List[Tuple[int, int, int]], section: Dict[str, Any],
                    max_tokens: int, overlap: int) -> List[Dict[str, Any]]:
    """Sliding token window; chunk text is an exact substring of the source."""
    chunks = []
    for n, (start_char, end_char, n_tok) in enumerate(token_windows(text, max_tokens, overlap), start=1):
        pages = pages_for_span(page_marks, start_char, end_char)
        chunks.append({"chunk_id": f"{section['section_id']}_n{n:03d}", "chunk_type": "narrative",
                       "section_id": section["section_id"], "section_title": section["title"],
                       "page_number": pages[0], "page_end": pages[-1], "pages": pages,
                       "text": text[start_char:end_char], "token_count": n_tok})
    return chunks


def parse_multirow_table(raw_rows: List[List[Any]]) -> Tuple[str, str, List[str], List[List[Any]]]:
    """Fuse title / year / actual-projection / unit header rows into single column names."""
    if not raw_rows:
        return "", "", [], []
    n_cols = max(len(r) for r in raw_rows)
    title, unit, header_idx = "", "", []
    title_re = re.compile(r"^(table|annex|text table)\s+[A-Za-z0-9.]+", re.I)
    unit_re = re.compile(r"^\(in\s+(percent|millions|billions|mnt|usd)", re.I)
    for idx in range(min(5, len(raw_rows))):
        row_text = " ".join(str(c).strip() for c in raw_rows[idx] if c).strip()
        if not row_text:
            continue
        if title_re.search(row_text):
            title = row_text; header_idx.append(idx)
        elif unit_re.search(row_text):
            unit = row_text; header_idx.append(idx)
        elif len(re.findall(r"\b20[123]\d\b", row_text)) >= 2 or \
                re.search(r"\b(actual|projections?|estimates?|budget)\b", row_text, re.I):
            header_idx.append(idx)
    cut = (max(header_idx) + 1) if header_idx else 1
    headers = []
    for c in range(n_cols):
        pieces = []
        for row in raw_rows[:cut]:
            val = str(row[c]).strip() if c < len(row) and row[c] else ""
            if val and not title_re.search(val) and not unit_re.search(val) and val not in pieces:
                pieces.append(val)
        headers.append(" ".join(pieces) or ("Indicator" if c == 0 else f"Col_{c + 1}"))
    return title, unit, headers, raw_rows[cut:]


def guard_header_size(headers: List[str], rows: List[List[Any]], all_rows: List[List[Any]],
                      max_tokens: int) -> Tuple[List[str], List[List[Any]]]:
    """
    Text-matrix tables (e.g. the Risk Assessment Matrix) have paragraph-length
    first rows; repeating them as headers in every chunk blows the token budget.
    If fused headers exceed a quarter of the budget, keep all rows as data and
    use generic column names instead.
    """
    if count_tokens(" | ".join(headers)) <= max_tokens // 4:
        return headers, rows
    return [f"Col_{i + 1}" for i in range(len(headers))], all_rows


def chunk_table(table: Dict[str, Any], page_num: int, section: Dict[str, Any], max_tokens: int) -> List[Dict[str, Any]]:
    """Row-wise table chunks; title, unit and fused headers repeated in every chunk."""
    raw = table.get("data", [])
    title, unit, headers, rows = parse_multirow_table(raw)
    if not rows:
        return []
    data_rows = [r for r in raw if " ".join(str(c) for c in r if c).strip() not in (title, unit)]
    headers, rows = guard_header_size(headers, rows, data_rows, max_tokens)
    rows = [([str(c) for c in r] + [""] * len(headers))[:len(headers)] for r in rows]
    tid = f"p{page_num}_{table['flavor']}_t{table['table_index']}"
    head = f"### [Page {page_num} Table {table['table_index']}{': ' + title if title else ''}]\n"
    head += f"*Unit: {unit}*\n\n" if unit else ""

    def make(sub_rows: List[List[str]], k: int) -> Dict[str, Any]:
        text = head + pd.DataFrame(sub_rows, columns=headers).to_markdown(index=False)
        return {"chunk_id": f"{section['section_id']}_{tid}_{k}", "chunk_type": "table",
                "section_id": section["section_id"], "section_title": section["title"],
                "page_number": page_num, "page_end": page_num, "pages": [page_num],
                "table_title": title, "table_unit": unit, "text": text, "token_count": count_tokens(text)}

    def split_oversize_row(row: List[str]) -> List[Dict[str, Any]]:
        """A single row above budget is flattened to 'header: value' text and windowed."""
        flat = "; ".join(f"{h}: {v}" for h, v in zip(headers, row) if str(v).strip())
        out = []
        for a, b, _ in token_windows(flat, max_tokens - count_tokens(head) - 5, CFG.chunk_overlap_tokens):
            text = head + flat[a:b]
            out.append({**make([], 0), "chunk_id": f"{section['section_id']}_{tid}_{len(chunks) + len(out) + 1}",
                        "text": text, "token_count": count_tokens(text)})
        return out

    chunks, current = [], []
    for row in rows:
        if make([row], 0)["token_count"] > max_tokens:            # row alone is too big
            if current:
                chunks.append(make(current, len(chunks) + 1)); current = []
            chunks += split_oversize_row(row)
            continue
        if current and make(current + [row], 0)["token_count"] > max_tokens:
            chunks.append(make(current, len(chunks) + 1)); current = []
        current.append(row)
    if current:
        chunks.append(make(current, len(chunks) + 1))
    return chunks


def chunk_document(parsed: Dict[str, Any], sections: List[Dict[str, Any]],
                   max_tokens: int, overlap: int) -> List[Dict[str, Any]]:
    by_id = {s["section_id"]: s for s in sections}
    chunks = []
    for s in sections:
        text, marks = section_text_stream(parsed, s["section_id"])
        chunks += chunk_narrative(text, marks, s, max_tokens, overlap)
    for page in parsed["pages"]:
        for table in page["tables"]:
            chunks += chunk_table(table, page["page_number"], by_id[table["section_id"]], max_tokens)
    return chunks


def verify_chunks_are_verbatim(chunks: List[Dict[str, Any]], parsed: Dict[str, Any]) -> float:
    """Share of narrative chunks that are exact substrings of their section text."""
    streams = {sid: section_text_stream(parsed, sid)[0] for sid in {c["section_id"] for c in chunks}}
    narr = [c for c in chunks if c["chunk_type"] == "narrative"]
    return sum(c["text"] in streams[c["section_id"]] for c in narr) / max(1, len(narr))


# EXECUTION
document_chunks = chunk_document(parsed_document, SECTION_MAP, CFG.chunk_max_tokens, CFG.chunk_overlap_tokens)
save_json(document_chunks, CFG.text_dir / "document_chunks.json")

chunk_df = pd.DataFrame(document_chunks)
print(f"✅ {len(document_chunks)} chunks | max tokens: {chunk_df['token_count'].max()} "
      f"(limit {CFG.chunk_max_tokens}) | verbatim narrative chunks: "
      f"{verify_chunks_are_verbatim(document_chunks, parsed_document):.0%}")
display(chunk_df.groupby(["chunk_type"])["token_count"].describe().round(1))

## 8. Prompt registry and output schemas

In [ ]:
# ==============================================================================
# CELL 12: PROMPT REGISTRY & OUTPUT SCHEMAS (Task 1)
# ------------------------------------------------------------------------------
# All prompts live here, versioned, so
# that changing a prompt automatically invalidates its cache entries.
#
# Prompt-design changes vs previous version (rationale for the report):
#  * Grounding rules moved into the SYSTEM instruction (applies to every call)
#    instead of being repeated loosely in each user prompt.
#  * Number fidelity made explicit: copy numbers exactly with unit + period;
#    status labels (outturn / projection / risk / recommendation / authorities'
#    view) are mandatory -> the "actual vs projected" failure mode is targeted.
#  * Page markers [p.N] in the context let the model cite pages, which the judge
#    and the numeric-grounding check then verify.
#  * Output is a schema (Pydantic -> native JSON schema), not free text, so
#    word counts, bullet counts and citations can be checked in code.
#  * Judge: claim-level verification with verbatim evidence + anchored 1-5
# ==============================================================================

PROMPT_VERSION = "t1-v2"
REPORT_TITLE = (document_metadata.get("title") or "IMF Country Report").split(";")[0]

STATUS = Literal["outturn", "projection", "risk", "recommendation", "authorities_view"]


class KeyFinding(BaseModel):
    statement: str = Field(description="One self-contained sentence with figure(s), unit and period.")
    category: STATUS
    financial_stability_relevance: Literal["high", "medium", "low"]
    pages: List[int] = Field(description="PDF page numbers from the [p.N] markers.")


class SectionDigest(BaseModel):
    summary: str = Field(description="One paragraph, fewer than 100 words.")
    key_findings: List[KeyFinding] = Field(description="Up to 6 findings, most material first.")


class ExecBullet(BaseModel):
    text: str = Field(description="One bullet, at most 45 words.")
    category: Literal["finding", "risk", "recommendation"]
    pages: List[int]
    source_section_ids: List[str]


class ExecutiveSummary(BaseModel):
    bullets: List[ExecBullet]


class ClaimCheck(BaseModel):
    claim: str
    verdict: Literal["SUPPORTED", "UNSUPPORTED", "CONTRADICTED"]
    evidence: str = Field(description="Shortest verbatim SOURCE excerpt (<=25 words), or empty.")


class JudgeAudit(BaseModel):
    """Required judge output: three 1-5 quality scores + unsupported claims + justification."""
    claims: List[ClaimCheck]
    factual_alignment_score: int = Field(description="Integer 1-5: agreement of every claim with SOURCE.")
    consistency_score: int = Field(description="Integer 1-5: internal coherence and correct status labels.")
    completeness_score: int = Field(description="Integer 1-5: coverage of the main points of REFERENCE.")
    unsupported_claims: List[str] = Field(description="Claims judged UNSUPPORTED or CONTRADICTED; [] if none.")
    omitted_key_points: List[str]
    justification: str = Field(description="2-4 sentences explaining all three scores with reference to SOURCE.")

    @field_validator("factual_alignment_score", "consistency_score", "completeness_score")
    @classmethod
    def _score_in_range(cls, v: int) -> int:
        # Kept out of the JSON schema (plain ints); a violation triggers the schema-repair retry.
        if not 1 <= v <= 5:
            raise ValueError("score must be an integer from 1 to 5")
        return v


JUDGE_PROMPT_VERSION = "judge-v3"     # judge prompts versioned separately: summaries' cache stays valid


# ------------------------------------------------------------------ generator
SYSTEM_SECTION = f"""You are an IMF macro-financial analyst summarising one section of: {REPORT_TITLE}.
Apply these rules to every sentence you write:
1. Use only the SOURCE in the user message. No outside knowledge.
2. Every number must appear in SOURCE with the same unit, reference period and status. Copy it exactly; never round, convert or compute.
3. State status precisely: outturn (already happened), IMF staff projection, risk (could happen), staff recommendation, or the authorities' view. Never present a projection as an outturn, or the authorities' view as staff's.
4. [p.N] markers give the PDF page of the text that follows; cite them in key_findings.pages.
5. If SOURCE is mainly a table or chart notes, report what it measures and its most decision-relevant values.
6. Use the report's own terms (e.g. DSTI, NBFI, structural deficit); add a plain-English gloss in parentheses on first use only if it costs six words or fewer."""

USER_SECTION = """SECTION: {title} (PDF pages {start}-{end})

TASK
A. summary: one paragraph of {target_min}-{target_max} words (hard limit: fewer than 100 words). Start with the substance, never "This section...". Cover the main finding, its key supporting figures, and the policy implication or recommendation if the section gives one.
B. key_findings: up to 6, most material first. Each is one self-contained sentence carrying its figure(s), unit and period, with category, pages and financial_stability_relevance (high = bears on bank/NBFI credit, sovereign debt, external buffers, inflation or systemic risk).

<source>
{context}
</source>"""

USER_SECTION_MERGE = """SECTION: {title} (PDF pages {start}-{end})
The section was longer than the context budget, so it was digested in {n_parts} consecutive parts (below).
Produce the final digest for the WHOLE section following the same TASK rules: summary of {target_min}-{target_max} words (fewer than 100), and up to 6 key_findings drawn only from the partial digests, keeping their pages.

<partial_digests>
{partials}
</partial_digests>"""

WORD_LIMIT_RETRY = ("\n\nYOUR PREVIOUS SUMMARY HAD {n} WORDS. Rewrite the summary in at most {target} words, "
                    "keeping every figure exact. Keep key_findings unchanged.")

SYSTEM_EXEC = f"""You are a senior IMF macro-financial economist writing the executive summary of: {REPORT_TITLE}.
You work only from the FINDINGS INVENTORY in the user message; each finding was extracted from the cited PDF pages of the report.
Rules:
1. Every bullet must be traceable to at least one inventory finding; list its pages and section ids.
2. Use no number that is not in the inventory, and keep its unit and period.
3. Keep status explicit in the wording (outturn, projection, risk, recommendation).
4. Where staff and the authorities disagree, say so rather than blending their views."""

USER_EXEC = """TASK: Write {min_b}-{max_b} bullets summarising the primary financial-stability findings and risk assessments of the complete report.

Selection and ordering:
- Prioritise items with financial_stability_relevance = high; order bullets by materiality for financial stability.
- Merge findings describing the same issue in different sections into one bullet that states the mechanism (cause -> consequence).
- At least {min_risk} bullets must be risk assessments or vulnerabilities; the final bullet states the IMF's overall policy priority.
Style: each bullet at most 45 words, one idea, plain English, figures with unit and period.

FINDINGS INVENTORY (section_id | category | relevance | pages | finding)
<inventory>
{inventory}
</inventory>"""

# ---------------------------------------------------------------------- judge
SYSTEM_JUDGE = """You are an independent auditor checking an AI-generated summary of an IMF country report against the report's own text. You did not write the summary.
Judge only against the texts in the user message; your own knowledge of the country or the IMF is irrelevant.
Be strict: a claim containing a number is SUPPORTED only if the same number, unit and period appear in SOURCE. Presenting a projection as an outturn is CONTRADICTED."""

USER_JUDGE = """CANDIDATE TYPE: {candidate_type}
Evaluate CANDIDATE on consistency, completeness and factual alignment with SOURCE. Return ALL fields.
STEP 1 - claims: split CANDIDATE into atomic factual claims (one fact each, at most 15). Verdict per claim: SUPPORTED (SOURCE states it), UNSUPPORTED (SOURCE does not state it) or CONTRADICTED (SOURCE states otherwise, incl. wrong number, unit, period or status). evidence = shortest verbatim SOURCE excerpt (<=25 words) behind the verdict, or "".
STEP 2 - factual_alignment_score (1-5): 5 = every claim SUPPORTED; 4 = all supported with minor imprecision in wording; 3 = one UNSUPPORTED claim, none contradicted; 2 = several UNSUPPORTED or one CONTRADICTED; 1 = multiple CONTRADICTED or mostly unsupported.
STEP 3 - consistency_score (1-5): 5 = no internal contradictions and status (outturn / projection / risk / recommendation) used correctly throughout; 3 = one ambiguity or status slip; 1 = self-contradictory.
STEP 4 - completeness_score (1-5) against the main points of REFERENCE: 5 = all main points present; 4 = one secondary point missing; 3 = one main point missing; 2 = several main points missing; 1 = misses the purpose. List omitted main points (at most 5) in omitted_key_points.
STEP 5 - unsupported_claims: copy every claim whose verdict is UNSUPPORTED or CONTRADICTED ([] if none).
STEP 6 - justification: 2-4 sentences that justify each of the three scores, citing SOURCE.

<reference>
{reference}
</reference>

<source>
{source}
</source>

<candidate>
{candidate}
</candidate>"""

print(f"✅ Prompt registry {PROMPT_VERSION} loaded | report title: {REPORT_TITLE}")

## 9. Task 1 — Section summaries (< 100 words)

In [ ]:
# ==============================================================================
# CELL 13: TASK 1 — CHAPTER / SECTION SUMMARIES (< 100 words) + KEY FINDINGS
# ------------------------------------------------------------------------------
# FIXES vs previous version:
#  * Covers all 26 summarisable sections from the TOC-derived map
#    it exceeds SECTION_CONTEXT_MAX_TOKENS, it is digested in token-budgeted
#    groups of chunks and merged (token-aware summarisation loop).
#  * One structured call returns both the <100-word summary and the section's
#    key findings, which feed the executive summary (map step) — halving calls.
#  * Word limit enforced in code with a targeted rewrite retry.
# ==============================================================================

def build_section_context(parsed: Dict[str, Any], section_id: str) -> str:
    """Full section text in reading order, with a [p.N] marker at each page change."""
    parts, current_page = [], None
    for page in parsed["pages"]:
        for block in page["narrative_blocks"]:
            if block.get("section_id") != section_id:
                continue
            if page["page_number"] != current_page:
                current_page = page["page_number"]
                parts.append(f"[p.{current_page}]")
            parts.append(block["text"])
        for table in page["tables"]:
            if table.get("section_id") == section_id:
                parts.append(f"[p.{page['page_number']}] TABLE\n" + dataframe_to_markdown(pd.DataFrame(table["data"])))
    return "\n\n".join(parts)


def chunks_of(section_id: str, chunks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    return [c for c in chunks if c["section_id"] == section_id]


def group_chunks_by_budget(chunks: List[Dict[str, Any]], budget: int) -> List[str]:
    """Pack consecutive chunks (with page markers) into groups under the token budget."""
    groups, current, used = [], [], 0
    for c in chunks:
        text = f"[p.{c['page_number']}]\n{c['text']}"
        if current and used + c["token_count"] > budget:
            groups.append("\n\n".join(current)); current, used = [], 0
        current.append(text); used += c["token_count"]
    if current:
        groups.append("\n\n".join(current))
    return groups


def word_count(text: str) -> int:
    return len(text.split())


def digest_with_word_limit(section: Dict[str, Any], base_prompt: str, max_words: int,
                           max_attempts: int = 3) -> Dict[str, Any]:
    """Generate a SectionDigest; re-ask for a shorter summary if over the word limit."""
    best, prompt = None, base_prompt
    for attempt in range(1, max_attempts + 1):
        rec = llm_structured(task=f"section_digest:{section['section_id']}", system=SYSTEM_SECTION,
                             prompt=prompt, schema=SectionDigest, prompt_version=PROMPT_VERSION)
        n = word_count(rec["parsed"]["summary"])
        if best is None or n < best["word_count"]:
            best = {**rec, "word_count": n, "attempts": attempt}
        if n <= max_words:
            break
        logging.warning(f"[{section['section_id']}] summary {n} words (> {max_words}); retrying.")
        prompt = base_prompt + WORD_LIMIT_RETRY.format(n=n, target=max_words - 20)
    best["passed_word_limit"] = best["word_count"] <= max_words
    return best


def summarise_section(section: Dict[str, Any], parsed: Dict[str, Any],
                      chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Single pass if the section fits the budget, otherwise map over chunk groups and merge."""
    fmt = dict(title=section["title"], start=section["start_page"], end=section["end_page"],
               target_min=60, target_max=85)
    context = build_section_context(parsed, section["section_id"])
    n_tokens = count_tokens(context)

    if n_tokens <= CFG.section_context_max_tokens:
        rec = digest_with_word_limit(section, USER_SECTION.format(context=context, **fmt),
                                     CFG.section_summary_max_words)
        strategy = "single_pass"
    else:
        partials = []
        for i, group in enumerate(group_chunks_by_budget(chunks_of(section["section_id"], chunks),
                                                         CFG.section_context_max_tokens), start=1):
            part = llm_structured(task=f"section_part:{section['section_id']}:{i}", system=SYSTEM_SECTION,
                                  prompt=USER_SECTION.format(context=group, **fmt),
                                  schema=SectionDigest, prompt_version=PROMPT_VERSION)
            partials.append(f"PART {i}\n" + json.dumps(part["parsed"], ensure_ascii=False))
        merge_prompt = USER_SECTION_MERGE.format(partials="\n\n".join(partials), n_parts=len(partials), **fmt)
        rec = digest_with_word_limit(section, merge_prompt, CFG.section_summary_max_words)
        strategy = f"map_reduce_{len(partials)}_parts"

    return {"section_id": section["section_id"], "title": section["title"], "kind": section["kind"],
            "start_page": section["start_page"], "end_page": section["end_page"],
            "context_tokens": n_tokens, "strategy": strategy,
            "summary": rec["parsed"]["summary"], "key_findings": rec["parsed"]["key_findings"],
            "word_count": rec["word_count"], "passed_word_limit": rec["passed_word_limit"],
            "attempts": rec["attempts"], "model_used": rec["model_used"],
            "latency_s": rec["latency_s"], "cached": rec["cached"], "status": "success"}


def run_section_summaries(sections: List[Dict[str, Any]], parsed: Dict[str, Any],
                          chunks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Summarise every section flagged `summarise`; one failure never aborts the loop."""
    results = []
    targets = [s for s in sections if s["summarise"]]
    if CFG.dev_max_sections:
        logging.warning(f"DEV_MAX_SECTIONS={CFG.dev_max_sections}: summarising {CFG.dev_max_sections} of "
                        f"{len(targets)} sections (development only).")
        targets = targets[:CFG.dev_max_sections]
    for i, s in enumerate(targets, start=1):
        logging.info(f"[{i}/{len(targets)}] {s['section_id']}")
        try:
            results.append(summarise_section(s, parsed, chunks))
        except LLMSkipped as exc:                  # one uncached item (no key): skip it, keep going
            logging.error(f"[{s['section_id']}] pending: {str(exc)[:120]}")
            results.append({"section_id": s["section_id"], "title": s["title"], "status": "pending"})
        except QuotaExhaustedError as exc:
            logging.error(f"Paused at {s['section_id']}; {len(targets) - i + 1} sections left pending. "
                          f"Re-run later — finished sections load from cache. ({str(exc)[:120]})")
            results += [{"section_id": t["section_id"], "title": t["title"], "status": "pending"}
                        for t in targets[i - 1:]]
            break
        except Exception as exc:
            logging.error(f"[{s['section_id']}] failed: {exc}")
            results.append({"section_id": s["section_id"], "title": s["title"], "status": "error",
                            "error": str(exc)[:500]})
    return results


# EXECUTION
section_summaries = run_section_summaries(SECTION_MAP, parsed_document, document_chunks)
save_json(section_summaries, CFG.task1_dir / "section_summaries.json")

ok = [r for r in section_summaries if r["status"] == "success"]
print(f"✅ Section summaries: {len(ok)}/{len(section_summaries)} succeeded | "
      f"word limit passed: {sum(r['passed_word_limit'] for r in ok)}/{len(ok)}")
if ok:
    display(pd.DataFrame(ok)[["section_id", "start_page", "end_page", "context_tokens", "strategy",
                              "word_count", "passed_word_limit", "attempts", "model_used"]])
for r in ok[:3]:
    print(f"\n### {r['title']} (pp. {r['start_page']}-{r['end_page']}, {r['word_count']} words)\n{r['summary']}")

## 10. Task 1 — Executive summary

In [ ]:
# ==============================================================================
# CELL 14: TASK 1 — EXECUTIVE SUMMARY OF THE COMPLETE REPORT (map-reduce)
# ------------------------------------------------------------------------------

NUMBER_RE = re.compile(r"(?<![\w.])-?\d{1,3}(?:,\d{3})+(?:\.\d+)?|(?<![\w.])-?\d+(?:\.\d+)?(?:½)?")


def extract_numbers(text: str) -> set:
    """Numbers as floats ('5½' -> 5.5, '17,146' -> 17146). Ignores bare integers < 10 (list
    markers, ranks) to keep the check focused on statistics."""
    out = set()
    for raw in NUMBER_RE.findall(text.replace("−", "-").replace("–", "-")):
        half = raw.endswith("½")
        val = float(raw.rstrip("½").replace(",", "")) + (0.5 if half else 0.0)
        if "." in raw or half or abs(val) >= 10:
            out.add(round(abs(val), 3))
    return out


def numeric_grounding(candidate: str, source: str) -> Dict[str, Any]:
    """Which numbers in the candidate do not occur anywhere in the source text?"""
    cand, src = extract_numbers(candidate), extract_numbers(source)
    missing = sorted(cand - src)
    return {"numbers_checked": len(cand), "ungrounded_numbers": missing,
            "numeric_grounding_pct": round(100 * (1 - len(missing) / len(cand)), 1) if cand else 100.0}


def build_findings_inventory(summaries: List[Dict[str, Any]], sections_by_id: Dict[str, Dict]) -> str:
    rows = []
    for r in summaries:
        if r.get("status") != "success" or not sections_by_id[r["section_id"]]["include_in_exec_summary"]:
            continue
        for f in r["key_findings"]:
            statement = re.sub(r"\s+", " ", f["statement"]).replace("|", "/").strip()   # one line per finding
            rows.append(f"{r['section_id']} | {f['category']} | {f['financial_stability_relevance']} | "
                        f"{','.join(map(str, f['pages']))} | {statement}")
    return "\n".join(rows)


def pages_text(parsed: Dict[str, Any], pages: set) -> str:
    """Raw text (narrative + tables) of a set of PDF pages."""
    out = []
    for page in parsed["pages"]:
        if page["page_number"] in pages:
            out += [b["text"] for b in page["narrative_blocks"]]
            out += [" ".join(" ".join(r) for r in t["data"]) for t in page["tables"]]
    return "\n".join(out)


def generate_executive_summary(summaries: List[Dict[str, Any]], parsed: Dict[str, Any]) -> Dict[str, Any]:
    inventory = build_findings_inventory(summaries, SECTIONS_BY_ID)
    min_risk = max(3, CFG.exec_summary_min_bullets // 3)
    prompt = USER_EXEC.format(min_b=CFG.exec_summary_min_bullets, max_b=CFG.exec_summary_max_bullets,
                              min_risk=min_risk, inventory=inventory)
    rec = llm_structured(task="executive_summary", system=SYSTEM_EXEC, prompt=prompt,
                         schema=ExecutiveSummary, prompt_version=PROMPT_VERSION)
    bullets = rec["parsed"]["bullets"]
    cited_pages = {p for b in bullets for p in b["pages"]}
    text = "\n".join(f"- {b['text']}" for b in bullets)
    checks = {
        "n_bullets": len(bullets),
        "bullet_count_ok": CFG.exec_summary_min_bullets <= len(bullets) <= CFG.exec_summary_max_bullets,
        "n_risk_bullets": sum(b["category"] == "risk" for b in bullets),
        "max_bullet_words": max((word_count(b["text"]) for b in bullets), default=0),
        "bullets_without_citation": sum(not b["pages"] for b in bullets),
        "sections_cited": sorted({s for b in bullets for s in b["source_section_ids"]}),
        **numeric_grounding(text, pages_text(parsed, cited_pages)),
    }
    return {"status": "success", "bullets": bullets, "markdown": text, "cited_pages": sorted(cited_pages),
            "inventory_size": inventory.count("\n") + 1 if inventory else 0, "checks": checks,
            "model_used": rec["model_used"], "latency_s": rec["latency_s"], "cached": rec["cached"]}


# EXECUTION
try:
    executive_summary = generate_executive_summary(section_summaries, parsed_document)
except Exception as exc:                       # recorded, never cached; judge cell skips it
    logging.error(f"Executive summary failed: {exc}")
    executive_summary = {"status": "error", "error": str(exc)[:500]}
save_json(executive_summary, CFG.task1_dir / "executive_summary.json")

print("=" * 80 + "\nTASK 1: EXECUTIVE SUMMARY\n" + "=" * 80)
for b in executive_summary.get("bullets", []):
    print(f"• [{b['category']}] {b['text']}  (pp. {', '.join(map(str, b['pages']))})")
print("\nAutomated checks:", json.dumps(executive_summary.get("checks", executive_summary), indent=2))

## 11. Task 1 — LLM-as-a-Judge evaluation

In [ ]:
# ==============================================================================
# CELL 15: TASK 1 — DUAL-LLM EVALUATOR (LLM-as-a-Judge) + DETERMINISTIC CHECKS
# ------------------------------------------------------------------------------
# * Judge = a different free-tier Gemini model from the generator (Cell 2), so no
#   score is a model grading its own output; independence is recorded per audit.
# * The judge reads the same full section text the summariser used (budgeted by
#   relevance only above JUDGE_MAX_SOURCE_TOKENS, with coverage logged).
# * The executive summary is audited too.
# * Required judge output: factual_alignment, consistency and completeness
#   scores (1-5, anchored rubrics), unsupported claims and a justification.
# ==============================================================================

PASS_THRESHOLD_SCORE = 4.0          # mean of the three 1-5 scores
JUDGE_METRICS = ["factual_alignment_score", "consistency_score", "completeness_score"]


def independence_label(judge_model: str, generator_model: str) -> str:
    """How independent was this audit? Uses the models that actually answered."""
    return "self_evaluation" if judge_model == generator_model else "same_family_different_model"


def budgeted_source(chunks: List[Dict[str, Any]], candidate: str, budget: int) -> Tuple[str, float]:
    """All chunks if they fit; otherwise the most candidate-relevant ones, kept in reading order."""
    total = sum(c["token_count"] for c in chunks) or 1
    if total <= budget:
        selected = chunks
    else:
        cand_words = set(re.findall(r"[a-z0-9.]+", candidate.lower()))
        ranked = sorted(chunks, key=lambda c: -len(cand_words & set(re.findall(r"[a-z0-9.]+", c["text"].lower()))))
        keep, used = set(), 0
        for c in ranked:
            if used + c["token_count"] <= budget:
                keep.add(c["chunk_id"]); used += c["token_count"]
        selected = [c for c in chunks if c["chunk_id"] in keep]
    text = "\n\n".join(f"[p.{c['page_number']}]\n{c['text']}" for c in selected)
    return text, round(100 * sum(c["token_count"] for c in selected) / total, 1)


def score_audit(audit: Dict[str, Any]) -> Dict[str, Any]:
    """The judge's three 1-5 scores (primary) + claim-verdict counts (diagnostic, computed in code)."""
    verdicts = Counter(c["verdict"] for c in audit["claims"])
    n = len(audit["claims"])
    scores = {m: audit[m] for m in JUDGE_METRICS}
    return {**scores, "overall_score": round(sum(scores.values()) / 3, 2),
            "n_claims": n, "supported": verdicts["SUPPORTED"], "unsupported": verdicts["UNSUPPORTED"],
            "contradicted": verdicts["CONTRADICTED"],
            "supported_claims_pct": round(100 * verdicts["SUPPORTED"] / n, 1) if n else None,
            "n_unsupported_listed": len(audit["unsupported_claims"])}


def run_judge(target_id: str, candidate_type: str, candidate: str, source: str, reference: str,
              generator_model: str) -> Dict[str, Any]:
    prompt = USER_JUDGE.format(candidate_type=candidate_type, reference=reference,
                               source=source, candidate=candidate)
    rec = llm_structured(task=f"judge:{target_id}", system=SYSTEM_JUDGE, prompt=prompt,
                         schema=JudgeAudit, models=JUDGE_MODELS, prompt_version=JUDGE_PROMPT_VERSION,
                         cache_only=not CFG.run_judge)      # RUN_JUDGE=0: cached audits only
    return {"audit": rec["parsed"], "scores": score_audit(rec["parsed"]),
            "judge_model": rec["model_used"], "generator_model": generator_model,
            "judge_independence": independence_label(rec["model_used"], generator_model),
            "latency_s": rec["latency_s"], "cached": rec["cached"]}


def audit_section_summary(rec: Dict[str, Any], chunks: List[Dict[str, Any]], parsed: Dict[str, Any]) -> Dict[str, Any]:
    source, coverage = budgeted_source(chunks_of(rec["section_id"], chunks), rec["summary"], CFG.judge_max_source_tokens)
    result = run_judge(rec["section_id"], "section summary (one paragraph, fewer than 100 words)",
                       rec["summary"], source, reference="Same as SOURCE.", generator_model=rec["model_used"])
    deterministic = {"word_count": rec["word_count"], "passed_word_limit": rec["passed_word_limit"],
                     **numeric_grounding(rec["summary"], build_section_context(parsed, rec["section_id"]))}
    return {"target_id": rec["section_id"], "target_type": "section_summary",
            "source_coverage_pct": coverage, "deterministic": deterministic, **result}


def audit_executive_summary(exec_rec: Dict[str, Any], chunks: List[Dict[str, Any]], parsed: Dict[str, Any],
                            reference_sections: Tuple[str, ...] = ("key_issues", "staff_appraisal")) -> Dict[str, Any]:
    """Facts are checked against the cited pages; completeness against the IMF's own overview
    sections (Key Issues + Staff Appraisal), which state what the report considers primary."""
    cited = set(exec_rec["cited_pages"])
    cited_chunks = [c for c in chunks if cited & set(c["pages"])]
    source, coverage = budgeted_source(cited_chunks, exec_rec["markdown"], CFG.judge_max_source_tokens)
    reference = "\n\n".join(build_section_context(parsed, sid) for sid in reference_sections if sid in SECTIONS_BY_ID)
    result = run_judge("executive_summary",
                       f"executive summary ({CFG.exec_summary_min_bullets}-{CFG.exec_summary_max_bullets} bullets) "
                       f"of the primary financial-stability findings and risk assessments of the whole report",
                       exec_rec["markdown"], source, reference, generator_model=exec_rec["model_used"])
    return {"target_id": "executive_summary", "target_type": "executive_summary",
            "source_coverage_pct": coverage, "deterministic": exec_rec["checks"], **result}


def run_task1_evaluation(summaries, exec_rec, chunks, parsed) -> List[Dict[str, Any]]:
    audits = []
    targets = [r for r in summaries if r.get("status") == "success"]
    for i, rec in enumerate(targets, start=1):
        logging.info(f"[judge {i}/{len(targets)}] {rec['section_id']}")
        try:
            audits.append(audit_section_summary(rec, chunks, parsed))
        except LLMSkipped as exc:                  # uncached audit (no key / RUN_JUDGE=0): skip only this one
            logging.error(f"[{rec['section_id']}] judge audit pending: {str(exc)[:100]}")
            audits.append({"target_id": rec["section_id"], "status": "pending"})
        except QuotaExhaustedError as exc:
            pending = [r["section_id"] for r in targets[i - 1:]] + ["executive_summary"]
            logging.error(f"Judge paused: {len(pending)} audits pending ({', '.join(pending[:5])}...). "
                          f"Reason: {str(exc)[:100]}. Finished audits are cached; re-run to complete.")
            audits += [{"target_id": t, "status": "pending"} for t in pending]
            return audits
        except Exception as exc:
            logging.error(f"Judge failed for {rec['section_id']}: {exc}")
            audits.append({"target_id": rec["section_id"], "target_type": "section_summary",
                           "status": "error", "error": str(exc)[:500]})
    if exec_rec.get("status") == "success":
        try:
            audits.append(audit_executive_summary(exec_rec, chunks, parsed))
        except QuotaExhaustedError:
            logging.error("Judge paused before the executive-summary audit (quota or RUN_JUDGE=0).")
            audits.append({"target_id": "executive_summary", "status": "pending"})
        except Exception as exc:
            logging.error(f"Judge failed for executive summary: {exc}")
    return audits


SCORE_COLUMNS = ["target_id", "target_type", "factual_alignment_score", "consistency_score", "completeness_score",
                 "overall_score", "n_claims", "supported", "unsupported", "contradicted", "supported_claims_pct",
                 "n_unsupported_listed", "numeric_grounding_pct", "ungrounded_numbers", "source_coverage_pct",
                 "judge_model", "generator_model", "judge_independence", "verdict"]


def audits_to_frame(audits: List[Dict[str, Any]]) -> pd.DataFrame:
    """One tidy row per audited output (feeds the Task 4 judge-score plot); fixed columns even if empty."""
    rows = []
    for a in audits:
        if "scores" not in a:
            continue
        verdict_ok = (a["scores"]["overall_score"] >= PASS_THRESHOLD_SCORE and a["scores"]["contradicted"] == 0
                      and a["deterministic"].get("passed_word_limit", a["deterministic"].get("bullet_count_ok", True)))
        rows.append({"target_id": a["target_id"], "target_type": a["target_type"], **a["scores"],
                     "numeric_grounding_pct": a["deterministic"].get("numeric_grounding_pct"),
                     "ungrounded_numbers": a["deterministic"].get("ungrounded_numbers"),
                     "source_coverage_pct": a["source_coverage_pct"], "judge_model": a["judge_model"],
                     "generator_model": a["generator_model"],
                     "judge_independence": a["judge_independence"],
                     "verdict": "PASS" if verdict_ok else "NEEDS_REVISION"})
    return pd.DataFrame(rows, columns=SCORE_COLUMNS)


# EXECUTION
judge_audits = run_task1_evaluation(section_summaries, executive_summary, document_chunks, parsed_document)
save_json(judge_audits, CFG.task1_dir / "judge_audits.json")
judge_scores_df = audits_to_frame(judge_audits)
n_pending = sum(a.get("status") == "pending" for a in judge_audits)
if n_pending:
    print(f"⚠️ {n_pending} audits pending (quota, or RUN_JUDGE=0). Re-run later to complete them; "
          f"the {len(judge_scores_df)} completed audits are cached and will not be re-sent.")
judge_scores_df.to_csv(CFG.task1_dir / "judge_scores.csv", index=False)

print("=" * 80 + "\nTASK 1: LLM-AS-A-JUDGE AUDIT\n" + "=" * 80)
display(judge_scores_df[["target_id", *JUDGE_METRICS, "overall_score", "unsupported", "contradicted",
                         "numeric_grounding_pct", "source_coverage_pct", "verdict"]])
if not judge_scores_df.empty:
    print("Mean judge scores (1-5):\n", judge_scores_df.groupby("target_type")[JUDGE_METRICS + ["overall_score"]]
          .mean().round(2))
    print("\nJudge independence:", judge_scores_df.groupby(["generator_model", "judge_model"])
          ["judge_independence"].first().to_dict())

## 12. Task 1 outputs

In [ ]:
# ==============================================================================
# CELL 16: TASK 1 OUTPUT MANIFEST + READABLE EXPORT FOR THE WRITTEN REPORT
# ==============================================================================

def export_task1_markdown(exec_rec: Dict[str, Any], summaries: List[Dict[str, Any]],
                          scores: pd.DataFrame, path: Path) -> Path:
    """Human-readable record of every Task 1 output and its audit score (for human review)."""
    score_by_id = scores.set_index("target_id").to_dict("index") if not scores.empty else {}
    lines = [f"# Task 1 outputs — {REPORT_TITLE}", "", "## Executive summary", ""]
    lines += [f"- {b['text']} *(pp. {', '.join(map(str, b['pages']))})*" for b in exec_rec.get("bullets", [])]
    s = score_by_id.get("executive_summary")
    if s:
        lines += ["", f"Judge (1-5): factual {s['factual_alignment_score']} | consistency {s['consistency_score']} | "
                      f"completeness {s['completeness_score']} | verdict {s['verdict']}"]
    lines += ["", "## Section summaries", ""]
    for r in summaries:
        if r.get("status") != "success":
            lines += [f"### {r['title']}", f"FAILED: {r.get('error', '')}", ""]
            continue
        s = score_by_id.get(r["section_id"], {})
        lines += [f"### {r['title']} (pp. {r['start_page']}-{r['end_page']})", r["summary"], "",
                  f"*{r['word_count']} words | judge mean {s.get('overall_score', 'n/a')}/5 | "
                  f"verdict {s.get('verdict', 'n/a')}*", ""]
    path.write_text("\n".join(lines), encoding="utf-8")
    return path


def freeze_requirements(packages: List[str], path: Path) -> Path:
    """
    Pin the exact versions this run used. Commit the file: a different PyMuPDF or
    Camelot version can change extracted text by a character, which changes the
    prompt hash and turns every cached LLM response into a cache miss.
    """
    from importlib.metadata import version, PackageNotFoundError
    lines = []
    for pkg in packages:
        try:
            lines.append(f"{pkg}=={version(pkg)}")
        except PackageNotFoundError:
            lines.append(pkg)
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return path


# EXECUTION
freeze_requirements(PACKAGES, REQUIREMENTS)
export_task1_markdown(executive_summary, section_summaries, judge_scores_df, CFG.task1_dir / "task1_outputs.md")
print("✅ Task 1 artefacts:")
for f in sorted(CFG.task1_dir.iterdir()):
    print(f"   {f.name:28s} {f.stat().st_size / 1024:8.1f} KB")
print("\n" + USAGE.report())

## 12b. Dense vector index (sentence-transformers + FAISS IndexFlatL2)
Built once and shared: Task 2 uses it for hybrid (dense + keyword) evidence retrieval, Task 3 for RAG.

In [ ]:
# ==============================================================================
# CELL 17: LOCAL DENSE EMBEDDINGS + FAISS IndexFlatL2 VECTOR STORE
# (built once, before Task 2: used by Task 2 hybrid evidence retrieval and Task 3 RAG)
# ------------------------------------------------------------------------------
# * Embeddings: sentence-transformers all-MiniLM-L6-v2 (local, 384-d), run on
#   the same 400-token chunks used everywhere else (narrative + tables).
# * Vectors are L2-normalised, so IndexFlatL2 ranking equals cosine ranking
#   (||a-b||^2 = 2 - 2cos for unit vectors) while keeping the exact flat index.
# * Each chunk is embedded as "section title: text" so section context helps
#   short table chunks match questions.
# * Embeddings + index are cached on disk, keyed by model and chunk contents.
# Note: MiniLM truncates inputs at 256 word-pieces, so the tail of a 400-token
# chunk is not embedded; the 50-token overlap and table header repetition mitigate this
# ==============================================================================

EMBEDDING_MODEL = _env("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
RAG_DIR = CFG.output_dir / "task3"
RAG_DIR.mkdir(parents=True, exist_ok=True)


def embedding_text(chunk: Dict[str, Any]) -> str:
    return f"{chunk['section_title']}: {chunk['text']}"


def build_or_load_index(chunks: List[Dict[str, Any]], model_name: str) -> Tuple["faiss.Index", np.ndarray, "SentenceTransformer"]:
    encoder = SentenceTransformer(model_name)
    key = stable_hash({"model": model_name, "chunks": [(c["chunk_id"], c["text"]) for c in chunks]})
    emb_path, idx_path = RAG_DIR / f"embeddings_{key}.npy", RAG_DIR / f"faiss_{key}.index"
    if emb_path.exists() and idx_path.exists():
        logging.info("⚡ Loaded cached embeddings and FAISS index.")
        return faiss.read_index(str(idx_path)), np.load(emb_path), encoder
    vectors = encoder.encode([embedding_text(c) for c in chunks], batch_size=32, show_progress_bar=True,
                             normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    index = faiss.IndexFlatL2(vectors.shape[1])
    index.add(vectors)
    np.save(emb_path, vectors)
    faiss.write_index(index, str(idx_path))
    return index, vectors, encoder


def retrieve(question: str, k: int = 5) -> List[Dict[str, Any]]:
    """Top-k chunks for a question with L2 distance and implied cosine similarity."""
    q = ENCODER.encode([question], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    distances, ids = FAISS_INDEX.search(q, k)
    return [{**document_chunks[i], "rank": r + 1, "l2_distance": round(float(d), 4),
             "cosine_similarity": round(1 - float(d) / 2, 4)}
            for r, (d, i) in enumerate(zip(distances[0], ids[0])) if i >= 0]


# EXECUTION
FAISS_INDEX, CHUNK_VECTORS, ENCODER = build_or_load_index(document_chunks, EMBEDDING_MODEL)
save_json([{"row": i, "chunk_id": c["chunk_id"], "section_id": c["section_id"], "pages": c["pages"]}
           for i, c in enumerate(document_chunks)], RAG_DIR / "index_chunk_map.json")
print(f"✅ FAISS IndexFlatL2: {FAISS_INDEX.ntotal} vectors x {FAISS_INDEX.d} dims ({EMBEDDING_MODEL})")
for hit in retrieve("Bank of Mongolia inflation target band", k=3):
    print(f"  #{hit['rank']} {hit['chunk_id']} (p.{hit['page_number']}, cos={hit['cosine_similarity']})")

# Task 2 — Thematic extraction, indicators and structured output
### Prompt registry and schemas (Tasks 2-3)

In [ ]:
# ==============================================================================
# CELL 18: PROMPT REGISTRY & SCHEMAS — TASKS 2 AND 3
# ------------------------------------------------------------------------------
# Design rules carried over from Task 1: grounding rules in the SYSTEM prompt,
# schema-constrained JSON output, verbatim evidence for every extracted item so
#  explicit permission to say "not reported".
# ==============================================================================

PROMPT_VERSION_T23 = "t23-v1"

# ----------------------------------------------------------------- indicators
# Field definitions are part of the prompt:
INDICATOR_SPECS = {
    "gdp_growth_forecast_pct": "IMF staff projection of real GDP growth (percent) for the report year; a projection, not an outturn.",
    "headline_inflation_rate_pct": "Most recent reported headline CPI inflation (year-on-year, percent), actual not projected.",
    "central_bank_policy_rate_pct": "Current central bank policy rate level (percent) at the time of the report. Do not use rates from earlier years.",
    "non_performing_loan_ratio": "Banking-system non-performing loans as a percent of total loans.",
    "unemployment_rate_pct": "Unemployment rate (percent of labour force).",
    "tier_1_capital_ratio": "Banking-system Tier 1 capital to risk-weighted assets (percent). A total capital adequacy ratio is NOT Tier 1.",
}
# Additional financial-stability / banking / policy indicators the report quantifies in its
# tables or text (checked against the report). Stored in extended_indicators.json; the
# six-field target schema above is NOT changed. Reserve requirement ratios appear only in
# charts, so a correct extraction returns null with a reason (expected fallback).
EXTENDED_INDICATOR_SPECS = {
    "credit_growth_pct": "Year-on-year growth of credit to the economy (percent), most recent ACTUAL value in a table or text.",
    "nbfi_credit_growth_pct": "Year-on-year credit growth of nonbank financial institutions (NBFIs), percent, most recent reported.",
    "credit_to_gdp_ratio_pct": "Credit outstanding as a percent of GDP, most recent actual.",
    "household_debt_pct_gdp": "Household debt as a percent of GDP, most recent reported.",
    "broad_money_growth_pct": "Broad money growth (percent change), most recent actual.",
    "reserve_requirement_ratio_pct": "Bank reserve requirement ratio in force (percent; MNT or FX). Not reserve-money growth.",
    "inflation_target_pct": "Central value of the Bank of Mongolia inflation target (percent); put the band (e.g. ±2) in unit.",
    "policy_rate_corridor_pct": "Half-width of the policy rate corridor (percentage points), e.g. 2 for a ±2 percent corridor.",
    "general_government_debt_pct_gdp": "General government (public) debt, percent of GDP, most recent actual year.",
    "external_debt_pct_gdp": "External debt, percent of GDP, most recent actual year.",
    "overall_fiscal_balance_pct_gdp": "General government overall balance (IMF definition), percent of GDP, most recent actual.",
    "current_account_balance_pct_gdp": "Current account balance, percent of GDP, most recent actual (negative = deficit).",
    "gross_reserves_months_of_imports": "Gross official reserves in months of imports, most recent actual.",
}
ALL_INDICATOR_SPECS = {**INDICATOR_SPECS, **EXTENDED_INDICATOR_SPECS}
INDICATOR_FIELD = Literal[tuple(ALL_INDICATOR_SPECS)]   # type: ignore[valid-type]


class IndicatorValue(BaseModel):
    field: INDICATOR_FIELD
    value: Optional[float] = Field(description="Exact number from SOURCE, or null if not reported.")
    unit: str
    reference_period: str = Field(description="Period the value refers to, e.g. '2025', 'June 2025'; '' if null.")
    status: Literal["actual", "projection", "not_reported"]
    source_chunk_id: str = Field(description="chunk id of the evidence, '' if null.")
    source_quote: str = Field(description="Verbatim SOURCE excerpt (<=30 words) containing the value, '' if null.")
    note: str = Field(description="If null: why the report does not provide it (one sentence).")


class IndicatorExtraction(BaseModel):
    country_name: str
    indicators: List[IndicatorValue]


SYSTEM_INDICATORS = f"""You extract macro-financial indicators from {REPORT_TITLE}.
Rules:
1. Use only SOURCE. Never use outside knowledge or estimate, compute, round or convert a number.
2. A value is valid only if it appears verbatim in SOURCE and matches the field DEFINITION (concept, unit, period and status).
3. If SOURCE has no value matching the definition, return value null, status "not_reported", and a one-sentence note explaining what the report does say (e.g. discussed only qualitatively). A null with a correct note is a correct answer; a near-miss substitute is a wrong answer.
4. source_quote must be copied exactly from SOURCE and contain the number; source_chunk_id is the id in the chunk header.
5. For a value read from a TABLE chunk, source_quote must be "ROW LABEL | COLUMN HEADER | VALUE", each part copied from that table (e.g. "Current account balance | 2024 | -10.5")."""

USER_INDICATORS = """Return the country name and one entry for EACH of these fields:
{definitions}

<source>
{context}
</source>"""

# ------------------------------------------------- supplementary indicators
# Quantities the report states in its narrative beyond the six-field target schema
# (e.g. credit growth, household debt, reserve cover, target bands). The target
# schema itself is NOT extended: these go to a separate file.
SUPPLEMENTARY_THEMES = ("Inflation & Prices", "Interest Rates & Credit", "Housing & Real Estate",
                        "Banking Capital & Liquidity", "Sovereign Debt", "Labor Market Dynamics",
                        "Systemic Financial Risk", "Growth & Output", "External Sector & Reserves",
                        "Fiscal Position")


class SupplementaryIndicator(BaseModel):
    indicator: str = Field(description="Specific name, e.g. 'Bank credit growth (y/y)'.")
    value: float = Field(description="The number exactly as written in SOURCE.")
    unit: str = Field(description="e.g. 'percent', 'percent of GDP', 'months of imports', 'US$ billion'.")
    reference_period: str
    status: Literal["actual", "projection", "target_or_limit", "authorities_view"]
    theme: Literal[SUPPLEMENTARY_THEMES]                   # type: ignore[valid-type]
    source_chunk_id: str
    source_quote: str = Field(description="Verbatim SOURCE excerpt (<=30 words) containing the number.")


class SupplementaryIndicators(BaseModel):
    indicators: List[SupplementaryIndicator]


SYSTEM_SUPPLEMENTARY = f"""You extract quantitative macro-financial indicators stated in the narrative of {REPORT_TITLE}.
Rules:
1. Use only SOURCE. Copy each number exactly as written; never compute, round, convert or combine numbers.
2. One indicator = one number with its unit, reference period and status. source_quote is copied exactly from SOURCE and contains the number; source_chunk_id is the id in that chunk's header.
3. status: actual (already happened), projection (IMF staff forecast), target_or_limit (policy target, band, ceiling or regulatory limit; for a band such as 6±2 report the central value and put the band in unit), authorities_view (stated as the authorities' position).
4. Skip numbers that are only years, dates, page/table/footnote references or counts of recommendations."""

USER_SUPPLEMENTARY = """TASK: List up to {k} quantitative indicators in SOURCE that matter most for financial stability and the macroeconomic outlook, most decision-relevant first. Assign each to one theme.
Exclude these six fields, which are extracted separately: {core_fields}.

<source>
{context}
</source>"""

# --------------------------------------------------------------- risk insights
class Insight(BaseModel):
    item: str = Field(description="Short label, at most 8 words.")
    explanation: str = Field(description="At most 30 words, grounded in SOURCE.")
    evidence: str = Field(description="Verbatim SOURCE excerpt, at most 25 words.")
    pages: List[int]


class RiskInsights(BaseModel):
    strengths: List[Insight]
    vulnerabilities: List[Insight]

    @field_validator("strengths", "vulnerabilities")
    @classmethod
    def _five_to_seven(cls, v: List[Insight]) -> List[Insight]:
        if not 5 <= len(v) <= 7:
            raise ValueError(f"need 5-7 items, got {len(v)}")
        return v


SYSTEM_INSIGHTS = f"""You are an IMF macro-financial analyst reading the narrative of {REPORT_TITLE}.
Rules: use only SOURCE; each item must be stated or directly evidenced in SOURCE; evidence is copied verbatim;
strengths are national growth drivers or buffers the report credits; vulnerabilities are risks or weaknesses the report flags.
Do not list a policy recommendation as a strength."""

USER_INSIGHTS = """TASK: Extract 5-7 macroeconomic growth drivers / strengths and 5-7 systemic vulnerabilities / challenges for the country, most material first. Merge duplicates. Cite PDF pages from the [p.N] markers.

<source>
{context}
</source>"""

# ------------------------------------------------------------- trend selection
SYSTEM_TRENDS = f"""You select time series from the statistical tables of {REPORT_TITLE} for longitudinal plots of economic and financial-stability trends.
Choose only from the CATALOGUE names given; never invent a series."""

USER_TRENDS = """From the CATALOGUE of multi-year series parsed from the report's tables (year:value pairs), select up to {k} series that best show historical and projected trends in inflation, interest rates, credit, growth, public debt and external buffers.
For each give: a one-sentence reason; trend_direction over the listed years (rising, falling, stable or volatile); and peak_year = the year of the highest listed value.

<catalogue>
{catalogue}
</catalogue>"""


def make_trend_schema(series_names: List[str]) -> Type[BaseModel]:
    """Schema whose series_name is an enum of the catalogue: the model cannot invent a series."""
    Name = Literal[tuple(series_names)]                  # type: ignore[valid-type]

    class TrendPick(BaseModel):
        series_name: Name
        reason: str
        trend_direction: Literal["rising", "falling", "stable", "volatile"]
        peak_year: int

    class TrendSelection(BaseModel):
        selected: List[TrendPick]

    return TrendSelection


# ----------------------------------------------------------------------- RAG
class RAGAnswer(BaseModel):
    answer: str = Field(description="At most 120 words, only from CONTEXT.")
    cited_chunk_ids: List[str]
    insufficient_context: bool


class ClosedBookAnswer(BaseModel):
    answer: str = Field(description="At most 120 words.")


SYSTEM_RAG = f"""You answer questions about {REPORT_TITLE} using ONLY the retrieved CONTEXT chunks.
Rules: every sentence must be supported by CONTEXT; cite the chunk ids you used; copy numbers exactly with unit and period;
if CONTEXT does not answer the question, say so and set insufficient_context true. Never fill gaps from memory."""

USER_RAG = """QUESTION: {question}

<context>
{context}
</context>"""

class RAGTriadAudit(BaseModel):
    """RAG Triad (TruLens) + the assignment's quality criteria, all on 1-5 scales."""
    context_relevance_score: int = Field(description="1-5: is the retrieved CONTEXT relevant to the QUESTION?")
    groundedness_score: int = Field(description="1-5: is every claim in ANSWER supported by CONTEXT?")
    answer_relevance_score: int = Field(description="1-5: does ANSWER directly address the QUESTION?")
    factual_alignment_score: int = Field(description="1-5: are numbers, units, periods and status exactly as in CONTEXT?")
    completeness_score: int = Field(description="1-5: does ANSWER cover what CONTEXT says that answers the QUESTION?")
    consistency_score: int = Field(description="1-5: is ANSWER internally coherent?")
    unsupported_claims: List[str] = Field(description="Claims in ANSWER not supported by CONTEXT; [] if none.")
    justification: str = Field(description="One consolidated justification covering all six scores.")

    @field_validator("context_relevance_score", "groundedness_score", "answer_relevance_score",
                     "factual_alignment_score", "completeness_score", "consistency_score")
    @classmethod
    def _score_in_range(cls, v: int) -> int:
        if not 1 <= v <= 5:
            raise ValueError("score must be an integer from 1 to 5")
        return v


RAG_TRIAD_METRICS = ["context_relevance_score", "groundedness_score", "answer_relevance_score"]
RAG_QUALITY_METRICS = ["factual_alignment_score", "completeness_score", "consistency_score"]

SYSTEM_RAG_JUDGE = """You are an independent evaluator of a retrieval-augmented QA system over an IMF country report (RAG Triad).
Judge only against the QUESTION and the retrieved CONTEXT; your own knowledge is irrelevant. Be strict: a number is supported only if the same number, unit and period appear in CONTEXT."""

USER_RAG_JUDGE = """Score each criterion 1-5 (5 = fully satisfied, 3 = partly, 1 = not at all) and return one JSON object.
1. context_relevance_score: is the retrieved CONTEXT relevant to the QUESTION? (judges the retriever)
2. groundedness_score: is the ANSWER entirely supported by CONTEXT? 5 = every claim supported; 3 = one unsupported claim; 1 = mostly unsupported. (judges hallucination)
3. answer_relevance_score: does the ANSWER directly address the QUESTION? (judges the generator)
4. factual_alignment_score: numbers, units, periods and status (outturn / projection / recommendation) exactly as in CONTEXT.
5. completeness_score: coverage of what CONTEXT says that answers the QUESTION.
6. consistency_score: internal coherence of the ANSWER.
List every unsupported claim, then give one consolidated justification for all scores.

QUESTION: {question}

<context>
{context}
</context>

<answer>
{answer}
</answer>"""

SYSTEM_CLOSED_BOOK = "You are an economist answering questions about Mongolia's economy from your own knowledge."
USER_CLOSED_BOOK = "QUESTION (about the IMF's 2025 Article IV consultation with Mongolia): {question}"

print(f"✅ Task 2/3 prompt registry {PROMPT_VERSION_T23} loaded")

## 13. Thematic distribution (7 themes)

In [ ]:
# ==============================================================================
# CELL 19: TASK 2 — THEMATIC DISTRIBUTION ACROSS THE 7 FINANCIAL-STABILITY THEMES
# ------------------------------------------------------------------------------
# Method (transparent and reproducible, no LLM): a curated lexicon per theme is
# counted with word-boundary regexes over every narrative chunk's NEW text (the
# 50-token overlaps are not double counted). Each term belongs to exactly one
# theme. Two measures:
#   * mentions        — total term occurrences per theme (and per 10k words)
#   * dominant_chunks — chunks whose highest-scoring theme is that theme
# Per-section counts form a wide matrix that Task 4 reshapes with pd.melt.
# ==============================================================================

THEME_LEXICON: Dict[str, List[str]] = {
    "Inflation & Prices": [r"inflation\w*", r"disinflation", r"consumer prices?", r"CPI", r"deflator",
                           r"price pressures?", r"target band", r"electricity tariffs?"],
    "Interest Rates & Credit": [r"interest rates?", r"policy rate", r"credit", r"lending", r"loans?",
                                r"DSTI", r"debt[- ]service[- ]to[- ]income", r"reserve requirements?", r"household debt"],
    "Housing & Real Estate": [r"housing", r"mortgages?", r"real estate", r"property", r"house prices?",
                              r"construction"],
    "Banking Capital & Liquidity": [r"banks?", r"banking", r"capital adequacy", r"capitali[sz]ed", r"recapitali[sz]ation",
                                    r"liquidity", r"deposits?", r"NBFIs?", r"nonbank\w*", r"FX swaps?"],
    "Sovereign Debt": [r"public debt", r"government debt", r"external debt", r"sovereign", r"Eurobonds?",
                       r"debt sustainability", r"debt ceiling", r"rollover", r"gross financing needs?", r"DBM"],
    "Labor Market Dynamics": [r"labou?r", r"employment", r"unemployment", r"wages?", r"jobs?", r"workers?"],
    "Systemic Financial Risk": [r"systemic", r"financial stability", r"contagion", r"spillovers?", r"macroprudential",
                                r"stress tests?", r"vulnerabilit(?:y|ies)", r"regulatory arbitrage", r"insolvency"],
}
THEME_PATTERNS = {t: re.compile(r"\b(?:" + "|".join(terms) + r")\b", re.I) for t, terms in THEME_LEXICON.items()}


def unique_text(chunk: Dict[str, Any], previous: Optional[Dict[str, Any]], overlap_tokens: int) -> str:
    """Drop the leading overlap a chunk shares with its predecessor in the same section."""
    if previous is None or previous["section_id"] != chunk["section_id"] or chunk["chunk_type"] != "narrative":
        return chunk["text"]
    offsets = tokenizer(chunk["text"], add_special_tokens=False, return_offsets_mapping=True,
                        verbose=False)["offset_mapping"]
    return chunk["text"][offsets[overlap_tokens][0]:] if len(offsets) > overlap_tokens else ""


def count_themes(text: str) -> Dict[str, int]:
    return {theme: len(rx.findall(text)) for theme, rx in THEME_PATTERNS.items()}


def thematic_distribution(chunks: List[Dict[str, Any]], sections_by_id: Dict[str, Dict]) -> Dict[str, Any]:
    narrative = [c for c in chunks if c["chunk_type"] == "narrative"
                 and sections_by_id[c["section_id"]]["kind"] not in ("toc", "front_matter")]
    per_section: Dict[str, Counter] = {}
    dominant: Counter = Counter()
    total_words, prev = 0, None
    for c in narrative:
        text = unique_text(c, prev, CFG.chunk_overlap_tokens)
        prev = c
        counts = count_themes(text)
        per_section.setdefault(c["section_id"], Counter()).update(counts)
        total_words += len(text.split())
        if max(counts.values()) > 0:
            dominant[max(counts, key=counts.get)] += 1
    totals = sum(per_section.values(), Counter())
    rows = [{"theme": t, "mentions": totals[t],
             "mentions_per_10k_words": round(10_000 * totals[t] / max(total_words, 1), 1),
             "share_pct": round(100 * totals[t] / max(sum(totals.values()), 1), 1),
             "dominant_chunks": dominant[t]} for t in THEME_LEXICON]
    by_section = [{"section_id": sid, "section_title": sections_by_id[sid]["title"], **{t: cnt[t] for t in THEME_LEXICON}}
                  for sid, cnt in per_section.items()]
    return {"method": "lexicon_regex_count", "lexicon": THEME_LEXICON, "words_scanned": total_words,
            "chunks_scanned": len(narrative), "themes": rows, "by_section": by_section}


# EXECUTION
TASK2_DIR = CFG.output_dir / "task2"
TASK2_DIR.mkdir(parents=True, exist_ok=True)
thematic = thematic_distribution(document_chunks, SECTIONS_BY_ID)
save_json(thematic, TASK2_DIR / "thematic_distribution.json")
theme_df = pd.DataFrame(thematic["themes"]).sort_values("mentions", ascending=False)
theme_by_section_wide = pd.DataFrame(thematic["by_section"])          # wide: section x theme
display(theme_df)
print(f"✅ Scanned {thematic['chunks_scanned']} narrative chunks / {thematic['words_scanned']:,} words")

## 14. Qualitative risk insights

In [ ]:
# ==============================================================================
# CELL 20: TASK 2 — QUALITATIVE RISK INSIGHTS (5-7 strengths, 5-7 vulnerabilities)
# ------------------------------------------------------------------------------
# Source = the report's narrative overview and risk chapters (configurable).
# Every item carries a verbatim evidence quote, checked in code against the
# source text; the count limits (5-7) are enforced by the schema validator.
# ==============================================================================

INSIGHT_SECTIONS = ("key_issues", "context", "recent_developments", "outlook_and_risks",
                    "d_macroprudential_and_financial_sector_policies", "staff_appraisal", "annex_III")


def normalise_for_match(text: str) -> str:
    """Lowercase, unify quotes/dashes/whitespace so verbatim checks ignore PDF typography."""
    text = text.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    text = re.sub(r"[‒–—−-]", "-", text.lower())
    return re.sub(r"\s+", " ", text).strip()


def quote_is_verbatim(quote: str, source: str) -> bool:
    q = normalise_for_match(quote).strip(" .\"'")
    return bool(q) and q in normalise_for_match(source)


def extract_risk_insights(parsed: Dict[str, Any]) -> Dict[str, Any]:
    used = [sid for sid in INSIGHT_SECTIONS if sid in SECTIONS_BY_ID]
    context = "\n\n".join(f"## {SECTIONS_BY_ID[sid]['title']}\n{build_section_context(parsed, sid)}" for sid in used)
    rec = llm_structured(task="risk_insights", system=SYSTEM_INSIGHTS, prompt=USER_INSIGHTS.format(context=context),
                         schema=RiskInsights, prompt_version=PROMPT_VERSION_T23)
    out = {}
    for group in ("strengths", "vulnerabilities"):
        out[group] = [{**i, "evidence_verbatim": quote_is_verbatim(i["evidence"], context)} for i in rec["parsed"][group]]
    out["checks"] = {g: {"n_items": len(out[g]), "evidence_verbatim_pct":
                         round(100 * sum(i["evidence_verbatim"] for i in out[g]) / max(1, len(out[g])), 1)}
                     for g in ("strengths", "vulnerabilities")}
    return {"status": "success", "source_sections": used, "context_tokens": count_tokens(context),
            "model_used": rec["model_used"], **out}, context


def insights_to_text(ins: Dict[str, Any]) -> str:
    lines = []
    for group in ("strengths", "vulnerabilities"):
        lines += [f"{group.upper()[:-1]}: {i['item']} — {i['explanation']}" for i in ins[group]]
    return "\n".join(lines)


# EXECUTION
try:
    risk_insights, insights_context = extract_risk_insights(parsed_document)
    try:
        audit = run_judge("risk_insights", "list of 5-7 strengths and 5-7 vulnerabilities extracted from the report",
                          insights_to_text(risk_insights), insights_context, reference="Same as SOURCE.",
                          generator_model=risk_insights["model_used"])
        risk_insights["judge"] = {**audit["scores"], "judge_model": audit["judge_model"],
                                  "judge_independence": audit["judge_independence"],
                                  "claims": audit["audit"]["claims"]}
    except QuotaExhaustedError:               # insights are kept; the audit runs on the next session
        logging.error("Judge paused (quota or RUN_JUDGE=0); risk-insight audit pending.")
        risk_insights["judge"] = {"status": "pending"}
except Exception as exc:
    logging.error(f"Risk insights failed: {exc}")
    risk_insights = {"status": "error", "error": str(exc)[:500], "strengths": [], "vulnerabilities": []}
save_json(risk_insights, TASK2_DIR / "risk_insights.json")

for group in ("strengths", "vulnerabilities"):
    print(f"\n{group.upper()}")
    for i in risk_insights[group]:
        flag = "✓" if i["evidence_verbatim"] else "✗ evidence not verbatim"
        print(f"  • {i['item']}: {i['explanation']} (pp. {', '.join(map(str, i['pages']))}) {flag}")
print("\nChecks:", risk_insights.get("checks"), "| Judge:", {k: risk_insights.get("judge", {}).get(k)
      for k in ("factual_alignment_score", "consistency_score", "completeness_score", "judge_model")})

## 15. Core indicator JSON, fallbacks and model comparison

In [ ]:
# ==============================================================================
# CELL 21: TASK 2 — CORE INDICATOR JSON, FALLBACK CAPTURE, MODEL COMPARISON
# ------------------------------------------------------------------------------

#    Anything unverified/absent -> fallback + a recorded reason. In memory the
#    value is None (never plotted or averaged); JSON files get the clean default
#    FALLBACK_SENTINEL (-1.0, the brief's example) plus status "not_reported"
#    and the reason, so no output field is left as an unexplained null.
#    The judge LLM audits each model's raw extraction against the same evidence.
# Target schema written to outputs/task2/core_indicators.json exactly as specified;
# 13 additional financial-stability/banking/policy indicators -> extended_indicators.json.
# Red-team test: the judge must catch a deliberately tampered extraction.
# ==============================================================================

FALLBACK_VALUE = None          # in-memory: None can never be plotted or averaged as a real rate
FALLBACK_SENTINEL = float(_env("FALLBACK_SENTINEL", "-1.0"))   # written to JSON outputs only
DENSE_CHUNKS_PER_FIELD = int(_env("DENSE_CHUNKS_PER_FIELD", "2"))
# Model comparison: the generator and the judge model each extract the indicators
# (one call each). Code-based verification (verbatim quote + number + recency) is
# model-independent; the judge audit of the judge model's own extraction is labelled
# self_evaluation, so compare models primarily on the verified percentage.
EXTRACTION_MODELS = tuple(dict.fromkeys(m.strip() for m in _env(
    "EXTRACTION_MODELS", ",".join([CFG.gemini_models[0], CFG.gemini_judge_models[0]])).split(",") if m.strip()))
EVIDENCE_PATTERNS = {
    "gdp_growth_forecast_pct": r"real GDP growth|growth (?:is|was) projected|growth in 20\d\d",
    "headline_inflation_rate_pct": r"inflation|consumer prices",
    "central_bank_policy_rate_pct": r"policy rate",
    "non_performing_loan_ratio": r"non-?\s?performing|\bNPLs?\b",
    "unemployment_rate_pct": r"unemploy",
    "tier_1_capital_ratio": r"tier[- ]1|capital adequacy|capitali[sz]ed|capital ratio",
    # extended indicators (tables + narrative)
    "credit_growth_pct": r"credit growth",
    "nbfi_credit_growth_pct": r"NBFI|nonbank financial",
    "credit_to_gdp_ratio_pct": r"credit to GDP|credit-to-GDP",
    "household_debt_pct_gdp": r"household debt",
    "broad_money_growth_pct": r"broad money",
    "reserve_requirement_ratio_pct": r"reserve requirement",
    "inflation_target_pct": r"target band|inflation target|6±2|6 ± 2",
    "policy_rate_corridor_pct": r"policy rate corridor|corridor",
    "general_government_debt_pct_gdp": r"general government debt|public debt",
    "external_debt_pct_gdp": r"external debt",
    "overall_fiscal_balance_pct_gdp": r"overall balance",
    "current_account_balance_pct_gdp": r"current account",
    "gross_reserves_months_of_imports": r"months of imports|gross official reserves",
}
CURRENT_STATE_FIELDS = {"headline_inflation_rate_pct", "central_bank_policy_rate_pct",
                        "non_performing_loan_ratio", "unemployment_rate_pct", "tier_1_capital_ratio",
                        "credit_growth_pct", "nbfi_credit_growth_pct", "credit_to_gdp_ratio_pct",
                        "household_debt_pct_gdp", "broad_money_growth_pct", "reserve_requirement_ratio_pct",
                        "general_government_debt_pct_gdp", "external_debt_pct_gdp", "overall_fiscal_balance_pct_gdp",
                        "current_account_balance_pct_gdp", "gross_reserves_months_of_imports"}
RED_TEAM_JUDGE = _env("RED_TEAM_JUDGE", "1") == "1"     # one extra judge call: deceptive extraction test
EVIDENCE_CHUNKS_PER_FIELD = 4
REPORT_YEAR = int(re.search(r"(20\d\d) Article IV", document_metadata.get("title") or "").group(1)) \
    if re.search(r"(20\d\d) Article IV", document_metadata.get("title") or "") else None


def retrieve_evidence(chunks: List[Dict[str, Any]]) -> Tuple[List[Dict[str, Any]], Dict[str, int]]:
    """Hybrid retrieval per indicator: dense FAISS top-k on the indicator definition
    UNION keyword-pattern top-k; plus whole-report keyword mention counts."""
    selected, mentions, found_by = {}, {}, {}
    for field, pattern in EVIDENCE_PATTERNS.items():
        rx = re.compile(pattern, re.I)
        scored = [(len(rx.findall(c["text"])), c) for c in chunks]
        mentions[field] = sum(s for s, _ in scored)
        for score, c in sorted(scored, key=lambda x: -x[0])[:EVIDENCE_CHUNKS_PER_FIELD]:
            if score > 0:
                selected[c["chunk_id"]] = c
                found_by.setdefault(c["chunk_id"], set()).add("keyword")
        query = f"{field.replace('_', ' ')}: {ALL_INDICATOR_SPECS[field]}"
        for hit in retrieve(query, DENSE_CHUNKS_PER_FIELD):           # Cell 17 dense index
            selected[hit["chunk_id"]] = chunks_by_id[hit["chunk_id"]]
            found_by.setdefault(hit["chunk_id"], set()).add("dense")
    ordered = [c for c in chunks if c["chunk_id"] in selected]          # reading order
    evidence_retrieval_log.extend({"chunk_id": cid, "found_by": " + ".join(sorted(m))}
                                  for cid, m in found_by.items())
    return ordered, mentions


def evidence_context(evidence: List[Dict[str, Any]]) -> str:
    return "\n\n".join(f"### [{c['chunk_id']} | p.{c['page_number']}]\n{c['text']}" for c in evidence)


def numbers_in(text: str) -> List[float]:
    return [float(n.replace(",", "")) for n in re.findall(r"\d+(?:,\d{3})*(?:\.\d+)?", text.replace("½", ".5"))]


def latest_year(period: str) -> Optional[int]:
    years = [int(y) for y in re.findall(r"\b(20\d\d)\b", period or "")]
    return max(years) if years else None


def _norm_cell(text: str) -> str:
    return re.sub(r"[^a-z0-9.%+-]+", " ", str(text).lower().replace("−", "-")).strip()


def table_quote_ok(quote: str, value: float, chunk_text: str) -> bool:
    """'ROW LABEL | COLUMN HEADER | VALUE' must match a real row/column cell of the markdown table chunk."""
    parts = [p.strip() for p in quote.split("|") if p.strip()]
    if len(parts) < 3:
        return False
    label, column, cell_value = _norm_cell(parts[0]), _norm_cell(parts[-2]), parts[-1]
    lines = [l for l in chunk_text.splitlines() if l.strip().startswith("|")]
    rows = [[c.strip() for c in l.strip().strip("|").split("|")] for l in lines
            if not re.fullmatch(r"[|\s:-]+", l.strip())]
    if len(rows) < 2:
        return False
    header = [_norm_cell(h) for h in rows[0]]
    cols = [j for j, h in enumerate(header) if column and column in h]
    for row in rows[1:]:
        if row and label and (label in _norm_cell(row[0]) or _norm_cell(row[0]) in label):
            for j in cols:
                if j < len(row) and numbers_in(row[j]) and abs(numbers_in(row[j])[0] - abs(value)) < 1e-9 \
                        and numbers_in(cell_value) and abs(numbers_in(cell_value)[0] - abs(value)) < 1e-9:
                    return True
    return False


def verify_indicator(item: Dict[str, Any], chunks_by_id: Dict[str, Dict], context: str,
                     mentions: int) -> Dict[str, Any]:
    """Apply verbatim, number-in-quote and recency checks; decide final value or fallback."""
    v, field = item.get("value"), item["field"]
    out = {**item, "mentions_in_report": mentions, "final_value": FALLBACK_VALUE, "fallback_reason": ""}
    if v is None:
        out["fallback_reason"] = (f"Not reported: {item.get('note') or 'no matching value'} "
                                  f"(pattern mentions in report: {mentions}).")
        return out
    chunk = chunks_by_id.get(item.get("source_chunk_id"), {})
    source = chunk.get("text", context)
    quote = item.get("source_quote", "")
    in_text = quote_is_verbatim(quote, source)
    in_table = chunk.get("chunk_type") == "table" and table_quote_ok(quote, v, source)
    checks = {"quote_verbatim": in_text or in_table,
              "number_in_quote": any(abs(n - abs(v)) < 1e-9 for n in numbers_in(item.get("source_quote", "")))}
    year = latest_year(item.get("reference_period", ""))
    checks["recent_enough"] = not (field in CURRENT_STATE_FIELDS and REPORT_YEAR and year and year < REPORT_YEAR - 1)
    out["checks"] = {**checks, "matched_as": "table_cell" if in_table and not in_text else "text"}
    if all(checks.values()):
        out["final_value"] = float(v)
    elif not checks["recent_enough"]:
        out["fallback_reason"] = (f"Stale: the report only gives {v} for {item.get('reference_period')}; "
                                  f"no level for {REPORT_YEAR - 1}-{REPORT_YEAR} is stated.")
    else:
        failed = [k for k, ok in checks.items() if not ok]
        out["fallback_reason"] = f"Unverifiable extraction ({', '.join(failed)} failed); rejected."
    return out


def indicators_candidate_text(items: List[Dict[str, Any]]) -> str:
    rows = [f"{i['field']} = {i['value']} {i['unit']} ({i['reference_period']}, {i['status']})"
            for i in items if i.get("value") is not None]
    return "\n".join(rows)


def run_indicator_extraction(model: str, context: str, chunks_by_id: Dict[str, Dict],
                             mentions: Dict[str, int]) -> Dict[str, Any]:
    definitions = "\n".join(f"- {f}: {d}" for f, d in ALL_INDICATOR_SPECS.items())
    rec = llm_structured(task=f"indicators:{model}", system=SYSTEM_INDICATORS,
                         prompt=USER_INDICATORS.format(definitions=definitions, context=context),
                         schema=IndicatorExtraction, models=(model,), prompt_version=PROMPT_VERSION_T23)
    raw = {i["field"]: i for i in rec["parsed"]["indicators"]}
    items = [verify_indicator(raw.get(f, {"field": f, "value": None, "note": "field missing from model output"}),
                              chunks_by_id, context, mentions[f]) for f in ALL_INDICATOR_SPECS]
    result = {"model": model, "latency_s": rec["latency_s"], "cached": rec["cached"],
              "country_name": rec["parsed"]["country_name"], "indicators": items}
    candidate = indicators_candidate_text(items)
    if candidate:
        try:
            audit = run_judge(f"indicators:{model}", "extracted numerical indicators (field = value unit (period, status))",
                              candidate, context, reference="FIELD DEFINITIONS:\n" + definitions, generator_model=model)
            result["judge"] = {**audit["scores"], "judge_model": audit["judge_model"],
                               "judge_independence": audit["judge_independence"], "claims": audit["audit"]["claims"],
                               "justification": audit["audit"]["justification"]}
        except QuotaExhaustedError:           # keep the (code-verified) extraction; audit later from cache
            logging.error(f"Judge paused (quota or RUN_JUDGE=0); indicator audit for {model} pending.")
            result["judge"] = {"status": "pending"}
    return result


def to_target_schema(result: Dict[str, Any]) -> Dict[str, Any]:
    """Exactly the assignment's JSON schema (floats or null)."""
    country = result.get("country_name") or (document_metadata.get("title") or "").split(":")[0]
    return {"country_name": country, **{i["field"]: i["final_value"] for i in result["indicators"]
                                        if i["field"] in INDICATOR_SPECS}}


def core_for_json(core: Dict[str, Any]) -> Dict[str, Any]:
    """Target schema for disk: missing values get the clean default, never a bare null."""
    return {k: (FALLBACK_SENTINEL if v is None else v) for k, v in core.items()}


def core_fallback_reasons(result: Dict[str, Any]) -> Dict[str, Any]:
    return {i["field"]: {"value": FALLBACK_SENTINEL, "status": "not_reported",
                         "fallback_reason": i["fallback_reason"] or "Not found in the report."}
            for i in result["indicators"] if i["field"] in INDICATOR_SPECS and i["final_value"] is None}


def extended_for_json(extended: Dict[str, Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    """Extended indicators for disk: fallbacks become {value: -1.0, status, reason}; no null fields."""
    return {f: ({"value": FALLBACK_SENTINEL, "status": "not_reported",
                 "fallback_reason": e["fallback_reason"] or "Not found in the report."}
                if e["value"] is None else {k: v for k, v in e.items() if v is not None})
            for f, e in extended.items()}


def to_extended(result: Dict[str, Any]) -> Dict[str, Any]:
    """Additional indicators with evidence (or fallback reason) — separate from the target schema."""
    return {i["field"]: {"value": i["final_value"], "unit": i.get("unit"), "reference_period": i.get("reference_period"),
                         "status": i.get("status"), "page": chunks_by_id.get(i.get("source_chunk_id"), {}).get("page_number"),
                         "source_quote": i.get("source_quote") if i["final_value"] is not None else None,
                         "fallback_reason": i["fallback_reason"] or None}
            for i in result["indicators"] if i["field"] in EXTENDED_INDICATOR_SPECS}


def red_team_judge(result: Dict[str, Any], context: str, definitions: str) -> Optional[Dict[str, Any]]:
    """Tutorial 4 Ex.5: feed the judge a deceptive version (one decimal shifted, one unit swapped)."""
    good = [i for i in result["indicators"] if i["final_value"] is not None]
    if len(good) < 2:
        return None
    tampered = [dict(i) for i in good]
    tampered[0]["value"] = round(tampered[0]["value"] + 0.3, 1)                       # subtle decimal change
    tampered[1]["unit"] = "percent of GDP" if "GDP" not in str(tampered[1]["unit"]) else "percent"
    audit = run_judge(f"redteam:{result['model']}", "extracted numerical indicators (field = value unit (period, status))",
                      indicators_candidate_text(tampered), context, reference="FIELD DEFINITIONS:\n" + definitions,
                      generator_model="red_team_tampered")
    caught = [c for c in audit["audit"]["claims"] if c["verdict"] != "SUPPORTED"]
    return {"tampered_fields": [tampered[0]["field"], tampered[1]["field"]], **audit["scores"],
            "claims_flagged": [c["claim"] for c in caught], "justification": audit["audit"]["justification"],
            "caught_both": len(caught) >= 2 and audit["scores"]["factual_alignment_score"] <= 3}


def comparison_row(result: Dict[str, Any]) -> Dict[str, Any]:
    by_group = {"core": INDICATOR_SPECS, "extended": EXTENDED_INDICATOR_SPECS}
    row = {"model": result["model"]}
    for group, specs in by_group.items():
        items = [i for i in result["indicators"] if i["field"] in specs]
        extracted = [i for i in items if i.get("value") is not None]
        verified = [i for i in items if i["final_value"] is not None]
        row.update({f"{group}_extracted": len(extracted), f"{group}_verified": len(verified),
                    f"{group}_verified_pct": round(100 * len(verified) / len(extracted), 1) if extracted else None,
                    f"{group}_fallbacks": len(specs) - len(verified)})
    j = result.get("judge", {})
    row.update({f"judge_{m}": j.get(m) for m in ("factual_alignment_score", "consistency_score", "completeness_score")})
    row.update({"judge_contradicted": j.get("contradicted"), "judge_independence": j.get("judge_independence"),
                "latency_s": result["latency_s"]})
    return row


# EXECUTION
chunks_by_id = {c["chunk_id"]: c for c in document_chunks}
evidence_retrieval_log: List[Dict[str, str]] = []
evidence, concept_mentions = retrieve_evidence(document_chunks)
indicator_context = evidence_context(evidence)
retrieval_mix = pd.DataFrame(evidence_retrieval_log)["found_by"].value_counts().to_dict()
print(f"Hybrid evidence retrieval: {len(evidence)} chunks — {retrieval_mix}")
extraction_results = []
for model in EXTRACTION_MODELS:
    try:
        extraction_results.append(run_indicator_extraction(model, indicator_context, chunks_by_id, concept_mentions))
    except Exception as exc:
        logging.error(f"Indicator extraction failed for {model}: {exc}")
        extraction_results.append({"model": model, "status": "error", "error": str(exc)[:300]})

successful = [r for r in extraction_results if "indicators" in r]
if not successful:
    raise RuntimeError("No extraction model succeeded; see errors above.")
core_indicators = to_target_schema(successful[0])                     # primary model = first that succeeded
extended_indicators = to_extended(successful[0])
red_team = None
if RED_TEAM_JUDGE:
    try:
        red_team = red_team_judge(successful[0], indicator_context,
                                  "\n".join(f"- {f}: {d}" for f, d in ALL_INDICATOR_SPECS.items()))
    except QuotaExhaustedError:
        logging.error("Red-team judge test pending (quota or RUN_JUDGE=0).")
model_comparison_df = pd.DataFrame([comparison_row(r) for r in successful])

save_json(core_for_json(core_indicators), TASK2_DIR / "core_indicators.json")
save_json(core_fallback_reasons(successful[0]), TASK2_DIR / "core_indicators_fallbacks.json")
save_json(extended_for_json(extended_indicators), TASK2_DIR / "extended_indicators.json")
pd.DataFrame(evidence_retrieval_log).to_csv(TASK2_DIR / "evidence_retrieval.csv", index=False)
save_json(red_team or {"status": "not_run"}, TASK2_DIR / "judge_red_team.json")
save_json({"evidence_chunk_ids": [c["chunk_id"] for c in evidence], "concept_mentions": concept_mentions,
           "report_year": REPORT_YEAR, "fallback_value_in_json": FALLBACK_SENTINEL,
           "evidence_found_by": evidence_retrieval_log, "results": extraction_results},
          TASK2_DIR / "indicator_extraction_audit.json")
model_comparison_df.to_csv(TASK2_DIR / "model_comparison.csv", index=False)

print(f"CORE INDICATORS (target schema; {FALLBACK_SENTINEL} = not reported, never plotted):\n"
      + json.dumps(core_for_json(core_indicators), indent=2))
print("\nFALLBACKS (primary model):")
for i in successful[0]["indicators"]:
    if i["final_value"] is None and i["field"] in INDICATOR_SPECS:
        print(f"  • {i['field']}: {i['fallback_reason']}")
print("\nEXTENDED INDICATORS:")
display(pd.DataFrame(extended_indicators).T[["value", "unit", "reference_period", "page", "fallback_reason"]])
display(model_comparison_df)
if red_team:
    print(f"\nRED-TEAM JUDGE TEST: tampered {red_team['tampered_fields']} -> factual score "
          f"{red_team['factual_alignment_score']}/5, flagged {len(red_team['claims_flagged'])} claims, "
          f"caught both: {red_team['caught_both']}")

## 15b. Supplementary indicators from the narrative (beyond the schema)

In [ ]:
# ==============================================================================
# CELL 21b: TASK 2 — SUPPLEMENTARY INDICATORS FROM THE NARRATIVE (beyond the schema)
# ------------------------------------------------------------------------------
# The target JSON schema has six fixed fields; the report quantifies many more
# metrics in its prose (e.g. bank and NBFI credit growth, household debt,
# reserve cover, inflation target band, sovereign-risk assessments). These are
# extracted here into a SEPARATE file, so core_indicators.json still matches the
# required schema exactly.
# ==============================================================================

SUPPLEMENTARY_BATCH_TOKENS = int(_env("SUPPLEMENTARY_BATCH_TOKENS", "12000"))
SUPPLEMENTARY_MAX_PER_BATCH = int(_env("SUPPLEMENTARY_MAX_PER_BATCH", "15"))
SUPPLEMENTARY_SKIP_KINDS = {"informational_annex"}
CORE_FIELD_THEMES = {
    "gdp_growth_forecast_pct": ["Growth & Output"],
    "headline_inflation_rate_pct": ["Inflation & Prices"],
    "central_bank_policy_rate_pct": ["Interest Rates & Credit", "Inflation & Prices"],
    "non_performing_loan_ratio": ["Banking Capital & Liquidity", "Interest Rates & Credit", "Systemic Financial Risk"],
    "unemployment_rate_pct": ["Labor Market Dynamics", "Growth & Output"],
    "tier_1_capital_ratio": ["Banking Capital & Liquidity", "Systemic Financial Risk"],
}


def supplementary_batches(chunks: List[Dict[str, Any]], budget: int) -> List[List[Dict[str, Any]]]:
    """Narrative chunks of summarised sections, in reading order, packed under the token budget."""
    eligible = [c for c in chunks if c["chunk_type"] == "narrative"
                and SECTIONS_BY_ID[c["section_id"]]["summarise"]
                and SECTIONS_BY_ID[c["section_id"]]["kind"] not in SUPPLEMENTARY_SKIP_KINDS]
    batches, current, used = [], [], 0
    for c in eligible:
        if current and used + c["token_count"] > budget:
            batches.append(current); current, used = [], 0
        current.append(c); used += c["token_count"]
    if current:
        batches.append(current)
    return batches


def batch_context(batch: List[Dict[str, Any]]) -> str:
    return "\n\n".join(f"### [{c['chunk_id']} | p.{c['page_number']} | {c['section_title']}]\n{c['text']}" for c in batch)


def verify_supplementary(item: Dict[str, Any], batch_by_id: Dict[str, Dict]) -> Dict[str, Any]:
    chunk = batch_by_id.get(item["source_chunk_id"])
    source = chunk["text"] if chunk else ""
    checks = {"chunk_id_valid": chunk is not None,
              "quote_verbatim": quote_is_verbatim(item["source_quote"], source),
              "number_in_quote": any(abs(n - abs(item["value"])) < 1e-9 for n in numbers_in(item["source_quote"]))}
    return {**item, "section_id": chunk["section_id"] if chunk else None,
            "page": chunk["page_number"] if chunk else None, "checks": checks, "verified": all(checks.values())}


def dedupe(items: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen, out = set(), []
    for i in items:
        key = (re.sub(r"[^a-z0-9]+", " ", i["indicator"].lower()).strip(), i["value"],
               re.sub(r"\s+", " ", i["reference_period"].lower()).strip())
        if key not in seen:
            seen.add(key); out.append(i)
    return out


def items_to_candidate(items: List[Dict[str, Any]]) -> str:
    return "\n".join(f"{i['indicator']} = {i['value']:g} {i['unit']} ({i['reference_period']}, {i['status']})"
                     for i in items)


def extract_supplementary(chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
    core_fields = ", ".join(INDICATOR_SPECS)
    verified, rejected, audits = [], [], []
    for b, batch in enumerate(supplementary_batches(chunks, SUPPLEMENTARY_BATCH_TOKENS), start=1):
        context = batch_context(batch)
        rec = llm_structured(task=f"supplementary:{b}", system=SYSTEM_SUPPLEMENTARY,
                             prompt=USER_SUPPLEMENTARY.format(k=SUPPLEMENTARY_MAX_PER_BATCH,
                                                              core_fields=core_fields, context=context),
                             schema=SupplementaryIndicators, prompt_version=PROMPT_VERSION_T23)
        batch_by_id = {c["chunk_id"]: c for c in batch}
        checked = [{**verify_supplementary(i, batch_by_id), "batch": b}
                   for i in rec["parsed"]["indicators"][:SUPPLEMENTARY_MAX_PER_BATCH]]
        good = [i for i in checked if i["verified"]]
        verified += good
        rejected += [i for i in checked if not i["verified"]]
        if good:
            try:
                audit = run_judge(f"supplementary:{b}", "list of quantitative indicators extracted from the report "
                                  "(indicator = value unit (period, status))", items_to_candidate(good), context,
                                  reference="Same as SOURCE.", generator_model=rec["model_used"])
                audits.append({"batch": b, **audit["scores"], "judge_model": audit["judge_model"],
                               "judge_independence": audit["judge_independence"]})
            except QuotaExhaustedError:
                logging.error(f"Judge paused (quota or RUN_JUDGE=0); supplementary batch {b} audit pending.")
                audits.append({"batch": b, "status": "pending"})
    verified = dedupe(verified)
    return {"status": "success", "n_verified": len(verified), "n_rejected": len(rejected),
            "indicators": verified, "rejected": rejected, "judge_audits": audits}


def fallback_related_evidence(core_result: Dict[str, Any], supplementary: List[Dict[str, Any]]) -> Dict[str, Any]:
    """For each core field that fell back, the verified indicators the report does give in related themes."""
    out = {}
    for item in core_result["indicators"]:
        if item["final_value"] is not None:
            continue
        themes = CORE_FIELD_THEMES.get(item["field"], [])
        related = [f"{s['indicator']}: {s['value']:g} {s['unit']} ({s['reference_period']}, p.{s['page']})"
                   for s in supplementary if s["theme"] in themes]
        out[item["field"]] = {"fallback_reason": item["fallback_reason"], "related_themes": themes,
                              "related_indicators_in_report": related}
    return out


# EXECUTION
try:
    supplementary = extract_supplementary(document_chunks)
except QuotaExhaustedError as exc:
    logging.error(f"Supplementary extraction pending (quota): {str(exc)[:120]}")
    supplementary = {"status": "pending", "indicators": [], "rejected": [], "judge_audits": []}
supplementary["fallback_related_evidence"] = fallback_related_evidence(successful[0], supplementary["indicators"])
save_json(supplementary, TASK2_DIR / "supplementary_indicators.json")

supp_df = pd.DataFrame(supplementary["indicators"])
print(f"✅ Supplementary indicators: {supplementary.get('n_verified', 0)} verified | "
      f"{supplementary.get('n_rejected', 0)} rejected (quote or number not found in source)")
if not supp_df.empty:
    display(supp_df[["theme", "indicator", "value", "unit", "reference_period", "status", "page"]]
            .sort_values(["theme", "page"]))
print("\nNearest quantitative evidence for each fallback field:")
for field, info in supplementary["fallback_related_evidence"].items():
    print(f"  • {field}: {len(info['related_indicators_in_report'])} related indicators "
          f"({', '.join(info['related_themes'])})")

## 16. Temporal trends

In [ ]:
# ==============================================================================
# CELL 22: TASK 2 — TEMPORAL TREND IDENTIFICATION (multi-year series)
# ------------------------------------------------------------------------------
# Numbers come ONLY from the parsed statistical tables (deterministic parsing:
# LLM transcription of 10-column tables is a known hallucination risk). The LLM
# IDENTIFIES the relevant trends (enum schema: it cannot invent a series) and
# describes direction and peak year, which code verifies against the numbers;
# the judge LLM then audits the extracted values against the source tables
# (Evaluator Validation for the trend JSON).
# Status: years <= REPORT_YEAR-1 are outturns, later years IMF projections
# (the report's tables label 2022-24 "Actual" and 2025+ "Projections").
# DASHBOARD_SERIES are always kept for Task 4. Excluded on purpose: the Annex IV
# DSA row "Real interest rate", which is a contribution to debt dynamics in that
# table, not an interest rate; only "Effective interest rate (percent)" is used.
# ==============================================================================

YEAR_RE = re.compile(r"^(20[0-4]\d)$")
DASHBOARD_SERIES = {
    "CPI inflation, period average (%)": r"^Consumer Prices \(Avg",
    "CPI inflation, end-period (%)": r"^Consumer Prices \(EoP",
    "Effective interest rate on public debt (%)": r"^Effective interest rate \(percent\)",
    "Credit growth (%)": r"^Credit growth \(percent change\)",
    "Real GDP growth (%)": r"^Real GDP growth \(percent change\)",
    "General government debt (% of GDP)": r"^General government debt",
    "Broad money growth (%)": r"^Broad money growth \(percent change\)",
    "Reserve money growth (%)": r"^Reserve money growth \(percent change\)",
    # Table 1 (% of GDP); first match in page order is Table 1, not the USD/MNT tables
    "Current account balance (% of GDP)": r"^Current account balance(\s*\(in percent of GDP\))?$",
    "Overall fiscal balance (% of GDP)": r"^Overall balance \(IMF definition\)",
    "Non-mineral primary balance (% of GDP)": r"^Non-mineral primary balance",
}
TREND_SELECTION_K = 8


def parse_cell_number(cell: str) -> Optional[float]:
    cell = str(cell).strip().replace(",", "").replace("−", "-")
    return float(cell) if re.fullmatch(r"-?\d+(?:\.\d+)?", cell) else None


def series_from_table(table: Dict[str, Any], page: int) -> List[Dict[str, Any]]:
    """Rows of a year-indexed table -> [{label, page, values: {year: value}}]."""
    rows = table["data"]
    header = next((r for r in rows[:8] if sum(bool(YEAR_RE.match(str(c).strip())) for c in r) >= 3), None)
    if header is None:
        return []
    col_year = {i: int(c) for i, c in enumerate(header) if YEAR_RE.match(str(c).strip())}
    out = []
    for r in rows:
        label = str(r[0]).strip() if r else ""
        if not label or re.fullmatch(r"[\d\s.-]+", label):
            continue
        values = {y: parse_cell_number(r[i]) for i, y in col_year.items() if i < len(r)}
        values = {y: v for y, v in values.items() if v is not None}
        if len(values) >= 3:
            out.append({"label": label, "page": page, "values": values})
    return out


def build_series_catalogue(parsed: Dict[str, Any]) -> List[Dict[str, Any]]:
    """All multi-year series in the staff report tables (press-release duplicates skipped)."""
    catalogue, seen = [], set()
    for page in parsed["pages"]:
        for table in page["tables"]:
            if SECTIONS_BY_ID.get(table.get("section_id"), {}).get("kind") == "press_release":
                continue
            for s in series_from_table(table, page["page_number"]):
                key = tuple(sorted(s["values"].items()))      # same numbers = same series (e.g. Table 1 vs p.37)
                if key not in seen:
                    seen.add(key)
                    s["name"] = f"{s['label']} [p.{s['page']}]"
                    catalogue.append(s)
    return catalogue


def pick_dashboard_series(catalogue: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    picked = {}
    for display_name, pattern in DASHBOARD_SERIES.items():
        match = next((s for s in catalogue if re.match(pattern, s["label"])), None)
        if match:
            picked[display_name] = match
        else:
            logging.warning(f"Dashboard series not found in tables: {display_name}")
    return picked


def llm_select_trends(catalogue: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    names = [s["name"] for s in catalogue]
    listing = "\n".join(f"- {s['name']}: " + ", ".join(f"{y}:{v:g}" for y, v in sorted(s["values"].items()))
                        for s in catalogue)
    rec = llm_structured(task="trend_selection", system=SYSTEM_TRENDS,
                         prompt=USER_TRENDS.format(k=TREND_SELECTION_K, catalogue=listing),
                         schema=make_trend_schema(names), prompt_version=PROMPT_VERSION_T23)
    return rec["parsed"]["selected"][:TREND_SELECTION_K]


def computed_direction(values: Dict[int, float], tol: float = 0.05) -> str:
    """Deterministic trend label used to check the LLM's trend_direction."""
    ys = [values[y] for y in sorted(values)]
    span = (max(ys) - min(ys)) or 1e-9
    steps = [b - a for a, b in zip(ys, ys[1:])]
    if span <= tol * max(abs(max(ys)), abs(min(ys)), 1):
        return "stable"
    ups, downs = sum(d > 0 for d in steps), sum(d < 0 for d in steps)
    if ups and downs and min(ups, downs) / len(steps) >= 0.34:
        return "volatile"
    return "rising" if ys[-1] > ys[0] else "falling"


def check_llm_trends(picks: List[Dict[str, Any]], by_name: Dict[str, Dict]) -> List[Dict[str, Any]]:
    """Code check of the LLM's trend description against the parsed numbers."""
    out = []
    for p in picks:
        vals = by_name[p["series_name"]]["values"]
        peak = max(vals, key=vals.get)
        out.append({**p, "computed_direction": computed_direction(vals), "computed_peak_year": peak,
                    "direction_matches": p["trend_direction"] == computed_direction(vals),
                    "peak_year_matches": p["peak_year"] == peak})
    return out


def audit_trends(records: List[Dict[str, Any]], chunks: List[Dict[str, Any]]) -> Optional[Dict[str, Any]]:
    """Evaluator validation: the judge checks the trend values against the source table chunks."""
    pages = {r["page"] for r in records}
    source_chunks = [c for c in chunks if c["chunk_type"] == "table" and c["page_number"] in pages]
    source, coverage = budgeted_source(source_chunks, " ".join(r["source_label"] for r in records),
                                       CFG.judge_max_source_tokens)
    by_series: Dict[str, List[str]] = {}
    for r in records:
        by_series.setdefault(f"{r['source_label']} (table p.{r['page']})", []).append(f"{r['year']}: {r['value']:g}")
    candidate = "\n".join(f"{k} -> {', '.join(v)}" for k, v in by_series.items())
    audit = run_judge("temporal_trends", "multi-year time series (row label -> year: value) read from report tables",
                      candidate, source, reference="Same as SOURCE.", generator_model="deterministic_table_parser")
    return {**audit["scores"], "source_coverage_pct": coverage, "judge_model": audit["judge_model"],
            "unsupported_claims": audit["audit"]["unsupported_claims"], "justification": audit["audit"]["justification"]}


def series_to_records(name: str, s: Dict[str, Any], origin: str) -> List[Dict[str, Any]]:
    return [{"series": name, "source_label": s["label"], "page": s["page"], "year": y, "value": v,
             "status": "actual" if REPORT_YEAR and y <= REPORT_YEAR - 1 else "projection", "origin": origin}
            for y, v in sorted(s["values"].items())]


# EXECUTION
series_catalogue = build_series_catalogue(parsed_document)
dashboard_series = pick_dashboard_series(series_catalogue)
try:
    llm_trends = llm_select_trends(series_catalogue)
except Exception as exc:
    logging.error(f"LLM trend selection failed ({exc}); keeping dashboard series only.")
    llm_trends = []

def merge_selections(dashboard: Dict[str, Dict], picks: List[Dict], catalogue: List[Dict]) -> List[Dict[str, Any]]:
    """One record set per catalogue series; a series chosen by both routes is kept once, flagged with both."""
    by_name = {s["name"]: s for s in catalogue}
    chosen: Dict[str, Dict[str, Any]] = {}
    for display_name, s in dashboard.items():
        chosen[s["name"]] = {"display": display_name, "series": s, "origins": ["dashboard"]}
    for p in picks:
        entry = chosen.setdefault(p["series_name"], {"display": p["series_name"], "series": by_name[p["series_name"]],
                                                     "origins": []})
        entry["origins"].append("llm_selected")
    return [r for e in chosen.values() for r in series_to_records(e["display"], e["series"], "+".join(e["origins"]))]


trend_records = merge_selections(dashboard_series, llm_trends, series_catalogue)
time_series_long = pd.DataFrame(trend_records)
llm_trend_checks = check_llm_trends(llm_trends, {s["name"]: s for s in series_catalogue})
try:
    trend_audit = audit_trends(trend_records, document_chunks)
except QuotaExhaustedError:
    logging.error("Trend audit pending (quota or RUN_JUDGE=0).")
    trend_audit = {"status": "pending"}
save_json({"report_year": REPORT_YEAR, "catalogue_size": len(series_catalogue), "llm_selection": llm_trend_checks,
           "judge_audit": trend_audit, "records": trend_records}, TASK2_DIR / "temporal_trends.json")

print(f"✅ {len(series_catalogue)} multi-year series parsed | dashboard series: {len(dashboard_series)} | "
      f"LLM-selected: {len(llm_trends)}")
for p in llm_trend_checks:
    print(f"  • {p['series_name']}: {p['trend_direction']} (code: {p['computed_direction']}), "
          f"peak {p['peak_year']} (code: {p['computed_peak_year']}) — {p['reason']}")
print("Judge audit of trend values (1-5):", {k: trend_audit.get(k) for k in
      ("factual_alignment_score", "consistency_score", "completeness_score", "source_coverage_pct")})
display(time_series_long.pivot_table(index="series", columns="year", values="value"))

## 17. Task 2 consolidated output

In [ ]:
# ==============================================================================
# CELL 23: TASK 2 — CONSOLIDATED MACHINE-READABLE OUTPUT
# ------------------------------------------------------------------------------
# One JSON with every Task 2 result (core + supplementary indicators, fallbacks, thematic counts,
# qualitative insights, trend series, evaluator scores) alongside the per-step
# files already written to outputs/task2/.
# ==============================================================================

primary_extraction = successful[0]
task2_bundle = {
    "report": REPORT_TITLE,
    "core_indicators": core_for_json(core_indicators),
    "indicator_fallbacks": core_fallback_reasons(primary_extraction),
    "indicator_evidence": {i["field"]: {"quote": i.get("source_quote"), "chunk_id": i.get("source_chunk_id"),
                                        "period": i.get("reference_period"), "status": i.get("status")}
                           for i in primary_extraction["indicators"] if i["final_value"] is not None},
    "indicator_judge": primary_extraction.get("judge"),
    "extended_indicators": extended_for_json(extended_indicators),
    "judge_red_team": red_team,
    "model_comparison": model_comparison_df.to_dict("records"),
    "supplementary_indicators": supplementary.get("indicators", []),
    "fallback_related_evidence": supplementary.get("fallback_related_evidence", {}),
    "supplementary_judge": supplementary.get("judge_audits", []),
    "thematic_distribution": thematic["themes"],
    "risk_insights": {k: risk_insights.get(k) for k in ("strengths", "vulnerabilities", "checks", "judge")},
    "temporal_trends": trend_records,
    "temporal_trends_llm_identification": llm_trend_checks,
    "temporal_trends_judge": trend_audit,
}
save_json(task2_bundle, TASK2_DIR / "task2_outputs.json")
print("✅ Task 2 artefacts:")
for f in sorted(TASK2_DIR.iterdir()):
    print(f"   {f.name:34s} {f.stat().st_size / 1024:8.1f} KB")

# Task 3 — RAG grounding and retrieval
## 19. Grounded question answering vs closed-book baseline
Uses the dense index built in section 12b.

In [ ]:
# ==============================================================================
# CELL 25: TASK 3 — GROUNDED QUERY ANSWERING + RAG TRIAD MONITORING
# ------------------------------------------------------------------------------
# For each factual question (brief's examples: capital buffer recommendations,
# stress/scenario assumptions, inflation-target timelines, adapted to this report):
#   1. RAG answer: retrieve top-k chunks (FAISS) -> answer ONLY from them.
#   2. Closed-book baseline: the same model answers WITHOUT retrieved context.
#      This is the hallucination control: same question, same model, no grounding.
#   3. The judge scores both answers against the SAME retrieved context with one
#      composite JSON: RAG Triad — context relevance (retriever),
#      groundedness (hallucination), answer relevance (generator) — plus factual
#      alignment, completeness, consistency (1-5 each), unsupported claims and a
#      consolidated justification. Context relevance is a property of retrieval,
#      so it is reported for the RAG run only.
#   4. Programmatic monitoring: thresholds flag retrieval failures and
#      hallucination risk per question (TruLens-style triad dashboard).
# ==============================================================================

RAG_TOP_K = int(_env("RAG_TOP_K", "5"))
TRIAD_ALERT_THRESHOLD = 3          # a score below this raises a monitoring flag
RAG_QUESTIONS = [
    {"id": "inflation_target", "question": "What is the Bank of Mongolia's inflation target band, and when is inflation expected to return to it?"},
    {"id": "capital_buffers", "question": "What capital buffers and other macroprudential tools does the IMF recommend for Mongolia's banks?"},
    {"id": "dsti_limits", "question": "What does the IMF recommend about debt-service-to-income (DSTI) limits for banks and nonbank financial institutions?"},
    {"id": "sovereign_risk", "question": "What is the IMF's assessment of Mongolia's sovereign risk and debt sustainability over the near and medium term?"},
    {"id": "coal_scenario", "question": "What assumptions does the coal transition analysis make about China's coal demand, and what is the estimated impact on Mongolia's GDP?"},
]


def format_context(hits: List[Dict[str, Any]]) -> str:
    return "\n\n".join(f"### [{h['chunk_id']} | p.{h['page_number']} | {h['section_title']}]\n{h['text']}" for h in hits)


def triad_judge(task_id: str, question: str, context: str, answer: str, generator_model: str) -> Dict[str, Any]:
    rec = llm_structured(task=f"rag_triad:{task_id}", system=SYSTEM_RAG_JUDGE,
                         prompt=USER_RAG_JUDGE.format(question=question, context=context, answer=answer),
                         schema=RAGTriadAudit, models=JUDGE_MODELS, prompt_version=JUDGE_PROMPT_VERSION,
                         cache_only=not CFG.run_judge)
    return {**rec["parsed"], "judge_model": rec["model_used"],
            "judge_independence": independence_label(rec["model_used"], generator_model)}


def answer_question(q: Dict[str, str]) -> Dict[str, Any]:
    hits = retrieve(q["question"], RAG_TOP_K)
    context = format_context(hits)
    rag = llm_structured(task=f"rag:{q['id']}", system=SYSTEM_RAG,
                         prompt=USER_RAG.format(question=q["question"], context=context),
                         schema=RAGAnswer, prompt_version=PROMPT_VERSION_T23)
    closed = llm_structured(task=f"closed_book:{q['id']}", system=SYSTEM_CLOSED_BOOK,
                            prompt=USER_CLOSED_BOOK.format(question=q["question"]),
                            schema=ClosedBookAnswer, prompt_version=PROMPT_VERSION_T23)
    retrieved_ids = {h["chunk_id"] for h in hits}
    cited = rag["parsed"]["cited_chunk_ids"]
    result = {**q, "retrieved": [{k: h[k] for k in ("rank", "chunk_id", "page_number", "section_id",
                                                     "cosine_similarity")} for h in hits],
              "rag_answer": rag["parsed"], "closed_book_answer": closed["parsed"]["answer"],
              "citation_valid_pct": round(100 * sum(c in retrieved_ids for c in cited) / len(cited), 1) if cited else 0.0,
              "generator_model": rag["model_used"], "judged": {}}
    for mode, text, gen_model in (("rag", rag["parsed"]["answer"], rag["model_used"]),
                                  ("closed_book", closed["parsed"]["answer"], closed["model_used"])):
        try:
            result["judged"][mode] = triad_judge(f"{mode}:{q['id']}", q["question"], context, text, gen_model)
        except QuotaExhaustedError:
            logging.error(f"Judge paused (quota or RUN_JUDGE=0); {mode} triad for '{q['id']}' pending.")
    return result


def triad_frame(results: List[Dict[str, Any]]) -> pd.DataFrame:
    """Wide frame, one row per question x mode, scores 1-5 (melted in Task 4)."""
    rows = []
    for r in results:
        for mode, j in r.get("judged", {}).items():
            row = {"question_id": r["id"], "mode": mode,
                   **{m: j[m] for m in RAG_TRIAD_METRICS + RAG_QUALITY_METRICS},
                   "n_unsupported_claims": len(j["unsupported_claims"]),
                   "mean_retrieval_cosine": round(float(np.mean([h["cosine_similarity"] for h in r["retrieved"]])), 3),
                   "citation_valid_pct": r["citation_valid_pct"] if mode == "rag" else None,
                   "judge_model": j["judge_model"]}
            if mode == "closed_book":
                row["context_relevance_score"] = None       # retrieval was not used by this answer
            rows.append(row)
    return pd.DataFrame(rows, columns=["question_id", "mode", *RAG_TRIAD_METRICS, *RAG_QUALITY_METRICS,
                                       "n_unsupported_claims", "mean_retrieval_cosine", "citation_valid_pct",
                                       "judge_model"])


def triad_monitor(df: pd.DataFrame) -> pd.DataFrame:
    """Programmatic monitoring of the RAG run: flag weak retrieval, hallucination risk, off-target answers."""
    rag = df[df["mode"] == "rag"].copy()
    rag["retrieval_flag"] = rag["context_relevance_score"] < TRIAD_ALERT_THRESHOLD
    rag["hallucination_flag"] = (rag["groundedness_score"] < TRIAD_ALERT_THRESHOLD) | (rag["n_unsupported_claims"] > 0)
    rag["answer_flag"] = rag["answer_relevance_score"] < TRIAD_ALERT_THRESHOLD
    return rag[["question_id", *RAG_TRIAD_METRICS, "n_unsupported_claims", "retrieval_flag",
                "hallucination_flag", "answer_flag"]]


# EXECUTION
rag_results = []
for q in RAG_QUESTIONS:
    try:
        rag_results.append(answer_question(q))
    except LLMSkipped as exc:                      # uncached answer (no key): skip only this question
        logging.error(f"RAG question '{q['id']}' pending: {str(exc)[:100]}")
        rag_results.append({**q, "status": "pending"})
    except QuotaExhaustedError as exc:
        logging.error(f"Paused at RAG question '{q['id']}'; remaining questions pending ({str(exc)[:100]}).")
        rag_results += [{**r, "status": "pending"} for r in RAG_QUESTIONS[RAG_QUESTIONS.index(q):]]
        break
    except Exception as exc:
        logging.error(f"RAG failed for {q['id']}: {exc}")
        rag_results.append({**q, "status": "error", "error": str(exc)[:300]})
save_json(rag_results, RAG_DIR / "rag_results.json")
rag_scores_df = triad_frame(rag_results)
rag_scores_df.to_csv(RAG_DIR / "rag_triad_scores.csv", index=False)
rag_monitor_df = triad_monitor(rag_scores_df)
rag_monitor_df.to_csv(RAG_DIR / "rag_triad_monitor.csv", index=False)

for r in rag_results:
    if "rag_answer" not in r:
        continue
    pages = ", ".join("p." + str(h["page_number"]) for h in r["retrieved"])
    j = r["judged"].get("rag", {})
    print(f"\nQ: {r['question']}\nRAG ({pages}): {r['rag_answer']['answer']}\n"
          f"   Triad (1-5): context {j.get('context_relevance_score', '-')} | grounded "
          f"{j.get('groundedness_score', '-')} | answer {j.get('answer_relevance_score', '-')} | "
          f"unsupported claims: {len(j.get('unsupported_claims', []))}")
    if j.get("justification"):
        print(f"   Justification: {j['justification']}")
if not rag_scores_df.empty:
    print("\nMean scores by mode (1-5):\n",
          rag_scores_df.groupby("mode")[RAG_TRIAD_METRICS + RAG_QUALITY_METRICS + ["n_unsupported_claims"]].mean().round(2))
    print("\nRAG TRIAD MONITOR (flags: score <", TRIAD_ALERT_THRESHOLD, "or unsupported claims):")
    display(rag_monitor_df)

## 19b. Additional — cross-family judge (self-preference bias check)
Re-scores a sample with a non-Gemini model on Groq's free tier, with scores and a justification per item. Replays cached audits without a key; skipped if there are none. Runs before Task 4 so the dashboard's model comparison includes it.

In [ ]:
# ==============================================================================
# CELL 25b: ADDITIONAL — CROSS-FAMILY JUDGE (self-preference bias check)
# ------------------------------------------------------------------------------
# The main judge (gemini-3.5-flash-lite) and generator (gemini-3.1-flash-lite) are
# different models but the same family, so a shared-family preference cannot be
# ruled out. This cell re-scores a SAMPLE of the same outputs with a different
# model family on Groq — openai/gpt-oss-120b (fallback qwen/qwen3.8-27b) — using
# the SAME prompts and schemas, then measures agreement with the Gemini judge.
# It changes nothing upstream. It runs before Task 4 so the dashboard's model
# comparison can include the Groq judge. Each Groq audit returns the same 1-5
# scores AND a justification string (strict schema); both judges' justifications
# are saved side by side. Without GROQ_API_KEY, cached Groq audits are replayed
# (key-free marking); with neither, the cell is skipped and Task 4 still runs.

import copy
import subprocess
import sys

CROSS_JUDGE_MODELS = tuple(m.strip() for m in _env("CROSS_JUDGE_MODELS",
                                                   "openai/gpt-oss-120b,qwen/qwen3.8-27b").split(",") if m.strip())
CROSS_TPM_LIMIT = int(_env("CROSS_TPM_LIMIT", "7500"))
CROSS_RPM_LIMIT = int(_env("CROSS_RPM_LIMIT", "28"))
CROSS_SOURCE_TOKENS = int(_env("CROSS_SOURCE_TOKENS", "3500"))
CROSS_OUTPUT_ALLOWANCE = 1500                 # tokens reserved for the reply (counted in TPM)
CROSS_SECTIONS = ["key_issues", "outlook_and_risks", "d_macroprudential_and_financial_sector_policies",
                  "staff_appraisal", "recent_developments", "context", "annex_III"]
CROSS_DIR = CFG.output_dir / "cross_family_judge"
GROQ_API_KEY = load_secret("GROQ_API_KEY", IN_COLAB)


def strict_json_schema(model: Type[BaseModel]) -> Dict[str, Any]:
    """Pydantic schema -> Groq strict schema: refs inlined, all fields required, no extra keys."""
    raw = model.model_json_schema()
    defs = raw.pop("$defs", {})

    def fix(node: Any) -> Any:
        if isinstance(node, dict):
            if "$ref" in node:
                return fix(copy.deepcopy(defs[node["$ref"].split("/")[-1]]))
            node = {k: fix(v) for k, v in node.items() if k not in ("title", "default", "description")}
            if node.get("type") == "object":
                node["additionalProperties"] = False
                node["required"] = list(node.get("properties", {}))
            return node
        if isinstance(node, list):
            return [fix(x) for x in node]
        return node

    return fix(raw)


def groq_wait_seconds(exc: Exception) -> Optional[float]:
    m = re.search(r"try again in (?:(\d+)m)?(\d+(?:\.\d+)?)s", str(exc))
    return (int(m.group(1) or 0) * 60 + float(m.group(2))) if m else None


class CrossFamilyJudge:
    """Groq caller with strict-schema output, token-aware pacing, disk cache and model fallback."""

    def __init__(self, api_key: Optional[str], models: Tuple[str, ...]):
        self.client = None
        if api_key:                                # no key -> cache replay only
            from groq import Groq
            self.client = Groq(api_key=api_key)
        self.models = list(models)
        self.limiter = RateLimiter(CROSS_RPM_LIMIT, CROSS_TPM_LIMIT)
        self.daily_exhausted: set = set()

    def _create(self, model: str, system: str, prompt: str, schema: Type[BaseModel]) -> str:
        base = dict(model=model, temperature=0.0, messages=[{"role": "system", "content": system},
                                                            {"role": "user", "content": prompt}])
        formats = [{"type": "json_schema", "json_schema": {"name": schema.__name__, "strict": True,
                                                           "schema": strict_json_schema(schema)}},
                   {"type": "json_object"}]
        last_exc = None
        use_low_reasoning = model.startswith("openai/gpt-oss")   # fewer output tokens -> fits 8K TPM
        for fmt in formats:                                   # strict schema first, JSON mode as fallback
            for attempt in range(4):
                self.limiter.acquire(model, estimate_tokens(system, prompt) + CROSS_OUTPUT_ALLOWANCE)
                try:
                    extra = {"reasoning_effort": "low"} if use_low_reasoning else {}
                    resp = self.client.chat.completions.create(**base, response_format=fmt, **extra)
                    return resp.choices[0].message.content
                except Exception as exc:
                    last_exc, text = exc, str(exc).lower()
                    if "reasoning_effort" in text and use_low_reasoning:
                        use_low_reasoning = False; continue    # parameter unsupported: retry without it
                    if "per day" in text or "tpd" in text or "rpd" in text:
                        raise QuotaExhaustedError(f"Groq daily limit reached for {model}: {str(exc)[:120]}")
                    if "429" in text or "rate limit" in text:
                        time.sleep((groq_wait_seconds(exc) or 20) + 1); continue
                    if "response_format" in text or "json_schema" in text or "schema" in text:
                        break                                 # format unsupported -> next format
                    raise
        raise RuntimeError(f"Groq call failed for {model}: {last_exc}")

    def structured(self, task: str, system: str, prompt: str, schema: Type[BaseModel]) -> Dict[str, Any]:
        key = stable_hash({"task": task, "provider": "groq", "models": self.models[:1], "system": system,
                           "prompt": prompt, "schema": schema.model_json_schema()})
        cached = LLM_CACHE.get(key)
        if cached is not None:
            return {**cached, "cached": True}
        if CFG.llm_mode == "mock":
            return mock_record(task, schema, tuple(self.models))
        if self.client is None:
            raise LLMSkipped(f"[{task}] no cached Groq audit and no GROQ_API_KEY")
        for model in [m for m in self.models if m not in self.daily_exhausted]:
            try:
                parsed = schema.model_validate_json(_strip_code_fences(self._create(model, system, prompt, schema)))
                record = {"task": task, "provider": "groq", "model_used": model, "parsed": parsed.model_dump()}
                LLM_CACHE.put(key, record)
                return {**record, "cached": False}
            except QuotaExhaustedError:
                self.daily_exhausted.add(model)
            except Exception as exc:
                logging.warning(f"[{task}] {model} failed ({str(exc)[:120]}); trying next model.")
        raise QuotaExhaustedError(f"[{task}] no Groq model available")


def fit_request(build: Any, budget: int) -> Tuple[str, float]:
    """Shrink the source budget until the whole request fits under the per-minute token limit."""
    while True:
        prompt, coverage = build(budget)
        if estimate_tokens(SYSTEM_JUDGE, prompt) + CROSS_OUTPUT_ALLOWANCE <= CROSS_TPM_LIMIT or budget < 800:
            return prompt, coverage
        budget = int(budget * 0.8)


def cross_items() -> List[Dict[str, Any]]:
    """Sample = the same outputs the Gemini judge scored: key sections, executive summary, indicators, RAG."""
    items = []
    summaries = {r["section_id"]: r for r in section_summaries if r.get("status") == "success"}
    for sid in [s for s in CROSS_SECTIONS if s in summaries]:
        rec = summaries[sid]
        def build(b, rec=rec):
            src, cov = budgeted_source(chunks_of(rec["section_id"], document_chunks), rec["summary"], b)
            return USER_JUDGE.format(candidate_type="section summary (one paragraph, fewer than 100 words)",
                                     reference="Same as SOURCE.", source=src, candidate=rec["summary"]), cov
        items.append({"target_id": sid, "kind": "judge", "build": build})
    if executive_summary.get("status") == "success":
        def build(b):
            cited = set(executive_summary["cited_pages"])
            src, cov = budgeted_source([c for c in document_chunks if cited & set(c["pages"])],
                                       executive_summary["markdown"], b)
            ref = build_section_context(parsed_document, "key_issues")[:4000] if "key_issues" in SECTIONS_BY_ID else ""
            return USER_JUDGE.format(candidate_type="executive summary of the whole report", reference=ref,
                                     source=src, candidate=executive_summary["markdown"]), cov
        items.append({"target_id": "executive_summary", "kind": "judge", "build": build})
    primary = successful[0]
    candidate = indicators_candidate_text(primary["indicators"])
    if candidate:
        def build(b):
            src, cov = budgeted_source(evidence, candidate, b)
            return USER_JUDGE.format(candidate_type="extracted numerical indicators", source=src, candidate=candidate,
                                     reference="Field definitions as in the extraction prompt."), cov
        items.append({"target_id": f"indicators:{primary['model']}", "kind": "judge", "build": build})
    for r in rag_results:
        if "rag_answer" in r:
            ctx = format_context(retrieve(r["question"], RAG_TOP_K))
            items.append({"target_id": f"rag:{r['id']}", "kind": "triad",
                          "prompt": USER_RAG_JUDGE.format(question=r["question"], context=ctx,
                                                          answer=r["rag_answer"]["answer"])})
    return items


def gemini_scores_for(target_id: str, kind: str) -> Dict[str, Any]:
    if kind == "triad":
        r = next((x for x in rag_results if f"rag:{x['id']}" == target_id), {})
        return r.get("judged", {}).get("rag", {})
    if target_id.startswith("indicators:"):
        return next((x.get("judge", {}) for x in successful if f"indicators:{x['model']}" == target_id), {})
    row = judge_scores_df[judge_scores_df["target_id"] == target_id]
    if row.empty:
        return {}
    audit = next((a.get("audit", {}) for a in judge_audits if a.get("target_id") == target_id), {})
    return {**row.iloc[0].to_dict(), "justification": audit.get("justification", "")}


CROSS_COLUMNS = ["target_id", "kind", "metric", "cross_family_score", "gemini_score", "cross_model",
                 "source_coverage_pct"]


def run_cross_family(judge: CrossFamilyJudge) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Long score table (one row per item x metric) + one justification row per item, from both judges."""
    rows, justifications = [], []
    for it in cross_items():
        try:
            if it["kind"] == "triad":
                rec = judge.structured(f"xjudge_triad:{it['target_id']}", SYSTEM_RAG_JUDGE, it["prompt"], RAGTriadAudit)
                metrics, coverage = RAG_TRIAD_METRICS + RAG_QUALITY_METRICS, 100.0
            else:
                prompt, coverage = fit_request(it["build"], CROSS_SOURCE_TOKENS)
                rec = judge.structured(f"xjudge:{it['target_id']}", SYSTEM_JUDGE, prompt, JudgeAudit)
                metrics = JUDGE_METRICS
        except LLMSkipped:
            continue                                   # not cached and no key: skip this item
        except QuotaExhaustedError as exc:
            logging.error(f"Cross-family judge stopped (Groq limit): {str(exc)[:120]}"); break
        gem = gemini_scores_for(it["target_id"], it["kind"])
        justifications.append({"target_id": it["target_id"], "kind": it["kind"], "cross_model": rec["model_used"],
                               "cross_family_justification": rec["parsed"]["justification"],
                               "cross_family_unsupported_claims": rec["parsed"]["unsupported_claims"],
                               "gemini_justification": gem.get("justification", "")})
        for m in metrics:
            rows.append({"target_id": it["target_id"], "kind": it["kind"], "metric": m,
                         "cross_family_score": rec["parsed"][m], "gemini_score": gem.get(m),
                         "cross_model": rec["model_used"], "source_coverage_pct": coverage})
    return (pd.DataFrame(rows, columns=CROSS_COLUMNS),
            pd.DataFrame(justifications, columns=["target_id", "kind", "cross_model", "cross_family_justification",
                                                  "cross_family_unsupported_claims", "gemini_justification"]))


def agreement_summary(df: pd.DataFrame) -> pd.DataFrame:
    d = df.dropna(subset=["gemini_score"]).copy()
    if d.empty:
        return pd.DataFrame()
    d["diff"] = d["gemini_score"].astype(float) - d["cross_family_score"].astype(float)
    return d.groupby("metric").agg(n=("diff", "size"), gemini_mean=("gemini_score", "mean"),
                                   cross_family_mean=("cross_family_score", "mean"),
                                   mean_diff_gemini_minus_cross=("diff", "mean"),
                                   exact_agreement_pct=("diff", lambda x: 100 * (x == 0).mean()),
                                   within_1_point_pct=("diff", lambda x: 100 * (x.abs() <= 1).mean())).round(2)


# EXECUTION
cross_df, cross_justifications_df = pd.DataFrame(columns=CROSS_COLUMNS), pd.DataFrame()
cross_summary = pd.DataFrame()
if GROQ_API_KEY:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "groq"], check=True)
if GROQ_API_KEY or CFG.llm_mode == "mock" or any(LLM_CACHE.cache_dir.glob("*.json")):
    cross_df, cross_justifications_df = run_cross_family(CrossFamilyJudge(GROQ_API_KEY, CROSS_JUDGE_MODELS))
if cross_df.empty:
    print("ℹ️ Cross-family judge skipped: no cached Groq audits and no GROQ_API_KEY "
          "(optional; add it to Colab Secrets to run this bias check).")
else:
    CROSS_DIR.mkdir(parents=True, exist_ok=True)
    cross_summary = agreement_summary(cross_df)
    cross_df.to_csv(CROSS_DIR / "cross_family_scores.csv", index=False)
    cross_justifications_df.to_csv(CROSS_DIR / "cross_family_justifications.csv", index=False)
    save_json({"models": list(CROSS_JUDGE_MODELS), "summary": cross_summary.reset_index().to_dict("records"),
               "scores": cross_df.to_dict("records"),
               "justifications": cross_justifications_df.to_dict("records")}, CROSS_DIR / "cross_family_judge.json")
    print(f"✅ Cross-family audits: {cross_df['target_id'].nunique()} outputs re-scored by "
          f"{', '.join(sorted(cross_df['cross_model'].unique()))}"
          f"{' (replayed from cache)' if not GROQ_API_KEY else ''}")
    print("Positive mean_diff = Gemini judge scores higher than the other family (possible self-preference):")
    display(cross_summary)
    for _, j in cross_justifications_df.head(4).iterrows():   # full set in cross_family_justifications.csv
        print(f"\n[{j['target_id']}] {j['cross_model']}: {j['cross_family_justification']}"
              f"\n   Gemini judge: {j['gemini_justification'] or '-'}")

## 19c. Evaluation summary: hallucination evidence, constraint adherence and failure modes
No API calls. Consolidates the code checks and judge verdicts from Tasks 1-3 into three tables saved to `outputs/evaluation/`.

In [ ]:
# ==============================================================================
# CELL 25c: EVALUATION SUMMARY — HALLUCINATION EVIDENCE, CONSTRAINT ADHERENCE,
#           FAILURE MODES (no API calls; reads results already computed)
# ------------------------------------------------------------------------------
# How do we know a model hallucinated? Each output is checked two ways:
#   * Deterministic (code) checks: exact tests that need no LLM, e.g. every
#     number in a summary must occur in the source; an indicator's quote must be
#     verbatim in its cited chunk; a RAG answer may cite only retrieved chunks.
#   * LLM-as-a-Judge checks: claim-by-claim verdicts (SUPPORTED / UNSUPPORTED /
#     CONTRADICTED), 1-5 scores and the RAG Triad's groundedness.
# Constraint adherence = did the model follow the explicit instructions in its
# prompt (word limit, bullet count, citations, schema, null when not reported)?
# Failure modes = the recurring ways outputs go wrong; every detected instance
# is logged with the stage, item and detail so the report can discuss them.
# Grounding evidence = RAG vs closed-book on the same questions and judge.
# ==============================================================================

EVAL_DIR = CFG.output_dir / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)


def pct(n: float, d: float) -> Optional[float]:
    return round(100 * n / d, 1) if d else None


def schema_retry_stats() -> Tuple[int, int]:
    """Cached LLM records that needed a schema-repair retry (structured-output adherence)."""
    n = retries = 0
    for f in LLM_CACHE.cache_dir.glob("*.json"):
        rec = load_json(f)
        if isinstance(rec, dict) and "parsed" in rec:
            n += 1
            retries += bool(rec.get("schema_retry"))
    return n, retries


def constraint_adherence() -> pd.DataFrame:
    """One row per explicit prompt constraint: how many outputs complied."""
    ok_sum = [r for r in section_summaries if r.get("status") == "success"]
    ex = executive_summary.get("checks", {})
    items = [i for r in successful for i in r["indicators"] if i.get("value") is not None]
    absent = [i for r in successful for i in r["indicators"] if i.get("value") is None]
    rag_ok = [r for r in rag_results if "rag_answer" in r]
    n_rec, n_retry = schema_retry_stats()
    rows = [
        ("Task 1", f"Section summary <= {CFG.section_summary_max_words} words", "code",
         sum(r["passed_word_limit"] for r in ok_sum), len(ok_sum)),
        ("Task 1", "Section summary within limit on first attempt", "code",
         sum(r.get("attempts", 1) == 1 for r in ok_sum), len(ok_sum)),
        ("Task 1", f"Executive summary has {CFG.exec_summary_min_bullets}-{CFG.exec_summary_max_bullets} bullets",
         "code", int(bool(ex.get("bullet_count_ok"))), 1 if ex else 0),
        ("Task 1", "Executive summary bullets cite pages", "code",
         ex.get("n_bullets", 0) - ex.get("bullets_without_citation", 0), ex.get("n_bullets", 0)),
        ("All", "Valid JSON schema without repair retry", "code (Pydantic)", n_rec - n_retry, n_rec),
        ("Task 2", "Indicator quote is verbatim in the cited chunk", "code",
         sum(i.get("checks", {}).get("quote_verbatim", False) for i in items), len(items)),
        ("Task 2", "Indicator value appears in its quote", "code",
         sum(i.get("checks", {}).get("number_in_quote", False) for i in items), len(items)),
        ("Task 2", "Not reported -> null with a reason (no guessing)", "code",
         sum(bool(i.get("fallback_reason")) for i in absent), len(absent)),
        ("Task 3", "RAG answer cites only retrieved chunks (all citations valid)", "code",
         sum(r["citation_valid_pct"] == 100.0 for r in rag_ok), len(rag_ok)),
    ]
    df = pd.DataFrame(rows, columns=["task", "constraint", "checked_by", "complied", "total"])
    df["adherence_pct"] = [pct(c, t) for c, t in zip(df["complied"], df["total"])]
    return df


def hallucination_evidence() -> pd.DataFrame:
    """Signals that an output states something the source does not support."""
    jd = judge_scores_df
    items = [i for r in successful for i in r["indicators"] if i.get("value") is not None]
    rejected = [i for i in items if i["final_value"] is None]
    trend_n = len(llm_trend_checks)
    rows = [
        ("Task 1 summaries", "Numbers not found in source (numeric grounding)", "code",
         f"{jd['numeric_grounding_pct'].mean():.1f}% of numbers grounded; "
         f"{int((jd['numeric_grounding_pct'] < 100).sum())}/{len(jd)} outputs with an ungrounded number" if not jd.empty else "-"),
        ("Task 1 summaries", "Claims judged UNSUPPORTED / CONTRADICTED", "LLM judge",
         f"{int(jd['unsupported'].sum())} unsupported, {int(jd['contradicted'].sum())} contradicted "
         f"of {int(jd['n_claims'].sum())} claims" if not jd.empty else "-"),
        ("Task 2 indicators", "Extracted values rejected by code verification", "code",
         f"{len(rejected)}/{len(items)} rejected -> null (never shown as data)"),
        ("Task 2 trends", "LLM trend direction / peak year contradicts the table", "code",
         f"direction wrong {sum(not p['direction_matches'] for p in llm_trend_checks)}/{trend_n}; "
         f"peak year wrong {sum(not p['peak_year_matches'] for p in llm_trend_checks)}/{trend_n}"),
    ]
    if not rag_scores_df.empty:
        g = rag_scores_df.groupby("mode")
        for mode, label in (("rag", "RAG (grounded)"), ("closed_book", "Closed-book (no retrieval)")):
            if mode in g.groups:
                d = g.get_group(mode)
                rows.append(("Task 3 " + label, "Groundedness (1-5) and unsupported claims per answer", "LLM judge",
                             f"groundedness {d['groundedness_score'].mean():.1f}/5; "
                             f"{d['n_unsupported_claims'].mean():.1f} unsupported claims per answer"))
        rows.append(("Task 3 monitor", "Questions flagged (score < 3 or any unsupported claim)", "code on judge",
                     f"{int(rag_monitor_df[['retrieval_flag', 'hallucination_flag', 'answer_flag']].any(axis=1).sum())}"
                     f"/{len(rag_monitor_df)} flagged"))
    return pd.DataFrame(rows, columns=["stage", "signal", "checked_by", "result"])


def failure_log() -> pd.DataFrame:
    """Every detected failure instance, labelled with its failure mode."""
    log = []
    add = lambda stage, item, mode, detail: log.append({"stage": stage, "item": item, "failure_mode": mode,
                                                         "detail": str(detail)[:200]})
    for r in section_summaries:
        if r.get("status") == "success" and not r["passed_word_limit"]:
            add("Task 1", r["section_id"], "length constraint violated", f"{r['word_count']} words")
    for _, r in judge_scores_df.iterrows():
        if r["ungrounded_numbers"]:
            add("Task 1", r["target_id"], "numeric fabrication (number not in source)", r["ungrounded_numbers"])
        if r["contradicted"]:
            add("Task 1", r["target_id"], "contradiction of source", f"{r['contradicted']} claim(s)")
        if r["source_coverage_pct"] is not None and r["source_coverage_pct"] < 100:
            add("Task 1", r["target_id"], "judge saw truncated source", f"coverage {r['source_coverage_pct']}%")
    for res in successful:
        for i in res["indicators"]:
            if i.get("value") is not None and i["final_value"] is None:
                mode = "stale value" if i["fallback_reason"].startswith("Stale") else "unverifiable quote / value"
                add("Task 2", f"{res['model']}:{i['field']}", mode, i["fallback_reason"])
    for p in llm_trend_checks:
        if not p["direction_matches"] or not p["peak_year_matches"]:
            add("Task 2", p["series_name"], "trend misread",
                f"LLM {p['trend_direction']}/{p['peak_year']} vs data {p['computed_direction']}/{p['computed_peak_year']}")
    for r in rag_results:
        if "rag_answer" not in r:
            continue
        if r["citation_valid_pct"] < 100:
            add("Task 3", r["id"], "citation outside retrieved context", f"{r['citation_valid_pct']}% valid")
        for mode, j in r.get("judged", {}).items():
            for claim in j.get("unsupported_claims", []):
                add("Task 3", f"{r['id']} ({mode})",
                    "closed-book hallucination" if mode == "closed_book" else "unsupported claim despite retrieval", claim)
            if mode == "rag" and j.get("context_relevance_score", 5) < TRIAD_ALERT_THRESHOLD:
                add("Task 3", r["id"], "retrieval miss (irrelevant context)", f"context relevance {j['context_relevance_score']}")
    if red_team and not red_team.get("caught_both"):
        add("Judge", "red-team extraction", "judge missed tampered values", red_team.get("tampered_fields"))
    if not cross_summary.empty and "mean_diff_gemini_minus_cross" in cross_summary:
        for metric, d in cross_summary["mean_diff_gemini_minus_cross"].items():
            if d >= 0.5:
                add("Judge", metric, "possible self-enhancement bias (Gemini judge higher)", f"+{d:.2f} vs Groq")
    return pd.DataFrame(log, columns=["stage", "item", "failure_mode", "detail"])


# EXECUTION
constraints_df = constraint_adherence()
hallucination_df = hallucination_evidence()
failures_df = failure_log()
for name, frame in {"constraint_adherence": constraints_df, "hallucination_evidence": hallucination_df,
                    "failure_log": failures_df}.items():
    frame.to_csv(EVAL_DIR / f"{name}.csv", index=False)
save_json({"constraint_adherence": constraints_df.to_dict("records"),
           "hallucination_evidence": hallucination_df.to_dict("records"),
           "failure_mode_counts": failures_df["failure_mode"].value_counts().to_dict(),
           "red_team": red_team or {"status": "not_run"}}, EVAL_DIR / "evaluation_summary.json")

print("CONSTRAINT ADHERENCE (did the model follow its prompt's explicit rules?)")
display(constraints_df)
print("\nHALLUCINATION EVIDENCE (RAG vs closed-book is the grounding test)")
display(hallucination_df)
print(f"\nFAILURE MODES: {len(failures_df)} instances")
display(failures_df["failure_mode"].value_counts().rename("instances").to_frame())
display(failures_df.head(15))

# Task 4 — Data visualisation and dashboard
## 20. Tidy transformation with pd.melt

In [ ]:
# ==============================================================================
# CELL 26: TASK 4 — TIDY TRANSFORMATION (wide matrices -> long with pd.melt)
# ------------------------------------------------------------------------------
# Every plot in the dashboard reads a long ("tidy") DataFrame: one observation
# per row, one variable per column. Each wide matrix produced by Tasks 1-3 is
# reshaped explicitly with pd.melt and saved to outputs/task4/ for inspection.
# ==============================================================================

TASK4_DIR = CFG.output_dir / "task4"
TASK4_DIR.mkdir(parents=True, exist_ok=True)


def melt_time_series(long_records: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """indicator x year wide matrix -> long, re-attaching actual/projection status."""
    wide = long_records.pivot_table(index="series", columns="year", values="value").reset_index()
    long = wide.melt(id_vars="series", var_name="year", value_name="value").dropna(subset=["value"])
    long["year"] = long["year"].astype(int)
    long["status"] = np.where(long["year"] <= (REPORT_YEAR or 9999) - 1, "actual", "projection")
    return wide, long


def melt_themes(by_section_wide: pd.DataFrame) -> pd.DataFrame:
    return by_section_wide.melt(id_vars=["section_id", "section_title"], var_name="theme", value_name="mentions")


def melt_scores(wide: pd.DataFrame, id_vars: List[str], metrics: List[str]) -> pd.DataFrame:
    """Wide score matrix -> long (metric, value); empty frame with the right columns if nothing to melt."""
    present = [m for m in metrics if m in wide.columns]
    if wide.empty or not present:
        return pd.DataFrame(columns=id_vars + ["metric", "value"])
    return wide[id_vars + present].melt(id_vars=id_vars, var_name="metric", value_name="value").dropna(subset=["value"])


def melt_indicators(core: Dict[str, Any], extended: Dict[str, Dict[str, Any]]) -> pd.DataFrame:
    """Core (target schema) + extended indicators -> long, flagging fallbacks."""
    core_long = pd.DataFrame([core]).melt(id_vars="country_name", var_name="indicator", value_name="value")
    core_long["group"] = "core"
    ext_wide = pd.DataFrame([{k: v["value"] for k, v in extended.items()}])
    ext_long = ext_wide.melt(var_name="indicator", value_name="value")
    ext_long["group"], ext_long["country_name"] = "extended", core.get("country_name")
    long = pd.concat([core_long, ext_long], ignore_index=True)
    long["fallback"] = long["value"].isna()               # NaN stays out of plots and means
    long["status"] = np.where(long["fallback"], "not_reported", "verified")
    return long


# Model scorecard: one row per model, one column per diagnostic; feeds the single
# model-comparison panel. Judge columns use the cross-family sample (both judges
# scored the same items) when available, else all Gemini judge audits.
SCORECARD_METRICS = [
    ("core_verified_pct", "Core\nverified %", "pct", "Extraction"),
    ("extended_verified_pct", "Extended\nverified %", "pct", "Extraction"),
    ("rag_groundedness", "Grounded\nRAG", "score", "Answers"),
    ("closed_book_groundedness", "Grounded\nclosed-book", "score", "Answers"),
    ("judge_factual_alignment_score", "Factual\nalignment", "score", "Scores given as judge"),
    ("judge_consistency_score", "Consis-\ntency", "score", "Scores given as judge"),
    ("judge_completeness_score", "Complete-\nness", "score", "Scores given as judge"),
    ("judge_agreement_within_1_pct", "Agrees ±1\nw/ Gemini %", "pct", "Scores given as judge"),
]


def short_model(name: Any) -> str:
    return str(name).replace("mock:", "").split("/")[-1]


def model_scorecard() -> Tuple[pd.DataFrame, str]:
    rows: Dict[str, Dict[str, Any]] = {}

    def put(model: Any, role: str, metric: str, value: Any) -> None:
        if model is None or pd.isna(value):
            return
        r = rows.setdefault(short_model(model), {"model": short_model(model), "roles": []})
        if role not in r["roles"]:
            r["roles"].append(role)
        r[metric] = float(value)

    for _, r in model_comparison_df.iterrows():
        put(r["model"], "extractor", "core_verified_pct", r["core_verified_pct"])
        put(r["model"], "extractor", "extended_verified_pct", r["extended_verified_pct"])
    if not rag_scores_df.empty:
        gen = next((x["generator_model"] for x in rag_results if x.get("generator_model")), None)
        for mode, col in (("rag", "rag_groundedness"), ("closed_book", "closed_book_groundedness")):
            put(gen, "generator", col, rag_scores_df.loc[rag_scores_df["mode"] == mode, "groundedness_score"].mean())
    gemini_judge = judge_scores_df["judge_model"].mode()[0] if not judge_scores_df.empty else JUDGE_MODELS[0]
    if not cross_df.empty:
        sample = f"judge columns: same {cross_df['target_id'].nunique()} outputs scored by both judges"
        d = cross_df.dropna(subset=["gemini_score"])
        cross_model = d["cross_model"].mode()[0] if not d.empty else cross_df["cross_model"].mode()[0]
        for m in JUDGE_METRICS:
            dm = d[d["metric"] == m]
            put(gemini_judge, "judge", f"judge_{m}", dm["gemini_score"].astype(float).mean())
            put(cross_model, "cross-family judge", f"judge_{m}", dm["cross_family_score"].astype(float).mean())
        if not d.empty:
            diff = (d["gemini_score"].astype(float) - d["cross_family_score"].astype(float)).abs()
            put(cross_model, "cross-family judge", "judge_agreement_within_1_pct", 100 * (diff <= 1).mean())
    else:
        sample = "judge columns: all Gemini judge audits (no cross-family audits)"
        for m in JUDGE_METRICS:
            put(gemini_judge, "judge", f"judge_{m}", judge_scores_df[m].mean() if not judge_scores_df.empty else np.nan)
    wide = pd.DataFrame(list(rows.values()), columns=["model", "roles"] + [m for m, *_ in SCORECARD_METRICS])
    wide["roles"] = wide["roles"].apply(lambda r: " · ".join(r) if isinstance(r, list) else "")
    return wide, sample


# EXECUTION
scorecard_wide, SCORECARD_SAMPLE = model_scorecard()
scorecard_long = melt_scores(scorecard_wide, ["model", "roles"], [m for m, *_ in SCORECARD_METRICS])
ts_wide, ts_long = melt_time_series(time_series_long)
themes_long = melt_themes(theme_by_section_wide)
judge_long = melt_scores(judge_scores_df, ["target_id", "target_type"], JUDGE_METRICS + ["overall_score"])
model_cmp_long = melt_scores(model_comparison_df, ["model"],
                             ["core_verified_pct", "extended_verified_pct", "judge_factual_alignment_score",
                              "core_fallbacks", "extended_fallbacks"])
rag_long = melt_scores(rag_scores_df, ["question_id", "mode"], RAG_TRIAD_METRICS + RAG_QUALITY_METRICS)
indicators_long = melt_indicators(core_indicators, extended_indicators)

for name, frame in {"time_series_wide": ts_wide, "time_series_long": ts_long, "themes_long": themes_long,
                    "judge_scores_long": judge_long, "model_comparison_long": model_cmp_long,
                    "rag_triad_long": rag_long, "indicators_long": indicators_long,
                    "model_scorecard_wide": scorecard_wide, "model_scorecard_long": scorecard_long}.items():
    frame.to_csv(TASK4_DIR / f"{name}.csv", index=False)
    print(f"{name:24s} {str(frame.shape):>10s}")
display(ts_long.head())

## 21. Dashboard (4 plots, dpi 300)

In [ ]:
# ==============================================================================
# CELL 27: TASK 4 — 4-PLOT DASHBOARD (dpi 300), exactly the brief's four plots
# ------------------------------------------------------------------------------
# A Thematic density across the 7 financial-stability themes (horizontal bar)
# B Inflation vs interest rate, actual and IMF projections (line)
# C Model comparison: extraction accuracy + judge scores, incl. the Groq
#   cross-family judge (grouped bar, one axis indexed to scale maximum)
# D Credit growth vs inflation by year (scatter) — the report publishes no NPL
#   ratio series, so NPLs cannot be the y-axis (see Task 2 fallbacks)
# Design: one y-axis per panel (no dual axes), fixed categorical colour order,
# direct labels, legends below the axes so they never cover data or titles,
# recessive grid. Every panel is also saved on its own for the written report.
# ==============================================================================

import matplotlib
import matplotlib.patches
import matplotlib.ticker

PALETTE = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a"}      # fixed categorical order
INK, INK_2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": GRID,
                     "axes.labelcolor": INK, "xtick.color": INK_2, "ytick.color": INK_2, "text.color": INK,
                     "grid.color": GRID, "grid.linewidth": 0.8, "axes.grid": True, "axes.axisbelow": True,
                     "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold", "legend.frameon": False})


def no_data(ax, title: str, msg: str) -> None:
    ax.set_title(title, loc="left")
    ax.text(0.5, 0.5, msg, ha="center", va="center", color=INK_2, transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])


def panel_themes(ax) -> None:
    title = "A. Thematic density across 7 financial-stability themes"
    totals = themes_long.groupby("theme", as_index=False)["mentions"].sum()
    words = max(thematic["words_scanned"], 1)
    totals["per_10k"] = 10_000 * totals["mentions"] / words
    totals = totals.sort_values("per_10k")
    if totals["mentions"].sum() == 0:
        return no_data(ax, title, "No theme mentions found")
    ax.barh(totals["theme"], totals["per_10k"], color=PALETTE["blue"], height=0.6)
    for y, (v, n) in enumerate(zip(totals["per_10k"], totals["mentions"])):
        ax.text(v, y, f"  {v:.0f} ({n:,})", va="center", color=INK_2, fontsize=10)
    ax.set_xlim(0, totals["per_10k"].max() * 1.25)
    ax.set_xlabel("Mentions per 10,000 words (total mentions in brackets)")
    ax.set_title(title, loc="left"); ax.grid(axis="y", visible=False)


def draw_lines(ax, title: str, specs: List[Tuple[str, str, str]], ylabel: str, zero_line: bool = False) -> None:
    """Report series over time: one y-axis, direct end labels, projection years shaded."""
    plotted = []
    for name, short, colour in specs:
        s = ts_long[ts_long["series"] == name].sort_values("year")
        if s.empty:
            continue
        ax.plot(s["year"], s["value"], color=colour, lw=2, marker="o", ms=5, label=short)
        plotted.append(s.assign(short=short))
    if not plotted:
        return no_data(ax, title, "Series not found in parsed tables")
    data = pd.concat(plotted)
    min_gap = 0.06 * float((data["value"].max() - data["value"].min()) or 1.0)
    ends = [(d["year"].iloc[-1], d["value"].iloc[-1], d["short"].iloc[0]) for d in plotted]
    last_y = -np.inf
    for x, y, label in sorted(ends, key=lambda e: e[1]):   # bottom-up; push labels apart by min_gap
        y_text = max(y, last_y + min_gap)
        ax.annotate(label, (x, y), xytext=(x + 0.3, y_text), va="center", fontsize=9, color=INK_2)
        last_y = y_text
    x_min, x_max = data["year"].min(), data["year"].max()
    if zero_line:
        ax.axhline(0, color=INK_2, lw=1)
    if REPORT_YEAR and x_max >= REPORT_YEAR:
        ax.axvspan(REPORT_YEAR - 0.5, x_max + 0.5, color=GRID, alpha=0.5, lw=0)
        ax.text(REPORT_YEAR - 0.3, 0.97, "IMF projections →", color=INK_2, fontsize=9, va="top",
                transform=ax.get_xaxis_transform())
    ax.set_xlim(x_min - 0.5, x_max + 3.5)                          # room for direct labels
    ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(integer=True))
    ax.set_xlabel("Year"); ax.set_ylabel(ylabel)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=len(plotted), fontsize=9)
    ax.set_title(title, loc="left")


def panel_inflation_rates(ax) -> None:
    draw_lines(ax, "B. Inflation vs interest rate: actual and IMF projections",
               [("CPI inflation, period average (%)", "CPI, period avg", PALETTE["blue"]),
                ("CPI inflation, end-period (%)", "CPI, end-period", PALETTE["orange"]),
                ("Effective interest rate on public debt (%)", "Effective rate, public debt", PALETTE["aqua"])],
               "Percent")


def panel_credit_inflation(ax) -> None:
    title = "D. Credit growth vs inflation by year (NPL ratio not reported)"
    x = ts_long[ts_long["series"] == "Credit growth (%)"][["year", "value", "status"]]
    y = ts_long[ts_long["series"] == "CPI inflation, end-period (%)"][["year", "value"]]
    df = x.merge(y, on="year", suffixes=("_credit", "_cpi"))
    if df.empty:
        return no_data(ax, title, "Credit growth / CPI series not found")
    for status, face in (("actual", PALETTE["blue"]), ("projection", SURFACE)):
        d = df[df["status"] == status]
        ax.scatter(d["value_credit"], d["value_cpi"], s=70, facecolor=face, edgecolor=PALETTE["blue"],
                   linewidth=2, label=status.capitalize(), zorder=3)
    for _, r in df.iterrows():
        ax.annotate(str(r["year"]), (r["value_credit"], r["value_cpi"]), xytext=(5, 4),
                    textcoords="offset points", fontsize=9, color=INK_2)
    ax.set_xlabel("Credit growth (percent change)"); ax.set_ylabel("CPI inflation, end-period (percent)")
    ax.legend(loc="lower right", fontsize=9); ax.set_title(title, loc="left")


MODEL_COLOURS = [PALETTE["blue"], PALETTE["orange"], PALETTE["aqua"]]      # fixed order: scorecard rows


def panel_model_comparison(ax) -> None:
    """ONE model-comparison bar chart: extraction accuracy, answer grounding and judge scores per model.
    Mixed units share one axis by indexing to the scale maximum (% for %, score/5 for 1-5 scores);
    every bar is labelled with its raw value."""
    title = "C. Model comparison: extraction accuracy and judge scores"
    if scorecard_wide.empty:
        return no_data(ax, title, "No model diagnostics")
    metrics = [m for m in SCORECARD_METRICS if scorecard_wide[m[0]].notna().any()]
    width = 0.8 / max(len(scorecard_wide), 1)
    handles = [matplotlib.patches.Patch(color=MODEL_COLOURS[k % 3], label=f"{r['model']} ({r['roles']})")
               for k, (_, r) in enumerate(scorecard_wide.iterrows())]
    for g, (col, label, kind, _) in enumerate(metrics):
        present = [(k, r[col]) for k, (_, r) in enumerate(scorecard_wide.iterrows()) if pd.notna(r[col])]
        for slot, (k, v) in enumerate(present):
            x = g + (slot - (len(present) - 1) / 2) * width
            h = v if kind == "pct" else 100 * v / 5
            ax.bar(x, h, width=width - 0.03, color=MODEL_COLOURS[k % 3])
            ax.text(x, h + 1.5, f"{v:.0f}%" if kind == "pct" else f"{v:.1f}", ha="center", fontsize=9, color=INK_2)
    groups = [m[3] for m in metrics]
    for g in range(1, len(groups)):
        if groups[g] != groups[g - 1]:
            ax.axvline(g - 0.5, color=INK_2, lw=1, ls=(0, (4, 3)))
    for name in dict.fromkeys(groups):
        idx = [g for g, x in enumerate(groups) if x == name]
        ax.text(np.mean(idx), 112, name, ha="center", va="bottom", fontsize=10, fontweight="bold", color=INK)
    ax.set_xticks(range(len(metrics)), [m[1] for m in metrics], fontsize=9)
    ax.set_xlim(-0.6, len(metrics) - 0.4); ax.set_ylim(0, 120); ax.set_yticks(range(0, 101, 20))
    ax.set_ylabel("% of scale maximum (labels: raw % or 1-5 score)")
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=len(handles), fontsize=9)
    ax.set_title(title, loc="left", pad=8); ax.grid(axis="x", visible=False)


DASHBOARD_TITLE = re.split(r"\s*[-—–]\s*Press Release", REPORT_TITLE)[0]

PANELS = [("A_thematic_density", panel_themes), ("B_inflation_vs_interest", panel_inflation_rates),
          ("C_model_comparison", panel_model_comparison), ("D_credit_vs_inflation", panel_credit_inflation)]


def safe_draw(ax, name: str, draw) -> None:
    """One failing panel must never blank the rest of the dashboard."""
    try:
        draw(ax)
    except Exception as exc:
        logging.error(f"Panel {name} failed: {exc}")
        ax.cla()
        no_data(ax, name.replace("_", " "), f"Panel error: {str(exc)[:80]}")


def indicator_footer(indicators: Dict[str, Any]) -> str:
    shown = [f"{k}: {v:g}" for k, v in indicators.items() if k != "country_name" and v is not None]
    missing = [k for k, v in indicators.items() if k != "country_name" and v is None]
    return (f"{indicators.get('country_name', '')} core indicators — " + ("; ".join(shown) or "none verified")
            + (f".  Not reported (JSON fallback {FALLBACK_SENTINEL}, not plotted): {', '.join(missing)}." if missing else ""))


def build_dashboard(path: Path) -> Path:
    fig, axes = plt.subplots(2, 2, figsize=(20, 14))
    for ax, (name, draw) in zip(axes.flat, PANELS):
        safe_draw(ax, name, draw)
    mock_tag = "[MOCK DATA — development run]  " if CFG.llm_mode == "mock" else ""
    fig.suptitle(f"{mock_tag}{DASHBOARD_TITLE}: macro-financial dashboard and model diagnostics",
                 x=0.01, ha="left", fontsize=18, fontweight="bold")
    fig.text(0.01, 0.01, indicator_footer(core_indicators) + "  Source: IMF Country Report No. 25/265; "
             "pipeline outputs (Tasks 1-3).", fontsize=10, color=INK_2, wrap=True)
    fig.tight_layout(rect=(0, 0.03, 1, 0.96), h_pad=3, w_pad=3)
    fig.savefig(path, dpi=300, bbox_inches="tight")
    return path


def save_individual_panels(out_dir: Path) -> List[Path]:
    paths = []
    for name, draw in PANELS:
        fig, ax = plt.subplots(figsize=(11, 6.5))
        safe_draw(ax, name, draw)
        fig.tight_layout()
        p = out_dir / f"panel_{name}.png"
        fig.savefig(p, dpi=300, bbox_inches="tight")
        plt.close(fig)
        paths.append(p)
    return paths


# EXECUTION
dashboard_path = build_dashboard(CFG.figure_dir / "dashboard.png")
panel_paths = save_individual_panels(CFG.figure_dir)
plt.show()
print(f"✅ Dashboard: {dashboard_path} | individual panels: {len(panel_paths)} (dpi 300)")
print(USAGE.report())

## 22. Figures with data-driven commentary
Each panel with commentary whose numbers are computed from the pipeline outputs (no hand-typed or LLM-written figures).

In [ ]:
# ==============================================================================
# CELL 27b: TASK 4 — FIGURES WITH DATA-DRIVEN COMMENTARY
# ------------------------------------------------------------------------------
# Each dashboard panel is displayed on its own with commentary underneath.
# Every number in the commentary is computed here from the pipeline's outputs
# (parsed IMF tables, thematic counts, judge scores) — nothing is typed by hand
# or written by an LLM, so the commentary cannot introduce hallucinated figures.
# Interpretive sentences are conditional on the computed values. The text is
# also saved to outputs/figures/dashboard_commentary.md for the written report.
# ==============================================================================
from IPython.display import Image, Markdown


def series_values(name: str) -> pd.DataFrame:
    return ts_long[ts_long["series"] == name].sort_values("year")


def last_actual_year() -> Optional[int]:
    return REPORT_YEAR - 1 if REPORT_YEAR else None


def comment_themes() -> str:
    totals = themes_long.groupby("theme")["mentions"].sum().sort_values(ascending=False)
    if totals.sum() == 0:
        return "No theme mentions were found."
    share = 100 * totals / totals.sum()
    per10k = 10_000 * totals / max(thematic["words_scanned"], 1)
    top2 = totals.index[:2]
    return (f"**{top2[0]}** ({per10k[top2[0]]:.0f} mentions per 10,000 words) and **{top2[1]}** "
            f"({per10k[top2[1]]:.0f}) together account for {share[top2].sum():.0f}% of all theme mentions, "
            f"so these two themes dominate the consultation's financial-stability language. "
            f"**{totals.index[-1]}** is the least discussed theme ({int(totals.iloc[-1])} mentions, "
            f"{share.iloc[-1]:.0f}%), a minor channel in this report. "
            f"Counts are keyword-based (Task 2), so they measure emphasis, not severity.")


def comment_inflation_rates() -> str:
    cpi = series_values("CPI inflation, period average (%)")
    eff = series_values("Effective interest rate on public debt (%)")
    if cpi.empty:
        return "CPI series not found in the parsed tables."
    first, last = cpi.iloc[0], cpi.iloc[-1]
    actual = cpi[cpi["year"] <= (last_actual_year() or 9999)]
    low = actual.loc[actual["value"].idxmin()] if not actual.empty else first
    proj = cpi[cpi["year"] > (last_actual_year() or 9999)]
    text = (f"Average CPI inflation moved from {first['value']:.1f}% in {int(first['year'])} to a low of "
            f"{low['value']:.1f}% in {int(low['year'])}.")
    if not proj.empty:
        hi = proj.loc[proj["value"].idxmax()]
        if hi["value"] > low["value"] and hi["year"] < last["year"]:
            text += (f" IMF staff project a rebound to {hi['value']:.1f}% in {int(hi['year'])}, then easing to "
                     f"{last['value']:.1f}% by {int(last['year'])}.")
        else:
            trend = "lower" if last["value"] < low["value"] else "higher"
            text += f" IMF staff project {last['value']:.1f}% by {int(last['year'])}, {trend} than the latest outturn."
    if not eff.empty:
        both = cpi.merge(eff, on="year", suffixes=("_cpi", "_eff"))
        below = both[both["value_eff"] < both["value_cpi"]]
        text += (f" The effective interest rate on public debt stays between {eff['value'].min():.1f}% and "
                 f"{eff['value'].max():.1f}%, below average inflation in {len(below)} of {len(both)} overlapping years. "
                 f"A real interest rate below zero lowers the cost of servicing government debt, but this "
                 f"advantage narrows as inflation declines.")
    policy = next((i for i in successful[0]["indicators"] if i["field"] == "central_bank_policy_rate_pct"), {})
    if policy.get("final_value") is None:
        text += (" The central bank policy rate is not tabulated as a time series in the report "
                 "(Task 2 fallback), so the effective rate on public debt is the only interest-rate series.")
    return text


def comment_model_comparison() -> str:
    sc = scorecard_wide.set_index("model")
    parts = []
    ext = sc["core_verified_pct"].dropna()
    if not ext.empty:
        parts.append("Extraction: " + "; ".join(f"{m} verified {v:.0f}% of core and "
                                                 f"{sc.loc[m, 'extended_verified_pct']:.0f}% of extended values"
                                                 for m, v in ext.items()) +
                     ". Unverified values are rejected to null, so the dashboard shows only code-verified numbers.")
    rag = sc["rag_groundedness"].dropna()
    if not rag.empty:
        m = rag.index[0]
        cb = sc.loc[m, "closed_book_groundedness"]
        verdict = ("evidence that retrieval reduces hallucination on the same questions" if rag.iloc[0] > cb
                   else "so retrieval did not improve groundedness here; see the RAG monitor")
        parts.append(f"Grounding: the generator's answers score {rag.iloc[0]:.1f}/5 for groundedness with retrieval vs "
                     f"{cb:.1f}/5 closed-book, {verdict}.")
    if not cross_summary.empty:
        d = cross_summary["mean_diff_gemini_minus_cross"]
        agree = cross_summary["within_1_point_pct"].mean()
        parts.append(f"Judges: the Groq cross-family judge agrees with the Gemini judge within one point "
                     f"{agree:.0f}% of the time. The Gemini judge scores higher by {d.max():.2f} points at most "
                     f"({d.idxmax().replace('_score', '').replace('_', ' ')}); a consistent positive gap would "
                     f"indicate self-enhancement bias (Lecture 4).")
    return " ".join(parts) or "No model diagnostics available."


def comment_credit_inflation() -> str:
    credit = series_values("Credit growth (%)")
    cpi = series_values("CPI inflation, end-period (%)")
    df = credit.merge(cpi, on="year", suffixes=("_credit", "_cpi"))
    if df.empty:
        return "Credit growth / CPI series not found."
    actual = df[df["year"] <= (last_actual_year() or 9999)]
    peak = (actual if not actual.empty else df).loc[lambda d: d["value_credit"].idxmax()]
    r = df["value_credit"].corr(df["value_cpi"])
    proj = df[df["year"] > (last_actual_year() or 9999)]
    text = (f"Credit growth peaked at {peak['value_credit']:.1f}% in {int(peak['year'])}, when end-period CPI "
            f"inflation was {peak['value_cpi']:.1f}%.")
    if not proj.empty:
        verb = "slow" if proj["value_credit"].iloc[-1] < peak["value_credit"] else "raise"
        cpi_dir = "lower" if proj["value_cpi"].iloc[-1] < peak["value_cpi"] else "higher"
        text += (f" IMF projections {verb} credit growth to {proj['value_credit'].iloc[-1]:.1f}% by "
                 f"{int(proj['year'].iloc[-1])}, with end-period inflation {cpi_dir} than in "
                 f"{int(peak['year'])} ({proj['value_cpi'].iloc[-1]:.1f}%).")
    text += (f" Across {len(df)} years (actuals and projections) the correlation is r = {r:.2f}, a descriptive "
             f"association, not a causal estimate. The report publishes no NPL ratio series (Task 2 fallback), "
             f"so the credit-quality side of the brief's credit-vs-NPL plot cannot be shown from the report's own data.")
    return text


COMMENTARY = {"A_thematic_density": ("Thematic density", comment_themes),
              "B_inflation_vs_interest": ("Inflation vs interest rate", comment_inflation_rates),
              "C_model_comparison": ("Model comparison", comment_model_comparison),
              "D_credit_vs_inflation": ("Credit growth vs inflation", comment_credit_inflation)}

# EXECUTION
md_lines = [f"# Dashboard commentary — {DASHBOARD_TITLE}\n"]
for (name, _), path in zip(PANELS, panel_paths):
    heading, fn = COMMENTARY[name]
    try:
        text = fn()
    except Exception as exc:                       # commentary must never break the run
        text = f"Commentary unavailable ({exc})."
    display(Image(filename=str(path), width=900))
    display(Markdown(f"**{name[0]}. {heading}.** {text}"))
    md_lines += [f"## {name[0]}. {heading}\n", f"![{heading}](panel_{name}.png)\n", text + "\n"]
(CFG.figure_dir / "dashboard_commentary.md").write_text("\n".join(md_lines), encoding="utf-8")
print(f"✅ Commentary saved: {CFG.figure_dir / 'dashboard_commentary.md'}")

## Author only: publish outputs to GitHub
Skipped unless `PUBLISH_TO_GITHUB=1`;

In [ ]:
# ==============================================================================
# CELL 99: AUTHOR ONLY — PUBLISH RUN OUTPUTS TO GITHUB
# ------------------------------------------------------------------------------
# Runs only when PUBLISH_TO_GITHUB=1. Commits Data/, outputs/ (incl. the LLM
# cache that lets users replay without API keys) and requirements.txt from the
# cloned repo, then pushes using a fine-grained token stored in Colab Secrets as
# GITHUB_TOKEN. The token is sent as an HTTP header for this one command only;
# it is never printed, written to .git/config, or committed.
# ==============================================================================
import base64

PUBLISH_PATHS = ["Data", "outputs", "requirements.txt"]


def git(*args: str, cwd: Path, extra_config: Optional[List[str]] = None) -> str:
    cfg = sum((["-c", c] for c in (extra_config or [])), [])
    result = subprocess.run(["git", *cfg, *args], cwd=cwd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed: {result.stderr.strip()[:300]}")
    return result.stdout.strip()


def publish_outputs(project_dir: Path, token: str, message: str) -> str:
    """Commit and push pipeline outputs from a cloned repository."""
    if not (project_dir / ".git").exists():
        raise RuntimeError(f"{project_dir} is not a git clone; run the notebook via Cell 0 (REPO_URL).")
    identity = [f"user.name={_env('GIT_USER_NAME', 'Colab pipeline run')}",
                f"user.email={_env('GIT_USER_EMAIL', 'colab-run@users.noreply.github.com')}"]
    git("add", "--", *[p for p in PUBLISH_PATHS if (project_dir / p).exists()], cwd=project_dir)
    if not git("diff", "--cached", "--name-only", cwd=project_dir):      # staged changes only
        return "Nothing to publish: outputs already up to date."
    git("commit", "-m", message, cwd=project_dir, extra_config=identity)
    auth = []
    if token:
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        auth = [f"http.extraHeader=Authorization: Basic {basic}"]
    git("push", "origin", "HEAD", cwd=project_dir, extra_config=auth)
    return f"Pushed commit {git('rev-parse', '--short', 'HEAD', cwd=project_dir)}."


# EXECUTION
if os.environ.get("PUBLISH_TO_GITHUB") == "1":
    print(publish_outputs(CFG.project_dir, load_secret("GITHUB_TOKEN", IN_COLAB),
                          message="Update pipeline outputs and LLM cache"))
else:
    print("ℹ️ Publishing skipped (author-only step; set PUBLISH_TO_GITHUB=1 to enable).")